# People Domain — Benchmark Record Generator (Scientists, Physicists, Mathematicians)

This notebook generates Russian/English people-domain benchmark tasks with complete Wikidata gold answers,
using the same JSONL schema as the cinema and geo domains. It covers scientists, physicists, mathematicians,
and other prominent persons via multi-hop SPARQL templates at complexity levels L1–L5.

Outputs:
- `out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_fast_clean.jsonl` — primary dataset (L3/L5 supplement, 35 L3 + 35 L5 records)


## 1. Common helpers

In [1]:
# Load common helpers only if this domain notebook is run standalone.
# This avoids `%run ./00_common_helpers.ipynb`, which requires nbformat.
from pathlib import Path as _Path

if "BenchmarkExample" not in globals():
    helper_path = _Path("common_helpers.py")
    if not helper_path.exists():
        raise FileNotFoundError("common_helpers.py must be in the same directory as 08_people.ipynb")
    exec(helper_path.read_text(encoding="utf-8"), globals())

print("✅ common helpers loaded")

✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label
✅ common helpers loaded


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [2]:
from __future__ import annotations

import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from dataclasses import asdict, fields, is_dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

try:
    from tqdm.auto import tqdm
except ImportError:
    import sys
    !{sys.executable} -m pip install tqdm
    from tqdm.auto import tqdm

# -----------------------------
# Output paths
# -----------------------------
PEOPLE_DOMAIN = "people"
PEOPLE_OUTPUT_DIR = Path("out_wikidata_benchmark/domain_outputs")
PEOPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep exactly one dataset JSONL for this domain.
PEOPLE_OUTPUT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v34_fast_clean.jsonl"
PEOPLE_AUDIT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v34_fast_clean_audit.json"
PEOPLE_CHECKPOINT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v34_fast_clean_checkpoint.json"

# Optional reference files are NOT copied into the output. They are used only
# to avoid duplicate query/constraint pairs and to continue IDs after an already
# curated people JSONL. Put your current final file next to the notebook or in
# the domain output directory under one of these names.
PEOPLE_REFERENCE_JSONL_PATHS = [
    Path("people_final_curated_v2.jsonl"),
    PEOPLE_OUTPUT_DIR / "people_final_curated_v2.jsonl",
    PEOPLE_OUTPUT_DIR / "people.jsonl",
]

# -----------------------------
# Generation targets
# -----------------------------
# Total = 185. Hard levels intentionally dominate because this domain is meant
# to add interesting multi-hop examples rather than many trivial people lists.
# v30 default: generate a focused hard-multihop supplement, not another full
# L1-L5 dataset. 45 + 65 = 110 records, matching the requested 100-120 range.
# If you want to continue an existing final file instead, copy/rename it to
# PEOPLE_OUTPUT_PATH before running the generation cell.
PEOPLE_TARGET_PER_LEVEL = {
    "L1": 0,
    "L2": 0,
    "L3": 35,
    "L4": 0,
    "L5": 35,
}

PEOPLE_LEVEL_ORDER = ["L1", "L2", "L3", "L4", "L5"]
PEOPLE_REQUESTED_COUNT = {"L1": 5, "L2": 5, "L3": 5, "L4": 5, "L5": 5}
PEOPLE_MIN_GOLD_BY_LEVEL = {"L1": 8, "L2": 8, "L3": 8, "L4": 8, "L5": 8}

# Reject overly broad prompts. This mirrors the later cinema filtering idea,
# but applies at generation time so accepted gold lists remain compact and reviewable.
PEOPLE_MAX_GOLD_BY_LEVEL = {"L1": 50, "L2": 50, "L3": 50, "L4": 50, "L5": 50}

# Dataset-level diversity guard: avoid many accepted examples in the same level
# with the same main occupation. This keeps the people domain from starting with
# a run of nearly identical prompts such as physicist/physicist/physicist.
PEOPLE_MAX_SAME_OCCUPATION_PER_LEVEL = {"L1": 2, "L2": 5, "L3": 5, "L4": 4, "L5": 6}

# L2 should be a diverse direct multi-criteria level, not an award-only level.
# Award templates are useful, but capped; the rest should come from non-award
# direct criteria such as occupation + meaningful field specialization, or
# footballer + playing position.  Language knowledge (P1412) is not used.
PEOPLE_L2_MAX_AWARD_SHARE = 0.30

# Large enough to detect broad / incomplete queries; examples that hit this
# limit are rejected, so accepted examples have complete gold relative to WDQS.
PEOPLE_WDQS_LIMIT_DEFAULT = 101  # max accepted gold is 50; 151 is enough to detect broad queries faster
PEOPLE_GOLD_LIMIT = 300

# Quality filters for the people domain.
# English labels are mandatory; Russian labels are used when Wikidata has them,
# otherwise gold_answer_labels_ru falls back to the English label. No other
# languages are used. We do NOT reject records just because Russian labels are
# missing.
PEOPLE_REJECT_DUPLICATE_ANSWER_LABELS = True

# User-facing text says simply "с профессией X", so gold collection must
# use direct P106. We do not use hidden P106/P279* subclass expansion.
PEOPLE_OCCUPATION_MATCH_PATH = "P106"

# "Well-known" is intentionally only a natural-language hint in query_text.
# Do not encode it in SPARQL: no sitelinks/notability filter is applied.

# Birth ranges must use at least year-level precision. This prevents Wikidata
# values like "20th century" from leaking into prompts that say "born from
# 1990 to 2005".
PEOPLE_MIN_BIRTH_DATE_PRECISION = 9

PEOPLE_RANDOM_SEED = 8080
PEOPLE_MAX_ATTEMPTS_PER_LEVEL = 1800
PEOPLE_MAX_ATTEMPTS_PER_EXAMPLE = 80

# Keep WDQS gentle. common_helpers already has retries/backoff; these values
# make long generation more robust rather than fast.
try:
    wd.timeout = max(getattr(wd, "timeout", 30), 45)
    wd.max_retries = max(getattr(wd, "max_retries", 4), 5)
except Exception:
    pass

print("output:", PEOPLE_OUTPUT_PATH.resolve())
print("audit:", PEOPLE_AUDIT_PATH.resolve())
print("checkpoint:", PEOPLE_CHECKPOINT_PATH.resolve())
print("targets:", PEOPLE_TARGET_PER_LEVEL, "total=", sum(PEOPLE_TARGET_PER_LEVEL.values()))


output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_fast_clean.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_fast_clean_audit.json
checkpoint: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_fast_clean_checkpoint.json
targets: {'L1': 0, 'L2': 0, 'L3': 35, 'L4': 0, 'L5': 35} total= 70


## 3. Domain vocabulary

Occupations, countries, fields, awards, and language lookup tables used by all templates.


In [3]:
Q_HUMAN = "Q5"
Q_MALE = "Q6581097"
Q_FEMALE = "Q6581072"

SEXES = {
    "male": {"qid": Q_MALE, "en": "male", "ru": "мужской"},
    "female": {"qid": Q_FEMALE, "en": "female", "ru": "женский"},
}

# Occupations are matched directly with P106. The public prompt says
# "с профессией X", so using hidden P106/P279* subclass closure would make
# golds semantically broader than the query text.
OCCUPATIONS = {
    "actor": {"qid": "Q33999", "en": "actor", "ru": "актёр"},
    "film director": {"qid": "Q2526255", "en": "film director", "ru": "кинорежиссёр"},
    "writer": {"qid": "Q36180", "en": "writer", "ru": "писатель"},
    "poet": {"qid": "Q49757", "en": "poet", "ru": "поэт"},
    "journalist": {"qid": "Q1930187", "en": "journalist", "ru": "журналист"},
    "singer": {"qid": "Q177220", "en": "singer", "ru": "певец"},
    "composer": {"qid": "Q36834", "en": "composer", "ru": "композитор"},
    "painter": {"qid": "Q1028181", "en": "painter", "ru": "художник-живописец"},
    "architect": {"qid": "Q42973", "en": "architect", "ru": "архитектор"},
    "politician": {"qid": "Q82955", "en": "politician", "ru": "политик"},
    "lawyer": {"qid": "Q40348", "en": "lawyer", "ru": "юрист"},
    "association football player": {"qid": "Q937857", "en": "association football player", "ru": "футболист"},
    "basketball player": {"qid": "Q3665646", "en": "basketball player", "ru": "баскетболист"},
    "tennis player": {"qid": "Q10833314", "en": "tennis player", "ru": "теннисист"},
    "mathematician": {"qid": "Q170790", "en": "mathematician", "ru": "математик"},
    "physicist": {"qid": "Q169470", "en": "physicist", "ru": "физик"},
    "chemist": {"qid": "Q593644", "en": "chemist", "ru": "химик"},
    "biologist": {"qid": "Q864503", "en": "biologist", "ru": "биолог"},
    "philosopher": {"qid": "Q4964182", "en": "philosopher", "ru": "философ"},
    "economist": {"qid": "Q188094", "en": "economist", "ru": "экономист"},
    "historian": {"qid": "Q201788", "en": "historian", "ru": "историк"},
}

COUNTRIES = {
    "United States": {"qid": "Q30", "en": "United States", "ru": "США"},
    "United Kingdom": {"qid": "Q145", "en": "United Kingdom", "ru": "Великобритания"},
    "France": {"qid": "Q142", "en": "France", "ru": "Франция"},
    "Germany": {"qid": "Q183", "en": "Germany", "ru": "Германия"},
    "Italy": {"qid": "Q38", "en": "Italy", "ru": "Италия"},
    "Spain": {"qid": "Q29", "en": "Spain", "ru": "Испания"},
    "Russia": {"qid": "Q159", "en": "Russia", "ru": "Россия"},
    "Japan": {"qid": "Q17", "en": "Japan", "ru": "Япония"},
    "China": {"qid": "Q148", "en": "China", "ru": "Китай"},
    "Canada": {"qid": "Q16", "en": "Canada", "ru": "Канада"},
    "Australia": {"qid": "Q408", "en": "Australia", "ru": "Австралия"},
    "India": {"qid": "Q668", "en": "India", "ru": "Индия"},
    "Poland": {"qid": "Q36", "en": "Poland", "ru": "Польша"},
    "Netherlands": {"qid": "Q55", "en": "Netherlands", "ru": "Нидерланды"},
    "Sweden": {"qid": "Q34", "en": "Sweden", "ru": "Швеция"},
    "Brazil": {"qid": "Q155", "en": "Brazil", "ru": "Бразилия"},
    "Argentina": {"qid": "Q414", "en": "Argentina", "ru": "Аргентина"},
}

FIELDS = {
    # Broad/root fields are kept for validation of old records, but the generator
    # should not pair them with the matching occupation as a user-facing
    # criterion, e.g. philosopher + philosophy or physicist + physics.
    "mathematics": {"qid": "Q395", "en": "mathematics", "ru": "математика"},
    "physics": {"qid": "Q413", "en": "physics", "ru": "физика"},
    "chemistry": {"qid": "Q2329", "en": "chemistry", "ru": "химия"},
    "medicine": {"qid": "Q11190", "en": "medicine", "ru": "медицина"},
    "economics": {"qid": "Q8134", "en": "economics", "ru": "экономика"},
    "literature": {"qid": "Q8242", "en": "literature", "ru": "литература"},
    "philosophy": {"qid": "Q5891", "en": "philosophy", "ru": "философия"},
    "astronomy": {"qid": "Q333", "en": "astronomy", "ru": "астрономия"},
    "biology": {"qid": "Q420", "en": "biology", "ru": "биология"},
    "computer science": {"qid": "Q21198", "en": "computer science", "ru": "информатика"},
    "history": {"qid": "Q309", "en": "history", "ru": "история"},
    "psychology": {"qid": "Q9418", "en": "psychology", "ru": "психология"},
    "sociology": {"qid": "Q21201", "en": "sociology", "ru": "социология"},

    # Allowed specialized fields for the occupation+field template. These make
    # the field constraint meaningful instead of restating the occupation.
    "number theory": {"qid": "Q12479", "en": "number theory", "ru": "теория чисел"},
    "topology": {"qid": "Q42989", "en": "topology", "ru": "топология"},
    "algebra": {"qid": "Q3968", "en": "algebra", "ru": "алгебра"},
    "geometry": {"qid": "Q8087", "en": "geometry", "ru": "геометрия"},
    "statistics": {"qid": "Q12483", "en": "statistics", "ru": "статистика"},
    "mathematical analysis": {"qid": "Q7754", "en": "mathematical analysis", "ru": "математический анализ"},
    "quantum mechanics": {"qid": "Q944", "en": "quantum mechanics", "ru": "квантовая механика"},
    "particle physics": {"qid": "Q18334", "en": "particle physics", "ru": "физика элементарных частиц"},
    "astrophysics": {"qid": "Q162", "en": "astrophysics", "ru": "астрофизика"},
    "condensed matter physics": {"qid": "Q214781", "en": "condensed matter physics", "ru": "физика конденсированного состояния"},
    "theoretical physics": {"qid": "Q18362", "en": "theoretical physics", "ru": "теоретическая физика"},
    "optics": {"qid": "Q14620", "en": "optics", "ru": "оптика"},
    "organic chemistry": {"qid": "Q11351", "en": "organic chemistry", "ru": "органическая химия"},
    "inorganic chemistry": {"qid": "Q11165", "en": "inorganic chemistry", "ru": "неорганическая химия"},
    "analytical chemistry": {"qid": "Q2346", "en": "analytical chemistry", "ru": "аналитическая химия"},
    "physical chemistry": {"qid": "Q11372", "en": "physical chemistry", "ru": "физическая химия"},
    "biochemistry": {"qid": "Q7094", "en": "biochemistry", "ru": "биохимия"},
    "genetics": {"qid": "Q7162", "en": "genetics", "ru": "генетика"},
    "ecology": {"qid": "Q7150", "en": "ecology", "ru": "экология"},
    "molecular biology": {"qid": "Q7202", "en": "molecular biology", "ru": "молекулярная биология"},
    "microbiology": {"qid": "Q7193", "en": "microbiology", "ru": "микробиология"},
    "evolutionary biology": {"qid": "Q12870", "en": "evolutionary biology", "ru": "эволюционная биология"},
    "botany": {"qid": "Q441", "en": "botany", "ru": "ботаника"},
    "macroeconomics": {"qid": "Q39680", "en": "macroeconomics", "ru": "макроэкономика"},
    "microeconomics": {"qid": "Q133192", "en": "microeconomics", "ru": "микроэкономика"},
    "econometrics": {"qid": "Q194940", "en": "econometrics", "ru": "эконометрика"},
}

AWARDS = {
    "Nobel Prize": {"qid": "Q7191", "en": "Nobel Prize", "ru": "Нобелевская премия"},
    "Academy Award": {"qid": "Q19020", "en": "Academy Award", "ru": "Оскар"},
    "Grammy Award": {"qid": "Q41254", "en": "Grammy Award", "ru": "Грэмми"},
    "Pulitzer Prize": {"qid": "Q46525", "en": "Pulitzer Prize", "ru": "Пулитцеровская премия"},
    "Fields Medal": {"qid": "Q28835", "en": "Fields Medal", "ru": "Филдсовская премия"},
    "Wolf Prize": {"qid": "Q189153", "en": "Wolf Prize", "ru": "премия Вольфа"},
    "BAFTA Award": {"qid": "Q732997", "en": "BAFTA Award", "ru": "BAFTA"},
    "César Award": {"qid": "Q238177", "en": "César Award", "ru": "Сезар"},
    "Booker Prize": {"qid": "Q160082", "en": "Booker Prize", "ru": "Букеровская премия"},
}

LANGUAGES = {
    "English": {"qid": "Q1860", "en": "English", "ru": "английский язык"},
    "French": {"qid": "Q150", "en": "French", "ru": "французский язык"},
    "Russian": {"qid": "Q7737", "en": "Russian", "ru": "русский язык"},
    "Spanish": {"qid": "Q1321", "en": "Spanish", "ru": "испанский язык"},
    "German": {"qid": "Q188", "en": "German", "ru": "немецкий язык"},
    "Italian": {"qid": "Q652", "en": "Italian", "ru": "итальянский язык"},
    "Japanese": {"qid": "Q5287", "en": "Japanese", "ru": "японский язык"},
    "Portuguese": {"qid": "Q5146", "en": "Portuguese", "ru": "португальский язык"},
    "Polish": {"qid": "Q809", "en": "Polish", "ru": "польский язык"},
    "Dutch": {"qid": "Q7411", "en": "Dutch", "ru": "нидерландский язык"},
}

FOOTBALL_POSITIONS = {
    "goalkeeper": {"qid": "Q201330", "en": "goalkeeper", "ru": "вратарь"},
    "defender": {"qid": "Q336286", "en": "defender", "ru": "защитник"},
    "midfielder": {"qid": "Q193592", "en": "midfielder", "ru": "полузащитник"},
    "forward": {"qid": "Q280658", "en": "forward", "ru": "нападающий"},
}

# Work genres/forms used only in L5 bridge templates through a person's notable
# work or created work. They are intentionally broad but still useful for multi-hop.
WORK_GENRES = {
    "science fiction": {"qid": "Q24925", "en": "science fiction", "ru": "научная фантастика"},
    "crime fiction": {"qid": "Q959790", "en": "crime fiction", "ru": "детективная литература"},
    "drama": {"qid": "Q130232", "en": "drama", "ru": "драма"},
    "comedy": {"qid": "Q40831", "en": "comedy", "ru": "комедия"},
    "rock music": {"qid": "Q11399", "en": "rock music", "ru": "рок-музыка"},
    "classical music": {"qid": "Q9730", "en": "classical music", "ru": "классическая музыка"},
}

# Work-link templates below do not use P1412 (languages spoken by a person).
# They use P407 on a concrete work, which is much less noisy and gives a real
# person -> work -> work-language / genre bridge.
LITERARY_WORK_GENRES = {
    "science fiction": WORK_GENRES["science fiction"],
    "crime fiction": WORK_GENRES["crime fiction"],
    "drama": WORK_GENRES["drama"],
    "comedy": WORK_GENRES["comedy"],
}

FILM_GENRES = {
    "drama": {"qid": "Q130232", "en": "drama", "ru": "драма"},
    "comedy": {"qid": "Q40831", "en": "comedy", "ru": "комедия"},
    "science fiction": {"qid": "Q24925", "en": "science fiction", "ru": "научная фантастика"},
    "action film": {"qid": "Q188473", "en": "action film", "ru": "боевик"},
    "thriller film": {"qid": "Q2484376", "en": "thriller film", "ru": "триллер"},
}

MAJOR_FILM_COUNTRIES = ["United States", "United Kingdom", "France", "Germany", "Italy", "Spain", "Japan", "China", "India", "Canada", "Australia"]
MAJOR_EMPLOYER_COUNTRIES = ["United States", "United Kingdom", "France", "Germany", "Italy", "Japan", "Canada", "Australia", "Netherlands", "Sweden"]
MAJOR_PARTY_COUNTRIES = ["United States", "United Kingdom", "France", "Germany", "Italy", "Spain", "India", "Canada", "Australia", "Netherlands", "Sweden", "Brazil", "Argentina"]

# For related-person constraints such as film director / advisor birth ranges.
# These are intentionally modern enough to avoid historical P27 ambiguity but
# broad enough to keep L5 gold sets viable.
RELATED_PERSON_BIRTH_RANGES = [(1925, 1949), (1930, 1959), (1940, 1969), (1950, 1979)]

BIRTH_RANGES = [
    (1800, 1849), (1850, 1899), (1900, 1924), (1925, 1949),
    (1950, 1969), (1970, 1989), (1990, 2005),
]

RECENT_BIRTH_RANGES = [(1930, 1959), (1950, 1979), (1960, 1989), (1970, 1999)]

# Do not use bare P27/citizenship constraints for historical people.
# Wikidata often stores modern country-of-citizenship values without temporal
# qualifiers, which can produce anachronistic golds for pre-modern persons.
# Therefore every template that uses person citizenship also adds one of these
# explicit birth-year ranges to the user query and to SPARQL.
MODERN_CITIZENSHIP_BIRTH_RANGES = [(1950, 1969), (1970, 1989), (1990, 2005)]
SCIENTIST_OCCS = ["mathematician", "physicist", "chemist", "biologist", "economist"]
CREATIVE_OCCS = ["actor", "film director", "writer", "poet", "singer", "composer", "painter"]
POLITICAL_OCCS = ["politician", "lawyer", "journalist"]
SPORT_OCCS = ["association football player", "basketball player", "tennis player"]

# Some semantically coherent award groups reduce invalid random combinations.
AWARDS_BY_OCCUPATION = {
    "actor": ["Academy Award", "BAFTA Award", "César Award"],
    "film director": ["Academy Award", "BAFTA Award", "César Award"],
    "writer": ["Pulitzer Prize", "Booker Prize", "Nobel Prize"],
    "poet": ["Pulitzer Prize", "Nobel Prize"],
    "journalist": ["Pulitzer Prize"],
    "singer": ["Grammy Award"],
    "composer": ["Grammy Award", "Academy Award"],
    "mathematician": ["Fields Medal", "Wolf Prize"],
    "physicist": ["Nobel Prize", "Wolf Prize"],
    "chemist": ["Nobel Prize", "Wolf Prize"],
    "biologist": ["Nobel Prize", "Wolf Prize"],
    "economist": ["Nobel Prize", "Wolf Prize"],
}


# Public constraints should use concrete named awards, e.g.
# {"award_received": "Academy Award"}, not {"award_group": ...}.
# Internally, some famous awards are represented in Wikidata as families with
# official category awards (Oscar categories, Nobel categories, Grammy categories,
# Pulitzer categories, etc.).  For these keys the SPARQL matcher accepts either
# direct P166=root award or an official sub-award/category connected by
# P31/P279/P361.  This keeps the prompt natural ("received the Academy Award")
# without treating arbitrary related prizes as valid.
PEOPLE_AWARD_FAMILY_MATCH_KEYS = {"Academy Award", "Grammy Award", "BAFTA Award", "César Award", "Pulitzer Prize", "Nobel Prize", "Wolf Prize"}
# Backward-compatible alias used only to reject old records with award_group.
PEOPLE_GROUP_AWARD_KEYS = PEOPLE_AWARD_FAMILY_MATCH_KEYS

# Root fields are used only to reject redundant generated/old records such as
# philosopher + philosophy, historian + history, writer + literature, etc.
ROOT_FIELD_BY_OCCUPATION = {
    "mathematician": "mathematics",
    "physicist": "physics",
    "chemist": "chemistry",
    "biologist": "biology",
    "economist": "economics",
    "philosopher": "philosophy",
    "historian": "history",
    "writer": "literature",
    "poet": "literature",
}

# Only these occupation->field pairs are allowed in the generated
# occupation+field template. They represent real specializations/directions,
# not a restatement of the profession.
FIELD_SPECIALIZATIONS_BY_OCCUPATION = {
    "mathematician": ["number theory", "topology", "algebra", "geometry", "statistics", "mathematical analysis"],
    "physicist": ["quantum mechanics", "particle physics", "astrophysics", "condensed matter physics", "theoretical physics", "optics"],
    "chemist": ["organic chemistry", "inorganic chemistry", "analytical chemistry", "physical chemistry", "biochemistry"],
    "biologist": ["genetics", "ecology", "molecular biology", "microbiology", "evolutionary biology", "botany"],
    "economist": ["macroeconomics", "microeconomics", "econometrics"],
}

# Backward-compatible name for older helper code. Do not put root fields here.
FIELD_BY_OCCUPATION = FIELD_SPECIALIZATIONS_BY_OCCUPATION

# QID-based versions of the field-policy tables.  Labels are used only for
# natural-language prompts; validation of occupation/field compatibility is
# based on Wikidata entity IDs whenever metadata contains them.
ROOT_FIELD_QID_BY_OCCUPATION_QID = {
    OCCUPATIONS[occ_key]["qid"]: FIELDS[field_key]["qid"]
    for occ_key, field_key in ROOT_FIELD_BY_OCCUPATION.items()
    if occ_key in OCCUPATIONS and field_key in FIELDS
}

SPECIALIZED_FIELD_QIDS_BY_OCCUPATION_QID = {
    OCCUPATIONS[occ_key]["qid"]: {FIELDS[field_key]["qid"] for field_key in field_keys if field_key in FIELDS}
    for occ_key, field_keys in FIELD_SPECIALIZATIONS_BY_OCCUPATION.items()
    if occ_key in OCCUPATIONS
}

SPECIALIZED_OCCUPATION_FIELD_QID_PAIRS = {
    (occupation_qid, field_qid)
    for occupation_qid, field_qids in SPECIALIZED_FIELD_QIDS_BY_OCCUPATION_QID.items()
    for field_qid in field_qids
}

ROOT_OCCUPATION_FIELD_QID_PAIRS = {
    (occupation_qid, field_qid)
    for occupation_qid, field_qid in ROOT_FIELD_QID_BY_OCCUPATION_QID.items()
}

print("Vocabulary loaded:", len(OCCUPATIONS), "occupations,", len(COUNTRIES), "countries")


Vocabulary loaded: 21 occupations, 17 countries


## 4. SPARQL / Wikidata helpers

In [4]:
def _p_choice(rng: random.Random, keys: Sequence[str]) -> str:
    return rng.choice(list(keys))


def _p_get(table: Dict[str, Dict[str, str]], key: str) -> Dict[str, str]:
    if key not in table:
        raise KeyError(key)
    return table[key]


def _p_clean_constraints(d: Dict[str, Any]) -> Dict[str, Any]:
    """Remove None/empty values and keep constraints JSON-clean."""
    out: Dict[str, Any] = {}
    for k, v in (d or {}).items():
        if v is None:
            continue
        if isinstance(v, str) and not v.strip():
            continue
        if isinstance(v, (list, tuple)) and len(v) == 0:
            continue
        if isinstance(v, dict):
            vv = _p_clean_constraints(v)
            if vv:
                out[k] = vv
            continue
        out[k] = v
    return out


# These suffixes/keys are internal generation metadata. They must NOT leak into
# the public `constraints` field. This mirrors the good cinema/geo JSONL style:
# constraints are clean and human-readable; QIDs and WDQS property paths live in
# gold_collection_meta.bridge_meta / SPARQL / ASK validators.
_PEOPLE_INTERNAL_CONSTRAINT_SUFFIXES = ("_qid", "_path", "_match")
_PEOPLE_INTERNAL_CONSTRAINT_KEYS = {
    "qid",
    "path",
    "match",
    "property_path",
    "wdqs_path",
}


def _p_is_internal_constraint_key(key: str) -> bool:
    k = str(key or "")
    return k in _PEOPLE_INTERNAL_CONSTRAINT_KEYS or any(k.endswith(suf) for suf in _PEOPLE_INTERNAL_CONSTRAINT_SUFFIXES)


def _p_public_constraints(d: Dict[str, Any]) -> Dict[str, Any]:
    """Keep only the clean user-facing constraint fields used in query text."""
    out: Dict[str, Any] = {}
    for k, v in (d or {}).items():
        if _p_is_internal_constraint_key(k):
            continue
        if v is None:
            continue
        if isinstance(v, str) and not v.strip():
            continue
        if isinstance(v, (list, tuple)) and len(v) == 0:
            continue
        if isinstance(v, dict):
            vv = _p_public_constraints(v)
            if vv:
                out[k] = vv
            continue
        out[k] = v
    return _p_clean_constraints(out)


def _p_constraint_metadata(d: Dict[str, Any]) -> Dict[str, Any]:
    """Extract QIDs/property paths from raw constraints into metadata."""
    entities: Dict[str, Any] = {}
    property_paths: Dict[str, Any] = {}
    other_internal: Dict[str, Any] = {}

    for k, v in (d or {}).items():
        if v is None:
            continue
        if isinstance(v, str) and not v.strip():
            continue
        if k.endswith("_qid") or k == "qid":
            entities[k] = v
        elif k.endswith("_path") or k.endswith("_match") or k in {"path", "match", "property_path", "wdqs_path"}:
            property_paths[k] = v
        elif isinstance(v, dict):
            nested = _p_constraint_metadata(v)
            if nested:
                other_internal[k] = nested

    meta: Dict[str, Any] = {}
    if entities:
        meta["constraint_entities"] = entities
    if property_paths:
        meta["constraint_property_paths"] = property_paths
    if other_internal:
        meta["nested_constraint_metadata"] = other_internal
    return meta


def _p_constraints_have_internal_metadata(d: Dict[str, Any]) -> bool:
    for k, v in (d or {}).items():
        if _p_is_internal_constraint_key(k):
            return True
        if isinstance(v, dict) and _p_constraints_have_internal_metadata(v):
            return True
    return False


def _p_cyrillic(s: str) -> bool:
    return bool(re.search(r"[А-Яа-яЁё]", str(s or "")))


def _p_json_sig(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def _p_rows_from_select(data: dict) -> List[Dict[str, str]]:
    return rows_from_select(data)


def _p_uri_to_qid(uri: Optional[str]) -> Optional[str]:
    return uri_to_qid(uri or "")


def _p_occupation_line(occ_qid: str, person_var: str = "person") -> str:
    # Direct occupation only: matches the natural prompt phrase "с профессией X".
    return f"?{person_var} wdt:P106 wd:{occ_qid} ."


def _p_birth_range_lines(start_year: int, end_year: int, var: str = "person") -> List[str]:
    """Birth range with explicit Wikidata time precision.

    Important: `wdt:P569 ?birthDate` alone can admit imprecise Wikidata values
    such as "20th century" into year-range filters. We use the full value node
    and require at least year precision (`wikibase:timePrecision >= 9`).
    The user-facing end_year is inclusive; SPARQL uses exclusive next year.
    """
    return [
        f"?{var} p:P569 ?birthStatement .",
        "?birthStatement psv:P569 ?birthValueNode .",
        "?birthValueNode wikibase:timeValue ?birthDate ; wikibase:timePrecision ?birthPrecision .",
        f"FILTER(?birthPrecision >= {int(PEOPLE_MIN_BIRTH_DATE_PRECISION)})",
        f'FILTER(?birthDate >= "{int(start_year)}-01-01"^^xsd:dateTime && ?birthDate < "{int(end_year) + 1}-01-01"^^xsd:dateTime)'
    ]


def _p_birth_range_lines_scoped(start_year: int, end_year: int, *, var: str = "person", scope: Optional[str] = None) -> List[str]:
    """Birth range helper safe to use multiple times in one SPARQL query.

    _p_birth_range_lines historically uses generic ?birthDate variables. That is
    fine when a template filters only the answer person or only one related
    person. L5 templates often need both person and director/advisor/spouse
    dates, so this helper gives every date filter its own variables.
    """
    safe = re.sub(r"[^A-Za-z0-9_]", "", scope or var or "person")
    safe = safe[:1].lower() + safe[1:] if safe else "person"
    stmt = f"?{safe}BirthStatement"
    node = f"?{safe}BirthValueNode"
    date = f"?{safe}BirthDate"
    prec = f"?{safe}BirthPrecision"
    return [
        f"?{var} p:P569 {stmt} .",
        f"{stmt} psv:P569 {node} .",
        f"{node} wikibase:timeValue {date} ; wikibase:timePrecision {prec} .",
        f"FILTER({prec} >= {int(PEOPLE_MIN_BIRTH_DATE_PRECISION)})",
        f'FILTER({date} >= "{int(start_year)}-01-01"^^xsd:dateTime && {date} < "{int(end_year) + 1}-01-01"^^xsd:dateTime)'
    ]


def _p_award_where_lines_for_var(award_key: str, *, var: str = "person", value_var: Optional[str] = None) -> List[str]:
    """Award matcher for answer person or a related human variable.

    Mirrors _p_award_where_lines but avoids hard-coding ?person and avoids
    variable collisions when an L5 template filters a spouse/parent/cast member.
    """
    qids = _p_award_family_member_qids(award_key)
    if len(qids) == 1:
        return [f"?{var} wdt:P166 wd:{qids[0]} ."]
    vv = value_var or f"?{var}AwardReceived"
    values = " ".join(f"wd:{qid}" for qid in qids)
    return [
        f"?{var} wdt:P166 {vv} .",
        f"VALUES {vv} {{ {values} }}",
    ]


def _p_build_select_query(
    where_lines: List[str],
    *,
    answer_var: str = "person",
    limit: int = PEOPLE_WDQS_LIMIT_DEFAULT,
    extra_select_vars: Optional[List[str]] = None,
) -> str:
    extra = " ".join(extra_select_vars or [])
    where = "\n      ".join(where_lines)
    # We require English labels for stable gold_answer_labels_en and use RU label
    # with EN fallback for user-facing Russian gold labels.
    return f"""
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
SELECT DISTINCT ?{answer_var} ?{answer_var}LabelEn ?{answer_var}LabelRu {extra} WHERE {{
      ?{answer_var} wdt:P31 wd:{Q_HUMAN} .
      {where}
      ?{answer_var} rdfs:label ?{answer_var}LabelEn FILTER(LANG(?{answer_var}LabelEn) = "en") .
      OPTIONAL {{ ?{answer_var} rdfs:label ?{answer_var}LabelRu FILTER(LANG(?{answer_var}LabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
""".strip()


def _p_build_ask_query(where_lines: List[str], *, answer_var: str = "person") -> str:
    where = "\n      ".join(where_lines)
    return f"""
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
# WDQS-only validator. All constraints are checked directly in Wikidata.
ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{answer_var})
      ?{answer_var} wdt:P31 wd:{Q_HUMAN} .
      {where}
    }}
""".strip()



def _p_collect_gold(
    *,
    sparql_query: str,
    answer_var: str = "person",
    limit: int = PEOPLE_WDQS_LIMIT_DEFAULT,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    data = wd.sparql_select(sparql_query, use_cache=True)
    rows = _p_rows_from_select(data)

    seen_qids = set()
    items: List[Dict[str, Any]] = []
    dropped_no_qid = 0
    dropped_no_en_label = 0
    dropped_duplicate_qid = 0
    ru_label_count = 0
    en_fallback_count = 0

    for row in rows:
        qid = _p_uri_to_qid(row.get(answer_var))
        if not qid:
            dropped_no_qid += 1
            continue
        if qid in seen_qids:
            dropped_duplicate_qid += 1
            continue

        label_en = (row.get(f"{answer_var}LabelEn") or "").strip()
        label_ru_raw = (row.get(f"{answer_var}LabelRu") or "").strip()

        if not label_en:
            dropped_no_en_label += 1
            continue

        label_ru = label_ru_raw or label_en
        if label_ru_raw:
            ru_label_count += 1
        else:
            en_fallback_count += 1

        seen_qids.add(qid)
        item: Dict[str, Any] = {"qid": qid, "label_en": label_en, "label_ru": label_ru}
        for k, v in row.items():
            if k not in {answer_var, f"{answer_var}LabelEn", f"{answer_var}LabelRu"}:
                item[k] = v
        items.append(item)

    # Cinema-style metadata shape: keep compact collection stats in gold_collection_meta,
    # put low-level bridge/QID/path details into bridge_meta, and do not duplicate
    # template fields that already exist at the top level.
    meta = {
        "source": "wikidata_sparql",
        "match_key": "wikidata_qid",
        "candidates_from_wdqs": len(rows),
        "wdqs_candidate_limit": int(limit),
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": len(items),
        "dropped_reasons": {
            "missing_qid": dropped_no_qid,
            "missing_en_label": dropped_no_en_label,
            "duplicate_qid": dropped_duplicate_qid,
        },
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en_label,
        "label_sources": {
            "ru_label": ru_label_count,
            "en_fallback_for_ru": en_fallback_count,
        },
        "note": "Gold is collected directly from Wikidata SPARQL. Labels are not used for matching; English label is required, Russian label uses English fallback when absent.",
        "gold_may_be_incomplete_due_to_wdqs_limit": len(rows) >= int(limit),
    }
    return items, meta


def _p_local_validator(clean_constraints: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "match_key": "wikidata_qid",
        "applies_after": "ask_validator_sparql",
        "filters": clean_constraints,
        "label_matching_used": False,
        "note": "Use ask_validator_sparql for all Wikidata constraints; no external local validator is required for people-domain examples.",
    }


def _p_is_advanced(complexity: str) -> bool:
    return complexity in {"L3", "L4", "L5"}


def _p_has_duplicate_answer_labels(gold_items: List[Dict[str, Any]]) -> bool:
    labels_en = [(it.get("label_en") or "").strip().casefold() for it in gold_items]
    labels_ru = [(it.get("label_ru") or "").strip().casefold() for it in gold_items]
    return len(labels_en) != len(set(labels_en)) or len(labels_ru) != len(set(labels_ru))


def _p_passes_label_quality(gold_items: List[Dict[str, Any]], meta: Dict[str, Any]) -> bool:
    # English labels are mandatory in _p_build_select_query. Russian labels are
    # optional; when missing, gold_answer_labels_ru uses the English label. This
    # matches the geo/cinema schema style while avoiding any non-ru/non-en label
    # fallback.
    if not gold_items:
        return False
    if PEOPLE_REJECT_DUPLICATE_ANSWER_LABELS and _p_has_duplicate_answer_labels(gold_items):
        return False
    return True


def _p_finalize_wdqs_example(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    q_ru: str,
    q_en: str,
    constraints: Dict[str, Any],
    where_lines: List[str],
    requested_count: int,
    extra_select_vars: Optional[List[str]] = None,
    gold_limit: int = PEOPLE_GOLD_LIMIT,
    bridge_meta: Optional[Dict[str, Any]] = None,
) -> Optional[BenchmarkExample]:
    raw_constraints = _p_clean_constraints(constraints)
    clean_constraints = _p_public_constraints(raw_constraints)
    constraint_meta = _p_constraint_metadata(raw_constraints)

    # Constraints must be English-only and public/user-facing only. Technical
    # QIDs/property paths are kept in gold_collection_meta instead.
    if _p_cyrillic(_p_json_sig(clean_constraints)):
        raise ValueError(f"Cyrillic leaked into constraints for {template_id}: {clean_constraints}")
    if _p_constraints_have_internal_metadata(clean_constraints):
        raise ValueError(f"Internal metadata leaked into constraints for {template_id}: {clean_constraints}")

    sparql = _p_build_select_query(where_lines, limit=PEOPLE_WDQS_LIMIT_DEFAULT, extra_select_vars=extra_select_vars)
    ask = _p_build_ask_query(where_lines)

    try:
        gold_items, meta = _p_collect_gold(
            sparql_query=sparql,
            answer_var="person",
            limit=PEOPLE_WDQS_LIMIT_DEFAULT,
        )
    except Exception as e:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
                return None

    n = len(gold_items)
    min_gold = max(PEOPLE_MIN_GOLD_BY_LEVEL.get(complexity, 5), int(requested_count))
    max_gold = PEOPLE_MAX_GOLD_BY_LEVEL.get(complexity, PEOPLE_GOLD_LIMIT)

    # Accepted records should have complete, compact, and unambiguous gold sets.
    if n < min_gold:
        return None
    if n > max_gold:
        return None
    if n > gold_limit:
        return None
    if meta.get("gold_may_be_incomplete_due_to_wdqs_limit"):
        return None
    if not _p_passes_label_quality(gold_items, meta):
        return None

    final_items = gold_items[:gold_limit]
    truncated_by_local_limit = len(final_items) < n

    bridge_meta_final: Dict[str, Any] = {}
    if bridge_meta:
        bridge_meta_final.update(bridge_meta)
    if constraint_meta:
        bridge_meta_final.update(constraint_meta)

    meta.update({
        "constraints_are_wdqs_only": True,
        "gold_limit": int(gold_limit),
        "gold_returned": len(final_items),
        "gold_total_before_limit": n,
        "gold_truncated_by_local_limit": truncated_by_local_limit,
        # Keep QIDs/property paths in bridge_meta, mirroring the cinema JSONL convention
        # where low-level bridge information lives in gold_collection_meta.bridge_meta.
        "bridge_meta": bridge_meta_final or None,
    })

    return BenchmarkExample(
        id=f"people_{complexity.lower()}_{idx:04d}",
        domain=PEOPLE_DOMAIN,
        complexity=complexity,
        query_text_ru=q_ru,
        query_text_en=q_en,
        constraints=clean_constraints,
        requested_count=int(requested_count),
        gold_answer_qids=[it["qid"] for it in final_items],
        gold_answer_labels_ru=[it["label_ru"] for it in final_items],
        gold_answer_labels_en=[it["label_en"] for it in final_items],
        sparql_query=sparql,
        created_at=utc_now_z(),
        is_advanced=_p_is_advanced(complexity),
        template_id=template_id,
        template_family=template_family,
        gold_truncated=truncated_by_local_limit,
        ask_validator_sparql=ask,
        local_validator=_p_local_validator(clean_constraints),
        gold_collection_meta=meta,
    )

print("✅ SPARQL helpers loaded")

✅ SPARQL helpers loaded


## 5. RU/EN query text helpers

In [5]:
def _p_head(requested_count: int) -> Tuple[str, str]:
    return f"Назови {requested_count} известных персон", f"Name {requested_count} well-known people"


def _ru_join(parts: List[str]) -> str:
    return ", ".join([p for p in parts if p]) + "."


def _en_join(parts: List[str]) -> str:
    if not parts:
        return "."
    return ", ".join(parts) + "."


def _birth_phrase_ru(start: int, end: int) -> str:
    return f"родившихся с {int(start)} по {int(end)} год"


def _birth_phrase_en(start: int, end: int) -> str:
    return f"born from {int(start)} to {int(end)}"


def _advisor_birth_phrase_ru(start: int, end: int) -> str:
    return f"у которых научный руководитель родился с {int(start)} по {int(end)} год"


def _advisor_birth_phrase_en(start: int, end: int) -> str:
    return f"whose doctoral advisor was born from {int(start)} to {int(end)}"


def _occ_phrase_ru(occ_key: str) -> str:
    return f"с профессией «{OCCUPATIONS[occ_key]['ru']}»"


def _occ_phrase_en(occ_key: str) -> str:
    return f"whose occupation is {OCCUPATIONS[occ_key]['en']}"


def _country_phrase_ru(country_key: str, prefix: str = "с гражданством") -> str:
    return f"{prefix}: {COUNTRIES[country_key]['ru']}"


def _country_phrase_en(country_key: str, prefix: str = "with citizenship") -> str:
    return f"{prefix}: {COUNTRIES[country_key]['en']}"


def _sex_phrase_ru(sex_key: str) -> str:
    return f"пол: {SEXES[sex_key]['ru']}"


def _sex_phrase_en(sex_key: str) -> str:
    return f"gender: {SEXES[sex_key]['en']}"


def _p_award_uses_family_match(award_key: str) -> bool:
    return award_key in PEOPLE_AWARD_FAMILY_MATCH_KEYS


# Hard fallback lists are intentionally small and conservative.  They are used
# only if the one-time WDQS expansion below fails.  The main path still expands
# award families from Wikidata once and then reuses fast VALUES lists in all
# people queries.
PEOPLE_AWARD_FAMILY_QIDS_FALLBACK = {
    "Nobel Prize": [
        "Q7191",   # Nobel Prize
        "Q38104",  # Nobel Prize in Physics
        "Q44585",  # Nobel Prize in Chemistry
        "Q37922",  # Nobel Prize in Literature
        "Q35637",  # Nobel Peace Prize
        "Q80061",  # Nobel Prize in Physiology or Medicine
        "Q47170",  # Nobel Memorial Prize in Economic Sciences
    ],
    "Academy Award": ["Q19020"],
    "Grammy Award": ["Q41254"],
    "Pulitzer Prize": ["Q46525"],
    "Wolf Prize": ["Q189153"],
}
PEOPLE_AWARD_FAMILY_QIDS_CACHE: Dict[str, List[str]] = {}


def _p_award_family_member_qids(award_key: str) -> List[str]:
    """Return concrete award QIDs for matching a public award_received criterion.

    v15 used a property path inside every generated people query:
        ?awardReceived (wdt:P31|wdt:P279|wdt:P361)+ wd:ROOT
    That is correct but very slow when generation rejects many candidates.  v16
    resolves the family once and then uses VALUES in candidate queries.
    """
    root_qid = AWARDS[award_key]["qid"]
    if not _p_award_uses_family_match(award_key):
        return [root_qid]
    if award_key in PEOPLE_AWARD_FAMILY_QIDS_CACHE:
        return PEOPLE_AWARD_FAMILY_QIDS_CACHE[award_key]

    qids: List[str] = [root_qid]
    try:
        expansion_query = f"""
SELECT DISTINCT ?award WHERE {{
  {{
    VALUES ?award {{ wd:{root_qid} }}
  }}
  UNION
  {{
    ?award (wdt:P31|wdt:P279|wdt:P361)+ wd:{root_qid} .
  }}
}}
LIMIT 500
""".strip()
        data = wd.sparql_select(expansion_query, use_cache=True)
        for row in _p_rows_from_select(data):
            qid = _p_uri_to_qid(row.get("award"))
            if qid and qid not in qids:
                qids.append(qid)
    except Exception:
        # Fall back to a conservative list; generation may produce fewer award
        # records, but it should not become slow or crash.
        qids = list(PEOPLE_AWARD_FAMILY_QIDS_FALLBACK.get(award_key, [root_qid]))

    if len(qids) <= 1 and award_key in PEOPLE_AWARD_FAMILY_QIDS_FALLBACK:
        qids = list(dict.fromkeys(PEOPLE_AWARD_FAMILY_QIDS_FALLBACK[award_key]))

    PEOPLE_AWARD_FAMILY_QIDS_CACHE[award_key] = qids
    return qids


def _p_award_where_lines(award_key: str) -> List[str]:
    """SPARQL lines for a public concrete award criterion.

    The prompt/constraints say award_received=<name>.  For awards that Wikidata
    stores as a family with categories, we match the root item plus resolved
    official sub-awards/categories through a fast VALUES clause.
    """
    qids = _p_award_family_member_qids(award_key)
    if len(qids) == 1:
        return [f"?person wdt:P166 wd:{qids[0]} ."]
    values = " ".join(f"wd:{qid}" for qid in qids)
    return [
        "?person wdt:P166 ?awardReceived .",
        f"VALUES ?awardReceived {{ {values} }}",
    ]


def _p_award_constraint_entries(award_key: str) -> Dict[str, Any]:
    award = AWARDS[award_key]
    return {
        "award_received": award["en"],
        "award_received_qid": award["qid"],
        "award_match": "P166/direct_or_official_subaward" if _p_award_uses_family_match(award_key) else "P166",
    }


def _award_phrase_ru(award_key: str) -> str:
    return f"получивших награду: {AWARDS[award_key]['ru']}"


def _award_phrase_en(award_key: str) -> str:
    return f"who received the {AWARDS[award_key]['en']}"


def _field_phrase_ru(field_key: str) -> str:
    return f"с областью деятельности «{FIELDS[field_key]['ru']}»"


def _field_phrase_en(field_key: str) -> str:
    return f"whose field of work is {FIELDS[field_key]['en']}"

print("✅ NLG helpers loaded")

✅ NLG helpers loaded


## 6. Template definitions

### 6a. L1/L2 direct citizenship and sex templates

In [ ]:

def _p_runtime_allowed_occupation_keys(complexity: str, occ_keys: Sequence[str]) -> List[str]:
    """Use current accepted records to avoid expensive WDQS calls for occupations
    that are already at the per-level quota.  If no runtime context is set, keep
    the original candidates.  Public constraints remain unchanged.
    """
    records_ctx = globals().get("PEOPLE_RUNTIME_RECORDS_FOR_QUOTAS") or []
    limit = PEOPLE_MAX_SAME_OCCUPATION_PER_LEVEL.get(complexity)
    if not records_ctx or limit is None:
        return list(occ_keys)
    allowed: List[str] = []
    for key in occ_keys:
        occ_label = OCCUPATIONS[key]["en"]
        if people_same_occupation_count(records_ctx, complexity, occ_label) < int(limit):
            allowed.append(key)
    return allowed


def _tpl_occ_citizenship(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L1":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = _p_choice(rng, list(OCCUPATIONS.keys()))
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    occ = OCCUPATIONS[occ_key]
    country = COUNTRIES[country_key]
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end)])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), _country_phrase_en(country_key), _birth_phrase_en(start, end)])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_occ_citizenship",
        template_family="occupation_citizenship", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
    )


def _tpl_occ_sex_citizenship(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L1", "L2"}:
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = _p_choice(rng, list(OCCUPATIONS.keys()))
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    sex_key = _p_choice(rng, list(SEXES.keys()))
    occ, country, sex = OCCUPATIONS[occ_key], COUNTRIES[country_key], SEXES[sex_key]
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        f"?person wdt:P21 wd:{sex['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), _country_phrase_ru(country_key), _sex_phrase_ru(sex_key), _birth_phrase_ru(start, end)])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), _country_phrase_en(country_key), _sex_phrase_en(sex_key), _birth_phrase_en(start, end)])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "sex_or_gender": sex["en"],
        "sex_or_gender_qid": sex["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_occ_sex_citizenship",
        template_family="occupation_citizenship_gender", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
    )


def _tpl_occ_birthrange_citizenship(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L2", "L3"}:
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = _p_choice(rng, list(OCCUPATIONS.keys()))
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    occ, country = OCCUPATIONS[occ_key], COUNTRIES[country_key]

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end)])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), _country_phrase_en(country_key), _birth_phrase_en(start, end)])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_occ_birthrange_citizenship",
        template_family="occupation_citizenship_birth_range", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
    )


### 6b. L2 award, field-of-work, and specialization templates

In [ ]:
def _tpl_occ_award(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    # Direct award constraints are L2 only. L3+ must contain an actual bridge.
    if complexity != "L2":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, list(AWARDS_BY_OCCUPATION.keys()))
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    award_key = _p_choice(rng, AWARDS_BY_OCCUPATION[occ_key])
    occ, award = OCCUPATIONS[occ_key], AWARDS[award_key]

    # Optional sex makes some L3 variants harder without changing the family.
    sex_key = _p_choice(rng, list(SEXES.keys())) if complexity == "L3" and rng.random() < 0.55 else None
    where = [
        _p_occupation_line(occ["qid"]),
        *_p_award_where_lines(award_key),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), _award_phrase_ru(award_key)]
    en_parts = [en_head, _occ_phrase_en(occ_key), _award_phrase_en(award_key)]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))
    q_ru, q_en = _ru_join(ru_parts), _en_join(en_parts)
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        **_p_award_constraint_entries(award_key),
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_occ_award",
        template_family="occupation_award", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
    )


def _tpl_field_country(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    # Direct occupation+field criteria are allowed only at L2, and only when the
    # field is a non-trivial specialization of the occupation. L3+ must use
    # explicit entity bridges instead of direct P101 filters.
    if complexity != "L2":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, list(FIELD_SPECIALIZATIONS_BY_OCCUPATION.keys()))
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    field_key = _p_choice(rng, FIELD_SPECIALIZATIONS_BY_OCCUPATION[occ_key])
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    occ, field, country = OCCUPATIONS[occ_key], FIELDS[field_key], COUNTRIES[country_key]

    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P101 wd:{field['qid']} .",
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), _field_phrase_ru(field_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end)]
    en_parts = [en_head, _occ_phrase_en(occ_key), _field_phrase_en(field_key), _country_phrase_en(country_key), _birth_phrase_en(start, end)]
    q_ru, q_en = _ru_join(ru_parts), _en_join(en_parts)
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "field_of_work": field["en"],
        "field_of_work_qid": field["qid"],
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_occ_specialized_field_country",
        template_family="occupation_specialized_field_country", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"field_policy": "only non-redundant occupation specializations are allowed"},
    )





def _tpl_specialized_field_birthrange(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L2 non-award direct criteria: occupation + meaningful subfield + birth range.

    This is intentionally not a bridge template, so it stays in L2.  The field
    must be a curated specialization of the occupation, e.g. physicist + quantum
    mechanics or mathematician + number theory.  It never uses root duplicates
    such as philosopher + philosophy.
    """
    if complexity != "L2":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, list(FIELD_SPECIALIZATIONS_BY_OCCUPATION.keys()))
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    field_key = _p_choice(rng, FIELD_SPECIALIZATIONS_BY_OCCUPATION[occ_key])
    occ, field = OCCUPATIONS[occ_key], FIELDS[field_key]

    # Recent ranges keep gold lists compact and avoid extremely sparse historical
    # field-of-work annotations. No citizenship is used here, so this is still a
    # direct L2 multi-criteria template rather than a country/citizenship prompt.
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P101 wd:{field['qid']} .",
        *_p_birth_range_lines(start, end),
    ]

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), _field_phrase_ru(field_key), _birth_phrase_ru(start, end)]
    en_parts = [en_head, _occ_phrase_en(occ_key), _field_phrase_en(field_key), _birth_phrase_en(start, end)]
    q_ru, q_en = _ru_join(ru_parts), _en_join(en_parts)
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "field_of_work": field["en"],
        "field_of_work_qid": field["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_occ_specialized_field_birthrange",
        template_family="occupation_specialized_field_birth_range", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"field_policy": "only non-redundant occupation specializations are allowed"},
    )


def _tpl_language_occ_birthrange(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    # Disabled by policy: language knowledge (P1412) is too sparse/noisy in Wikidata
    # and should never be used in generated people-domain prompts.
    return None


### 6c. L3/L4 birthplace and education country bridges

In [ ]:
def _tpl_born_in_country(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L4-only birthplace bridge.

    Birth-country by itself is too easy for L3.  In L4 it is accepted only with
    additional strong direct criteria: citizenship + modern birth range, and
    sometimes gender.  The hidden bridge remains P19 -> P17, but the prompt can
    naturally say "родившихся в стране".
    """
    if complexity != "L4":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, list(OCCUPATIONS.keys()))
    if not occ_candidates:
        return None

    occ_key = _p_choice(rng, occ_candidates)
    birth_country_key = _p_choice(rng, list(COUNTRIES.keys()))
    # Make the extra criterion real: citizenship is mandatory for L4 here.
    citizenship_candidates = [k for k in COUNTRIES.keys() if k != birth_country_key] or list(COUNTRIES.keys())
    citizenship_key = _p_choice(rng, citizenship_candidates)
    sex_key = _p_choice(rng, list(SEXES.keys())) if rng.random() < 0.35 else None
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    birth_country = COUNTRIES[birth_country_key]
    citizenship_country = COUNTRIES[citizenship_key]

    where = [
        _p_occupation_line(occ["qid"]),
        "?person wdt:P19 ?birthPlace .",
        f"?birthPlace wdt:P17 wd:{birth_country['qid']} .",
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [
        ru_head,
        _occ_phrase_ru(occ_key),
        f"родившихся в стране: {birth_country['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ]
    en_parts = [
        en_head,
        _occ_phrase_en(occ_key),
        f"born in {birth_country['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))
    q_ru, q_en = _ru_join(ru_parts), _en_join(en_parts)

    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "birth_country": birth_country["en"],
        "birth_country_qid": birth_country["qid"],
        "birth_country_path": "P19/P17",
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_born_in_country",
        template_family="birthplace_country_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> place of birth -> country"},
    )


def _tpl_educated_in_country(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, SCIENTIST_OCCS + CREATIVE_OCCS + POLITICAL_OCCS)
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    edu_country_key = _p_choice(rng, list(COUNTRIES.keys()))
    # v28: L3 must be visibly harder than L2: a bridge plus citizenship and
    # modern birth range, not just occupation + bridge + years.  L4/L5 add
    # still more constraints below, so the level ladder remains strict.
    citizenship_key = _p_choice(rng, list(COUNTRIES.keys())) if complexity in {"L3", "L4", "L5"} else None
    occ, edu_country = OCCUPATIONS[occ_key], COUNTRIES[edu_country_key]
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES) if citizenship_key else (None, None)

    where = [
        _p_occupation_line(occ["qid"]),
        "?person wdt:P69 ?educatedAt .",
        f"?educatedAt wdt:P17 wd:{edu_country['qid']} .",
    ]
    if citizenship_key:
        where.append(f"?person wdt:P27 wd:{COUNTRIES[citizenship_key]['qid']} .")
    if start is not None:
        where.extend(_p_birth_range_lines(start, end))
    # L4+ must remain harder than L3.  For education bridge, L4 gets
    # mandatory gender; L5 may also get it but has its own related/two-bridge
    # templates elsewhere.
    if complexity == "L4" or (complexity == "L5" and rng.random() < 0.55):
        sex_key = _p_choice(rng, list(SEXES.keys()))
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")
    else:
        sex_key = None

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), f"получивших образование в стране: {edu_country['ru']}"]
    en_parts = [en_head, _occ_phrase_en(occ_key), f"educated in {edu_country['en']}"]
    if citizenship_key:
        ru_parts.append(_country_phrase_ru(citizenship_key))
        en_parts.append(_country_phrase_en(citizenship_key))
    if start is not None:
        ru_parts.append(_birth_phrase_ru(start, end))
        en_parts.append(_birth_phrase_en(start, end))
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))
    q_ru, q_en = _ru_join(ru_parts), _en_join(en_parts)
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "education_country": edu_country["en"],
        "education_country_qid": edu_country["qid"],
        "education_country_path": "P69/P17",
        "citizenship_country": COUNTRIES[citizenship_key]["en"] if citizenship_key else None,
        "citizenship_country_qid": COUNTRIES[citizenship_key]["qid"] if citizenship_key else None,
        "birth_year_from": start,
        "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_educated_in_country",
        template_family="education_country_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        # Do not SELECT bridge variables: they create duplicate rows and false LIMIT hits.
        extra_select_vars=["?birthDate"] if start is not None else None,
        bridge_meta={"bridge": "person -> educated at -> institution country"},
    )


### 6d. L4/L5 two-bridge, sport, and person-relation templates

In [ ]:
def _tpl_born_and_educated_countries(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """Two-bridge template.

    L4: two bridges: birth place -> country and educated at -> country.
    L5: the same two bridges plus citizenship + modern birth range, so it is
    not just a duplicate of L4.
    """
    if complexity not in {"L4", "L5"}:
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, SCIENTIST_OCCS + CREATIVE_OCCS + POLITICAL_OCCS)
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    birth_country_key = _p_choice(rng, list(COUNTRIES.keys()))
    edu_country_key = _p_choice(rng, list(COUNTRIES.keys()))
    if birth_country_key == edu_country_key and rng.random() < 0.75:
        return None

    citizenship_key = None
    start = end = None
    if complexity == "L5":
        citizenship_candidates = [k for k in COUNTRIES.keys() if k not in {birth_country_key, edu_country_key}] or list(COUNTRIES.keys())
        citizenship_key = _p_choice(rng, citizenship_candidates)
        start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    sex_key = _p_choice(rng, list(SEXES.keys())) if complexity == "L5" and rng.random() < 0.35 else None

    occ = OCCUPATIONS[occ_key]
    birth_country = COUNTRIES[birth_country_key]
    edu_country = COUNTRIES[edu_country_key]

    where = [
        _p_occupation_line(occ["qid"]),
        "?person wdt:P19 ?birthPlace .",
        f"?birthPlace wdt:P17 wd:{birth_country['qid']} .",
        "?person wdt:P69 ?educatedAt .",
        f"?educatedAt wdt:P17 wd:{edu_country['qid']} .",
    ]
    if citizenship_key:
        where.append(f"?person wdt:P27 wd:{COUNTRIES[citizenship_key]['qid']} .")
    if start is not None:
        where.extend(_p_birth_range_lines(start, end))
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [
        ru_head,
        _occ_phrase_ru(occ_key),
        f"родившихся в стране: {birth_country['ru']}",
        f"получивших образование в стране: {edu_country['ru']}",
    ]
    en_parts = [
        en_head,
        _occ_phrase_en(occ_key),
        f"born in {birth_country['en']}",
        f"educated in {edu_country['en']}",
    ]
    if citizenship_key:
        ru_parts.append(_country_phrase_ru(citizenship_key))
        en_parts.append(_country_phrase_en(citizenship_key))
    if start is not None:
        ru_parts.append(_birth_phrase_ru(start, end))
        en_parts.append(_birth_phrase_en(start, end))
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))
    q_ru, q_en = _ru_join(ru_parts), _en_join(en_parts)

    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "birth_country": birth_country["en"],
        "birth_country_qid": birth_country["qid"],
        "birth_country_path": "P19/P17",
        "education_country": edu_country["en"],
        "education_country_qid": edu_country["qid"],
        "education_country_path": "P69/P17",
        "citizenship_country": COUNTRIES[citizenship_key]["en"] if citizenship_key else None,
        "citizenship_country_qid": COUNTRIES[citizenship_key]["qid"] if citizenship_key else None,
        "birth_year_from": start,
        "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_born_and_educated_countries",
        template_family="birthplace_and_education_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"] if start is not None else None,
        bridge_meta={"bridges": ["person -> birth place -> country", "person -> educated at -> institution country"]},
    )


def _tpl_award_country_birthrange(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    # L2 direct multi-criteria: occupation + concrete award + citizenship + birth range.
    # This is intentionally not used for L3+ because it has no bridge.
    if complexity != "L2":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, list(AWARDS_BY_OCCUPATION.keys()))
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    award_key = _p_choice(rng, AWARDS_BY_OCCUPATION[occ_key])
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    occ, award, country = OCCUPATIONS[occ_key], AWARDS[award_key], COUNTRIES[country_key]

    where = [
        _p_occupation_line(occ["qid"]),
        *_p_award_where_lines(award_key),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), _award_phrase_ru(award_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end)])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), _award_phrase_en(award_key), _country_phrase_en(country_key), _birth_phrase_en(start, end)])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        **_p_award_constraint_entries(award_key),
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_award_country_birthrange",
        template_family="award_country_birth_range", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
    )


def _tpl_language_award_occ(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    # Disabled by policy: language knowledge (P1412) is too sparse/noisy in Wikidata
    # and should never be used in generated people-domain prompts.
    return None


def _tpl_football_position_birthrange(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L2 non-award direct criteria: footballer + playing position + citizenship + birth range."""
    if complexity != "L2":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "association football player"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    position_key = _p_choice(rng, list(FOOTBALL_POSITIONS.keys()))
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    occ, pos, country = OCCUPATIONS[occ_key], FOOTBALL_POSITIONS[position_key], COUNTRIES[country_key]

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P413 wd:{pos['qid']} .",
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), f"с игровой позицией: {pos['ru']}", _country_phrase_ru(country_key), _birth_phrase_ru(start, end)])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), f"whose playing position is {pos['en']}", _country_phrase_en(country_key), _birth_phrase_en(start, end)])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "sports_position": pos["en"],
        "sports_position_qid": pos["qid"],
        "sports_position_path": "P413",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_football_position_birthrange",
        template_family="football_position_birth_range", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
    )



def _tpl_football_position_team_country(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """Sports-team bridge.

    L3: team country + playing position + birth range.
    L4: the same bridge plus citizenship + birth range, with optional gender.
    This template is not used for L5; L5 is reserved for related-human/work
    cross-entity constraints.
    """
    if complexity not in {"L3", "L4"}:
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    country_key = _p_choice(rng, ["United States", "United Kingdom", "France", "Germany", "Italy", "Spain", "Brazil", "Argentina", "Netherlands"])
    pos_key = _p_choice(rng, list(FOOTBALL_POSITIONS.keys()))
    country, pos = COUNTRIES[country_key], FOOTBALL_POSITIONS[pos_key]

    if complexity == "L4":
        citizenship_key = _p_choice(rng, list(COUNTRIES.keys()))
        sex_key = _p_choice(rng, list(SEXES.keys())) if rng.random() < 0.35 else None
    else:
        citizenship_key = None
        sex_key = None
    start, end = rng.choice(RECENT_BIRTH_RANGES)

    where = [
        _p_occupation_line(OCCUPATIONS["association football player"]["qid"]),
        "?person wdt:P54 ?team .",
        f"?team wdt:P17 wd:{country['qid']} .",
        f"?person wdt:P413 wd:{pos['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if citizenship_key:
        where.append(f"?person wdt:P27 wd:{COUNTRIES[citizenship_key]['qid']} .")
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [
        ru_head,
        "являющихся футболистами",
        f"игравших за клуб, расположенный в стране: {country['ru']}",
        f"позиция на поле: {pos['ru']}",
        _birth_phrase_ru(start, end),
    ]
    en_parts = [
        en_head,
        "who are association football players",
        f"who played for a team located in {country['en']}",
        f"playing position: {pos['en']}",
        _birth_phrase_en(start, end),
    ]
    if citizenship_key:
        ru_parts.append(_country_phrase_ru(citizenship_key))
        en_parts.append(_country_phrase_en(citizenship_key))
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))
    q_ru, q_en = _ru_join(ru_parts), _en_join(en_parts)

    constraints = {
        "kind": "human",
        "occupation": "association football player",
        "occupation_qid": OCCUPATIONS["association football player"]["qid"],
        "occupation_match": "P106",
        "member_of_sports_team_in_country": country["en"],
        "member_of_sports_team_in_country_qid": country["qid"],
        "team_country_path": "P54/P17",
        "playing_position": pos["en"],
        "playing_position_qid": pos["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "citizenship_country": COUNTRIES[citizenship_key]["en"] if citizenship_key else None,
        "citizenship_country_qid": COUNTRIES[citizenship_key]["qid"] if citizenship_key else None,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_football_position_team_country",
        template_family="sports_team_country_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> sports team -> country"},
    )


### 6d-ii. L4/L5 person-relation bridge templates (spouse, parent, work genre, advisor)

In [ ]:
def _tpl_spouse_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = _p_choice(rng, CREATIVE_OCCS + SCIENTIST_OCCS + POLITICAL_OCCS)
    spouse_occ_key = _p_choice(rng, CREATIVE_OCCS + SCIENTIST_OCCS + POLITICAL_OCCS)
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    occ, spouse_occ, country = OCCUPATIONS[occ_key], OCCUPATIONS[spouse_occ_key], COUNTRIES[country_key]
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
        "?person wdt:P26 ?spouse .",
        "?spouse wdt:P31 wd:Q5 .",
        f"?spouse wdt:P106 wd:{spouse_occ['qid']} .",
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(country_key),
        _birth_phrase_ru(start, end),
        f"у которых супруг или супруга имеет профессию «{spouse_occ['ru']}»",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(country_key),
        _birth_phrase_en(start, end),
        f"whose spouse's occupation is {spouse_occ['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "spouse_occupation": spouse_occ["en"],
        "spouse_occupation_qid": spouse_occ["qid"],
        "spouse_occupation_match": "P26/P106",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_spouse_bridge",
        template_family="spouse_occupation_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?spouse", "?birthDate"],
        bridge_meta={"bridge": "person -> spouse -> spouse occupation"},
    )


def _tpl_parent_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = _p_choice(rng, CREATIVE_OCCS + SCIENTIST_OCCS + POLITICAL_OCCS)
    parent_occ_key = _p_choice(rng, CREATIVE_OCCS + SCIENTIST_OCCS + POLITICAL_OCCS)
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    occ, parent_occ, country = OCCUPATIONS[occ_key], OCCUPATIONS[parent_occ_key], COUNTRIES[country_key]
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
        "{ ?person wdt:P22 ?parent . } UNION { ?person wdt:P25 ?parent . }",
        "?parent wdt:P31 wd:Q5 .",
        f"?parent wdt:P106 wd:{parent_occ['qid']} .",
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(country_key),
        _birth_phrase_ru(start, end),
        f"у которых отец или мать имеет профессию «{parent_occ['ru']}»",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(country_key),
        _birth_phrase_en(start, end),
        f"whose father or mother has occupation {parent_occ['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "parent_occupation": parent_occ["en"],
        "parent_occupation_qid": parent_occ["qid"],
        "parent_occupation_match": "P22_or_P25/P106",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_parent_bridge",
        template_family="parent_occupation_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?parent", "?birthDate"],
        bridge_meta={"bridge": "person -> father/mother -> parent occupation"},
    )



def _tpl_work_genre_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """Bridge through a notable work and its genre.

    L3: work genre bridge + citizenship + birth range.
    L5: work genre bridge + citizenship + birth range + gender, so it is
    visibly harder than the L3 version and not just the same template relabeled.
    """
    if complexity not in {"L3", "L5"}:
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_pool = ["writer", "poet", "film director", "composer", "singer", "actor"]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, occ_pool)
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    genre_key = _p_choice(rng, list(WORK_GENRES.keys()))
    occ, genre = OCCUPATIONS[occ_key], WORK_GENRES[genre_key]
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    # v28: citizenship is mandatory already at L3; without it these prompts
    # were too close to L2 direct multi-criteria tasks.
    country_key = _p_choice(rng, list(COUNTRIES.keys())) if complexity in {"L3", "L5"} else None
    sex_key = _p_choice(rng, list(SEXES.keys())) if complexity == "L5" else None

    where = [
        _p_occupation_line(occ["qid"]),
        "?person wdt:P800 ?notableWork .",
        f"?notableWork wdt:P136 wd:{genre['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if country_key:
        where.append(f"?person wdt:P27 wd:{COUNTRIES[country_key]['qid']} .")
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), f"у которых есть известная работа жанра «{genre['ru']}»"]
    en_parts = [en_head, _occ_phrase_en(occ_key), f"who have a notable work in the genre {genre['en']}"]
    if country_key:
        ru_parts.append(_country_phrase_ru(country_key))
        en_parts.append(_country_phrase_en(country_key))
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))
    ru_parts.append(_birth_phrase_ru(start, end))
    en_parts.append(_birth_phrase_en(start, end))
    q_ru, q_en = _ru_join(ru_parts), _en_join(en_parts)

    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "notable_work_genre": genre["en"],
        "notable_work_genre_qid": genre["qid"],
        "notable_work_genre_path": "P800/P136",
        "citizenship_country": COUNTRIES[country_key]["en"] if country_key else None,
        "citizenship_country_qid": COUNTRIES[country_key]["qid"] if country_key else None,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_work_genre_bridge",
        template_family="notable_work_genre_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> notable work -> genre"},
    )


def _tpl_academic_advisor_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = _p_choice(rng, SCIENTIST_OCCS + ["philosopher", "historian", "economist"])
    advisor_country_key = _p_choice(rng, list(COUNTRIES.keys()))
    field_key = _p_choice(rng, [f for f in FIELD_BY_OCCUPATION.get(occ_key, list(FIELDS.keys())) if f in FIELDS])
    occ, advisor_country, field = OCCUPATIONS[occ_key], COUNTRIES[advisor_country_key], FIELDS[field_key]
    advisor_birth_start, advisor_birth_end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P101 wd:{field['qid']} .",
        "?person wdt:P184 ?advisor .",
        "?advisor wdt:P31 wd:Q5 .",
        f"?advisor wdt:P27 wd:{advisor_country['qid']} .",
        *_p_birth_range_lines(advisor_birth_start, advisor_birth_end, var="advisor"),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _field_phrase_ru(field_key),
        f"у которых научный руководитель имеет гражданство: {advisor_country['ru']}",
        _advisor_birth_phrase_ru(advisor_birth_start, advisor_birth_end),
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _field_phrase_en(field_key),
        f"whose doctoral advisor has citizenship: {advisor_country['en']}",
        _advisor_birth_phrase_en(advisor_birth_start, advisor_birth_end),
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "field_of_work": field["en"],
        "field_of_work_qid": field["qid"],
        "doctoral_advisor_citizenship_country": advisor_country["en"],
        "doctoral_advisor_citizenship_country_qid": advisor_country["qid"],
        "doctoral_advisor_citizenship_path": "P184/P27",
        "doctoral_advisor_birth_year_from": advisor_birth_start,
        "doctoral_advisor_birth_year_to": advisor_birth_end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_academic_advisor_bridge",
        template_family="advisor_country_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?advisor", "?birthDate"],
        bridge_meta={"bridge": "person -> doctoral advisor -> advisor citizenship"},
    )


# ============================================================


### 6e. v30 hard L3/L5 templates (employer, political party, authored work, film direction)

In [ ]:
def _tpl_employer_country_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: person -> employer -> country, plus citizenship + modern birth range."""
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_pool = SCIENTIST_OCCS + POLITICAL_OCCS + ["writer", "architect", "painter"]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, occ_pool)
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    employer_country_key = _p_choice(rng, MAJOR_EMPLOYER_COUNTRIES)
    citizenship_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    sex_key = _p_choice(rng, list(SEXES.keys())) if rng.random() < 0.25 else None

    occ = OCCUPATIONS[occ_key]
    employer_country = COUNTRIES[employer_country_key]
    citizenship_country = COUNTRIES[citizenship_key]

    where = [
        _p_occupation_line(occ["qid"]),
        "?person wdt:P108 ?employer .",
        f"?employer wdt:P17 wd:{employer_country['qid']} .",
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [
        ru_head,
        _occ_phrase_ru(occ_key),
        f"работавших в организации из страны: {employer_country['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ]
    en_parts = [
        en_head,
        _occ_phrase_en(occ_key),
        f"who worked for an organization from {employer_country['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))

    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "employer_country": employer_country["en"],
        "employer_country_qid": employer_country["qid"],
        "employer_country_path": "P108/P17",
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_employer_country_bridge",
        template_family="employer_country_bridge", q_ru=_ru_join(ru_parts), q_en=_en_join(en_parts),
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> employer -> employer country"},
    )


def _tpl_political_party_country_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: politician/lawyer/journalist -> political party -> country."""
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_pool = ["politician", "lawyer", "journalist"]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, occ_pool)
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    party_country_key = _p_choice(rng, MAJOR_PARTY_COUNTRIES)
    citizenship_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    sex_key = _p_choice(rng, list(SEXES.keys())) if rng.random() < 0.35 else None

    occ = OCCUPATIONS[occ_key]
    party_country = COUNTRIES[party_country_key]
    citizenship_country = COUNTRIES[citizenship_key]
    where = [
        _p_occupation_line(occ["qid"]),
        "?person wdt:P102 ?politicalParty .",
        f"?politicalParty wdt:P17 wd:{party_country['qid']} .",
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [
        ru_head,
        _occ_phrase_ru(occ_key),
        f"связанных с политической партией из страны: {party_country['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ]
    en_parts = [
        en_head,
        _occ_phrase_en(occ_key),
        f"associated with a political party from {party_country['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))

    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "political_party_country": party_country["en"],
        "political_party_country_qid": party_country["qid"],
        "political_party_country_path": "P102/P17",
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_party_country_bridge",
        template_family="political_party_country_bridge", q_ru=_ru_join(ru_parts), q_en=_en_join(en_parts),
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> political party -> party country"},
    )


def _tpl_author_work_lang_genre(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: writer -> authored work -> language + genre, plus citizenship/birth."""
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "writer"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    genre_key = _p_choice(rng, list(LITERARY_WORK_GENRES.keys()))
    lang_key = _p_choice(rng, ["English", "French", "Spanish", "Russian", "German", "Italian", "Japanese", "Portuguese", "Polish"])
    citizenship_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    genre = LITERARY_WORK_GENRES[genre_key]
    lang = LANGUAGES[lang_key]
    citizenship_country = COUNTRIES[citizenship_key]

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
        "?work wdt:P50 ?person .",
        "?work wdt:P31/wdt:P279* wd:Q7725634 .",
        f"?work wdt:P136 wd:{genre['qid']} .",
        f"?work wdt:P407 wd:{lang['qid']} .",
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        f"у которых есть авторское произведение жанра «{genre['ru']}» на языке: {lang['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        f"who authored a work in the genre {genre['en']} and in {lang['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "authored_work_genre": genre["en"],
        "authored_work_genre_qid": genre["qid"],
        "authored_work_language": lang["en"],
        "authored_work_language_qid": lang["qid"],
        "authored_work_match_path": "inverse P50 + P136/P407",
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_author_work_lang_genre",
        template_family="authored_work_language_genre_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person <- authored work -> work genre and language"},
    )


def _tpl_directed_film_country_genre(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: film director -> film -> country of origin + genre."""
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "film director"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    genre_key = _p_choice(rng, list(FILM_GENRES.keys()))
    film_country_key = _p_choice(rng, MAJOR_FILM_COUNTRIES)
    citizenship_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    genre = FILM_GENRES[genre_key]
    film_country = COUNTRIES[film_country_key]
    citizenship_country = COUNTRIES[citizenship_key]

    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
        "?film wdt:P57 ?person .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        f"?film wdt:P495 wd:{film_country['qid']} .",
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        f"снявших фильм жанра «{genre['ru']}» со страной происхождения: {film_country['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        f"who directed a {genre['en']} film whose country of origin is {film_country['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "directed_film_genre": genre["en"],
        "directed_film_genre_qid": genre["qid"],
        "directed_film_country_of_origin": film_country["en"],
        "directed_film_country_of_origin_qid": film_country["qid"],
        "directed_film_match_path": "inverse P57 + P136/P495",
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_directed_film_country_genre",
        template_family="directed_film_country_genre_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person <- directed film -> film genre and country of origin"},
    )


### 6e-ii. v30 person-relation award bridges and multi-hop film templates

In [ ]:
def _tpl_spouse_award_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: person -> spouse -> spouse occupation + spouse award."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    spouse_occ_key = _p_choice(rng, list(AWARDS_BY_OCCUPATION.keys()))
    award_key = _p_choice(rng, AWARDS_BY_OCCUPATION[spouse_occ_key])
    occ_pool = CREATIVE_OCCS + SCIENTIST_OCCS + POLITICAL_OCCS
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, occ_pool)
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ, spouse_occ, country = OCCUPATIONS[occ_key], OCCUPATIONS[spouse_occ_key], COUNTRIES[country_key]
    award = AWARDS[award_key]
    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
        "?person wdt:P26 ?spouse .",
        "?spouse wdt:P31 wd:Q5 .",
        f"?spouse wdt:P106 wd:{spouse_occ['qid']} .",
        *_p_award_where_lines_for_var(award_key, var="spouse", value_var="?spouseAwardReceived"),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(country_key),
        _birth_phrase_ru(start, end),
        f"у которых супруг или супруга имеет профессию «{spouse_occ['ru']}» и получил(а) награду: {award['ru']}",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(country_key),
        _birth_phrase_en(start, end),
        f"whose spouse's occupation is {spouse_occ['en']} and whose spouse received the {award['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "spouse_occupation": spouse_occ["en"],
        "spouse_occupation_qid": spouse_occ["qid"],
        "spouse_award_received": award["en"],
        "spouse_award_received_qid": award["qid"],
        "spouse_bridge_path": "P26/P106/P166",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_spouse_award_bridge",
        template_family="spouse_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> spouse -> spouse occupation and award"},
    )


def _tpl_parent_award_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: person -> parent -> parent occupation + parent award."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    parent_occ_key = _p_choice(rng, list(AWARDS_BY_OCCUPATION.keys()))
    award_key = _p_choice(rng, AWARDS_BY_OCCUPATION[parent_occ_key])
    occ_pool = CREATIVE_OCCS + SCIENTIST_OCCS + POLITICAL_OCCS
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, occ_pool)
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    country_key = _p_choice(rng, list(COUNTRIES.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ, parent_occ, country = OCCUPATIONS[occ_key], OCCUPATIONS[parent_occ_key], COUNTRIES[country_key]
    award = AWARDS[award_key]
    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
        "{ ?person wdt:P22 ?parent . } UNION { ?person wdt:P25 ?parent . }",
        "?parent wdt:P31 wd:Q5 .",
        f"?parent wdt:P106 wd:{parent_occ['qid']} .",
        *_p_award_where_lines_for_var(award_key, var="parent", value_var="?parentAwardReceived"),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(country_key),
        _birth_phrase_ru(start, end),
        f"у которых отец или мать имеет профессию «{parent_occ['ru']}» и получил(а) награду: {award['ru']}",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(country_key),
        _birth_phrase_en(start, end),
        f"whose father or mother has occupation {parent_occ['en']} and received the {award['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "parent_occupation": parent_occ["en"],
        "parent_occupation_qid": parent_occ["qid"],
        "parent_award_received": award["en"],
        "parent_award_received_qid": award["qid"],
        "parent_bridge_path": "P22_or_P25/P106/P166",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_parent_award_bridge",
        template_family="parent_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> father/mother -> parent occupation and award"},
    )


def _tpl_academic_advisor_award_bridge(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: scholar -> doctoral advisor -> advisor award, with specialized field."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, ["mathematician", "physicist", "chemist", "biologist", "economist"])
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    field_key = _p_choice(rng, FIELD_SPECIALIZATIONS_BY_OCCUPATION[occ_key])
    award_options = {
        "mathematician": ["Fields Medal", "Wolf Prize"],
        "physicist": ["Nobel Prize", "Wolf Prize"],
        "chemist": ["Nobel Prize", "Wolf Prize"],
        "biologist": ["Nobel Prize", "Wolf Prize"],
        "economist": ["Nobel Prize", "Wolf Prize"],
    }[occ_key]
    award_key = _p_choice(rng, award_options)
    advisor_start, advisor_end = rng.choice(RELATED_PERSON_BIRTH_RANGES)

    occ, field, award = OCCUPATIONS[occ_key], FIELDS[field_key], AWARDS[award_key]
    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P101 wd:{field['qid']} .",
        "?person wdt:P184 ?advisor .",
        "?advisor wdt:P31 wd:Q5 .",
        *_p_birth_range_lines_scoped(advisor_start, advisor_end, var="advisor", scope="advisor"),
        *_p_award_where_lines_for_var(award_key, var="advisor", value_var="?advisorAwardReceived"),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _field_phrase_ru(field_key),
        f"у которых научный руководитель получил награду: {award['ru']}",
        _advisor_birth_phrase_ru(advisor_start, advisor_end),
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _field_phrase_en(field_key),
        f"whose doctoral advisor received the {award['en']}",
        _advisor_birth_phrase_en(advisor_start, advisor_end),
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "field_of_work": field["en"],
        "field_of_work_qid": field["qid"],
        "doctoral_advisor_award_received": award["en"],
        "doctoral_advisor_award_received_qid": award["qid"],
        "doctoral_advisor_award_path": "P184/P166",
        "doctoral_advisor_birth_year_from": advisor_start,
        "doctoral_advisor_birth_year_to": advisor_end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_advisor_award_bridge",
        template_family="advisor_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta={"bridge": "person -> doctoral advisor -> advisor award"},
    )


def _tpl_actor_film_director_country_genre(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: actor -> film -> director -> director citizenship/birth, plus film genre."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "actor"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    actor_country_key = _p_choice(rng, list(COUNTRIES.keys()))
    director_country_key = _p_choice(rng, MAJOR_FILM_COUNTRIES)
    genre_key = _p_choice(rng, list(FILM_GENRES.keys()))
    actor_start, actor_end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    director_start, director_end = rng.choice(RELATED_PERSON_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    actor_country = COUNTRIES[actor_country_key]
    director_country = COUNTRIES[director_country_key]
    genre = FILM_GENRES[genre_key]
    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{actor_country['qid']} .",
        *_p_birth_range_lines_scoped(actor_start, actor_end, var="person", scope="person"),
        "?film wdt:P161 ?person .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        "?film wdt:P57 ?director .",
        "?director wdt:P31 wd:Q5 .",
        f"?director wdt:P27 wd:{director_country['qid']} .",
        *_p_birth_range_lines_scoped(director_start, director_end, var="director", scope="director"),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(actor_country_key),
        _birth_phrase_ru(actor_start, actor_end),
        f"снимавшихся в фильме жанра «{genre['ru']}», режиссёр которого имеет гражданство: {director_country['ru']}",
        f"и режиссёр родился с {director_start} по {director_end} год",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(actor_country_key),
        _birth_phrase_en(actor_start, actor_end),
        f"who acted in a {genre['en']} film whose director has citizenship: {director_country['en']}",
        f"and whose director was born from {director_start} to {director_end}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": actor_country["en"],
        "citizenship_country_qid": actor_country["qid"],
        "birth_year_from": actor_start,
        "birth_year_to": actor_end,
        "acted_in_film_genre": genre["en"],
        "acted_in_film_genre_qid": genre["qid"],
        "film_director_citizenship_country": director_country["en"],
        "film_director_citizenship_country_qid": director_country["qid"],
        "film_director_birth_year_from": director_start,
        "film_director_birth_year_to": director_end,
        "film_director_bridge_path": "inverse P161/P57/P27",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_actor_film_director_country_genre",
        template_family="actor_film_director_country_genre_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta={"bridge": "actor <- film -> director -> director citizenship/birth; film -> genre"},
    )


def _tpl_director_film_cast_award(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: director -> film -> cast member -> cast award, plus film genre."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "film director"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    director_country_key = _p_choice(rng, list(COUNTRIES.keys()))
    genre_key = _p_choice(rng, list(FILM_GENRES.keys()))
    cast_award_key = _p_choice(rng, ["Academy Award", "BAFTA Award", "César Award"])
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    director_country = COUNTRIES[director_country_key]
    genre = FILM_GENRES[genre_key]
    cast_award = AWARDS[cast_award_key]
    where = [
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{director_country['qid']} .",
        *_p_birth_range_lines(start, end),
        "?film wdt:P57 ?person .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        "?film wdt:P161 ?castMember .",
        "?castMember wdt:P31 wd:Q5 .",
        *_p_award_where_lines_for_var(cast_award_key, var="castMember", value_var="?castAwardReceived"),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(director_country_key),
        _birth_phrase_ru(start, end),
        f"снявших фильм жанра «{genre['ru']}», в актёрском составе которого есть человек, получивший награду: {cast_award['ru']}",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(director_country_key),
        _birth_phrase_en(start, end),
        f"who directed a {genre['en']} film whose cast includes a person who received the {cast_award['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": director_country["en"],
        "citizenship_country_qid": director_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "directed_film_genre": genre["en"],
        "directed_film_genre_qid": genre["qid"],
        "cast_member_award_received": cast_award["en"],
        "cast_member_award_received_qid": cast_award["qid"],
        "cast_member_award_bridge_path": "inverse P57/P161/P166",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_director_film_cast_award",
        template_family="director_film_cast_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "director <- film -> cast member -> cast member award; film -> genre"},
    )

print("✅ Template builders loaded")

## 7. Template registry

In [7]:
PEOPLE_TEMPLATES_BY_LEVEL = {
    "L1": [
        (_tpl_occ_citizenship, 0.65),
        (_tpl_occ_sex_citizenship, 0.35),
    ],
    "L2": [
        (_tpl_specialized_field_birthrange, 0.32),
        (_tpl_field_country, 0.18),
        (_tpl_football_position_birthrange, 0.18),
        (_tpl_occ_award, 0.16),
        (_tpl_award_country_birthrange, 0.16),
    ],
    # v30 L3: one real bridge plus visible direct filters. The registry avoids
    # relying on the old two-template pattern (education + football only).
    "L3": [
        (_tpl_educated_in_country, 0.18),
        (_tpl_employer_country_bridge, 0.18),
        (_tpl_political_party_country_bridge, 0.12),
        (_tpl_author_work_lang_genre, 0.18),
        (_tpl_directed_film_country_genre, 0.16),
        (_tpl_football_position_team_country, 0.18),
    ],
    "L4": [
        (_tpl_born_in_country, 0.15),
        (_tpl_educated_in_country, 0.35),
        (_tpl_born_and_educated_countries, 0.30),
        (_tpl_football_position_team_country, 0.20),
    ],
    # v30 L5: hard cross-entity patterns. Simple spouse/parent-occupation
    # templates remain as fallback only and cannot dominate the dataset.
    "L5": [
        (_tpl_actor_film_director_country_genre, 0.17),
        (_tpl_director_film_cast_award, 0.15),
        (_tpl_spouse_award_bridge, 0.14),
        (_tpl_parent_award_bridge, 0.12),
        (_tpl_academic_advisor_award_bridge, 0.17),
        (_tpl_academic_advisor_bridge, 0.10),
        (_tpl_born_and_educated_countries, 0.08),
        (_tpl_spouse_bridge, 0.04),
        (_tpl_parent_bridge, 0.03),
    ],
}

PEOPLE_TEMPLATE_FN_TO_FAMILY = {
    "_tpl_educated_in_country": "education_country_bridge",
    "_tpl_football_position_team_country": "sports_team_country_bridge",
    "_tpl_employer_country_bridge": "employer_country_bridge",
    "_tpl_political_party_country_bridge": "political_party_country_bridge",
    "_tpl_author_work_lang_genre": "authored_work_language_genre_bridge",
    "_tpl_directed_film_country_genre": "directed_film_country_genre_bridge",
    "_tpl_actor_film_director_country_genre": "actor_film_director_country_genre_bridge",
    "_tpl_director_film_cast_award": "director_film_cast_award_bridge",
    "_tpl_spouse_award_bridge": "spouse_award_bridge",
    "_tpl_parent_award_bridge": "parent_award_bridge",
    "_tpl_academic_advisor_award_bridge": "advisor_award_bridge",
    "_tpl_academic_advisor_bridge": "advisor_country_bridge",
    "_tpl_born_and_educated_countries": "birthplace_and_education_bridge",
    "_tpl_spouse_bridge": "spouse_occupation_bridge",
    "_tpl_parent_bridge": "parent_occupation_bridge",
}

# Family caps are soft generation controls: if a family reaches the cap it is
# removed from random selection, but if every family is capped the generator falls
# back rather than crashing. This prevents L5 from degenerating into 50 spouse /
# parent records again.
PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL = {
    "L3": {
        "education_country_bridge": 0.30,
        "sports_team_country_bridge": 0.28,
        "employer_country_bridge": 0.26,
        "political_party_country_bridge": 0.20,
        "authored_work_language_genre_bridge": 0.26,
        "directed_film_country_genre_bridge": 0.24,
    },
    "L5": {
        "spouse_occupation_bridge": 0.10,
        "parent_occupation_bridge": 0.10,
        "spouse_award_bridge": 0.18,
        "parent_award_bridge": 0.16,
        "advisor_award_bridge": 0.20,
        "advisor_country_bridge": 0.16,
        "actor_film_director_country_genre_bridge": 0.22,
        "director_film_cast_award_bridge": 0.20,
        "birthplace_and_education_bridge": 0.16,
    },
}
PEOPLE_L3_TEMPLATE_FAMILY_CAPS = PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL["L3"]
PEOPLE_L3_MAX_CONSECUTIVE_SAME_FAMILY = 2
PEOPLE_MAX_CONSECUTIVE_SAME_FAMILY_BY_LEVEL = {"L3": 2, "L5": 2}


def _people_level_records_ctx(complexity: str) -> List[Dict[str, Any]]:
    records_ctx = globals().get("PEOPLE_RUNTIME_RECORDS_FOR_QUOTAS") or []
    return [r for r in records_ctx if r.get("complexity") == complexity]


def _people_template_family_count(complexity: str, family: str) -> int:
    return sum(1 for r in _people_level_records_ctx(complexity) if people_record_family(r) == family)


def _people_template_family_limit(complexity: str, family: str) -> Optional[int]:
    target = int(PEOPLE_TARGET_PER_LEVEL.get(complexity, 0) or 0)
    share = PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL.get(complexity, {}).get(family)
    if not target or share is None:
        return None
    return max(1, int(math.ceil(target * float(share))))


def _people_family_allowed_for_pick(complexity: str, family: Optional[str]) -> bool:
    if not family:
        return True
    limit = _people_template_family_limit(complexity, family)
    if limit is not None and _people_template_family_count(complexity, family) >= int(limit):
        return False
    max_run = PEOPLE_MAX_CONSECUTIVE_SAME_FAMILY_BY_LEVEL.get(complexity)
    if max_run:
        recent = _people_level_records_ctx(complexity)[-int(max_run):]
        if len(recent) >= int(max_run) and all(people_record_family(r) == family for r in recent):
            return False
    return True

# Backward-compatible wrappers used by audit/debug cells.
def _people_l3_template_family_count(family: str) -> int:
    return _people_template_family_count("L3", family)


def _people_l3_template_family_limit(family: str) -> Optional[int]:
    return _people_template_family_limit("L3", family)


def _people_l3_family_allowed_for_pick(family: Optional[str]) -> bool:
    return _people_family_allowed_for_pick("L3", family)


def _p_pick_template(complexity: str, rng: random.Random):
    weighted = PEOPLE_TEMPLATES_BY_LEVEL[complexity]
    disabled = set((globals().get("PEOPLE_RUNTIME_DISABLED_TEMPLATE_NAMES") or {}).get(complexity, set()))
    filtered = [(fn, w) for fn, w in weighted if fn.__name__ not in disabled]

    if complexity in PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL:
        diverse = []
        for fn, w in filtered:
            family = PEOPLE_TEMPLATE_FN_TO_FAMILY.get(fn.__name__)
            if _people_family_allowed_for_pick(complexity, family):
                diverse.append((fn, w))
        if diverse:
            filtered = diverse

    if not filtered:
        filtered = weighted
    funcs = [x[0] for x in filtered]
    weights = [x[1] for x in filtered]
    return rng.choices(funcs, weights=weights, k=1)[0]

print("Template registry loaded")


Template registry loaded


## 8. Incremental generation and validation helpers

In [ ]:
def people_example_to_dict(ex):
    if is_dataclass(ex):
        return asdict(ex)
    if hasattr(ex, "__dict__"):
        return dict(ex.__dict__)
    return dict(ex)


def people_record_to_example(obj: Any) -> BenchmarkExample:
    if isinstance(obj, BenchmarkExample):
        return obj
    kwargs = {f.name: obj.get(f.name) for f in fields(BenchmarkExample) if f.name in obj}
    return BenchmarkExample(**kwargs)


def people_make_record_key(r):
    return (
        r.get("query_text_ru"),
        json.dumps(r.get("constraints", {}), ensure_ascii=False, sort_keys=True),
    )


def people_read_existing_jsonl(path: Path):
    records = []
    bad_lines = []
    if not path.exists():
        return records, bad_lines
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                records.append(json.loads(raw))
            except Exception as e:
                bad_lines.append({"line": line_no, "error": str(e), "preview": raw[:500]})
    return records, bad_lines


def people_append_jsonl(path: Path, record: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()


def people_read_reference_jsonl(paths: Sequence[Path]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Read optional reference JSONLs for duplicate avoidance and ID continuation.

    Reference rows are not appended to PEOPLE_OUTPUT_PATH. They only prevent the
    L3/L5 supplement from regenerating a query already present in a curated file.
    """
    rows: List[Dict[str, Any]] = []
    issues: List[Dict[str, Any]] = []
    seen_paths = set()
    for p in paths or []:
        try:
            path = Path(p)
        except Exception:
            continue
        if not path.exists() or str(path.resolve()) in seen_paths:
            continue
        seen_paths.add(str(path.resolve()))
        recs, bad = people_read_existing_jsonl(path)
        for b in bad:
            issues.append({"path": str(path), **b})
        for r in recs:
            try:
                rows.append(people_normalize_record_format(r))
            except Exception:
                rows.append(r)
    return rows, issues


def people_write_jsonl(path: Path, rows: List[Dict[str, Any]]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()


def people_write_json(path: Path, obj: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()



def people_normalize_record_format(r: Dict[str, Any]) -> Dict[str, Any]:
    """Normalize old accepted records to the current cinema-style people format.

    This keeps substantively valid records during resume while cleaning metadata
    shape differences introduced by earlier notebook versions.
    """
    out = dict(r)

    # v27: keep the education-country bridge in SPARQL/metadata, but make the
    # public prompt and public constraint name natural. Older records used
    # "educated_at_in_country" and the awkward phrase
    # (Russian: "who received education at an organization located in the country").
    if isinstance(out.get("query_text_ru"), str):
        out["query_text_ru"] = out["query_text_ru"].replace(
            "получивших образование в организации, расположенной в стране:",
            "получивших образование в стране:",
        )
    if isinstance(out.get("query_text_en"), str):
        out["query_text_en"] = out["query_text_en"].replace(
            "educated at an institution located in",
            "educated in",
        )

    def _rename_education_country_keys(d: Dict[str, Any]) -> Dict[str, Any]:
        d = dict(d or {})
        rename = {
            "educated_at_in_country": "education_country",
            "educated_at_in_country_qid": "education_country_qid",
            "educated_at_country_path": "education_country_path",
        }
        for old, new in rename.items():
            if old in d and new not in d:
                d[new] = d.pop(old)
            elif old in d:
                d.pop(old, None)
        return d

    out["constraints"] = _rename_education_country_keys(out.get("constraints") or {})

    lv = dict(out.get("local_validator") or {})
    if lv and isinstance(lv.get("filters"), dict):
        lv["filters"] = _rename_education_country_keys(lv.get("filters") or {})
    if lv:
        lv.setdefault("match_key", "wikidata_qid")
        lv_order = ["type", "source", "match_key", "applies_after", "filters", "label_matching_used", "note"]
        out["local_validator"] = {k: lv[k] for k in lv_order if k in lv}

    meta = dict(out.get("gold_collection_meta") or {})
    if meta:
        meta.setdefault("match_key", "wikidata_qid")
        if "candidates_from_wdqs" not in meta:
            meta["candidates_from_wdqs"] = int(meta.get("rows_returned_by_wdqs", 0) or 0)

        # Cinema-style dropped_reasons object.
        dropped = dict(meta.get("dropped_reasons") or {})
        dropped.setdefault("missing_qid", int(meta.get("dropped_no_qid_count", 0) or 0))
        dropped.setdefault("missing_en_label", int(meta.get("dropped_no_en_label_count", 0) or 0))
        rows_returned = int(meta.get("rows_returned_by_wdqs", meta.get("candidates_from_wdqs", 0)) or 0)
        before_limits = int(meta.get("gold_returned_before_limits", meta.get("gold_total_before_limit", 0)) or 0)
        dropped.setdefault("duplicate_qid", max(0, rows_returned - before_limits - dropped["missing_qid"] - dropped["missing_en_label"]))
        meta["dropped_reasons"] = dropped

        meta.setdefault(
            "note",
            "Gold is collected directly from Wikidata SPARQL. Labels are not used for matching; English label is required, Russian label uses English fallback when absent.",
        )

        # Move low-level QID/path metadata into bridge_meta, mirroring cinema.
        bridge = dict(meta.get("bridge_meta") or {})
        if isinstance(bridge.get("constraint_entities"), dict):
            bridge["constraint_entities"] = _rename_education_country_keys(bridge.get("constraint_entities") or {})
        if isinstance(bridge.get("constraint_property_paths"), dict):
            bridge["constraint_property_paths"] = _rename_education_country_keys(bridge.get("constraint_property_paths") or {})
        for k in ("constraint_entities", "constraint_property_paths", "nested_constraint_metadata"):
            if k in meta:
                bridge[k] = meta.pop(k)
        meta["bridge_meta"] = bridge or None

        # These duplicate top-level fields or audit-only settings; keep them out
        # of per-record gold_collection_meta for cinema-style consistency.
        for k in ("template_id", "template_family", "quality_filters"):
            meta.pop(k, None)

        out["gold_collection_meta"] = meta

    # Preserve BenchmarkExample key order exactly.
    ordered = {}
    for f in fields(BenchmarkExample):
        if f.name in out:
            ordered[f.name] = out[f.name]
    for k, v in out.items():
        if k not in ordered:
            ordered[k] = v
    return ordered


def people_gold_count(record: Dict[str, Any]) -> int:
    return len(record.get("gold_answer_qids") or [])


def people_record_family(record: Dict[str, Any]) -> str:
    return record.get("template_family") or "unknown"


def people_record_template(record: Dict[str, Any]) -> str:
    return record.get("template_id") or "unknown"


def people_record_occupation(record: Dict[str, Any]) -> Optional[str]:
    constraints = record.get("constraints") or {}
    occ = constraints.get("occupation")
    return str(occ).strip() if occ else None


def people_same_occupation_count(records: List[Dict[str, Any]], complexity: str, occupation: Optional[str]) -> int:
    if not occupation:
        return 0
    return sum(
        1 for r in records
        if r.get("complexity") == complexity and people_record_occupation(r) == occupation
    )


def _people_constraint_entities(record: Dict[str, Any]) -> Dict[str, Any]:
    """Return low-level constraint QIDs regardless of old/new metadata layout."""
    meta = record.get("gold_collection_meta") or {}
    bridge = meta.get("bridge_meta") or {}
    out: Dict[str, Any] = {}
    for source in (
        meta.get("constraint_entities"),
        bridge.get("constraint_entities"),
    ):
        if isinstance(source, dict):
            out.update(source)
    return out


def _people_entity_qid_from_record(
    record: Dict[str, Any],
    *,
    entity_key: str,
    public_key: str,
    vocab: Dict[str, Dict[str, str]],
) -> Optional[str]:
    """Prefer metadata QID; use public English label only as legacy fallback."""
    entities = _people_constraint_entities(record)
    qid = entities.get(entity_key)
    if isinstance(qid, str) and qid.startswith("Q"):
        return qid

    value = (record.get("constraints") or {}).get(public_key)
    if value:
        key = next((k for k, v in vocab.items() if str(v.get("en")).casefold() == str(value).casefold()), None)
        if key and vocab.get(key, {}).get("qid"):
            return vocab[key]["qid"]
    return None


### 8b. Quality validation and generation dispatcher

In [ ]:
def people_record_quality_problems(record: Dict[str, Any]) -> List[str]:
    problems: List[str] = []
    expected_keys = [f.name for f in fields(BenchmarkExample)]

    if list(record.keys()) != expected_keys:
        problems.append("bad_key_order_or_missing_keys")
    if record.get("domain") != PEOPLE_DOMAIN:
        problems.append("bad_domain")
    if record.get("complexity") not in PEOPLE_LEVEL_ORDER:
        problems.append("bad_complexity")
    if not record.get("query_text_ru") or not record.get("query_text_en"):
        problems.append("missing_ru_or_en_query")
    query_ru = str(record.get("query_text_ru") or "")
    query_en = str(record.get("query_text_en") or "")
    # The public prompt should not expose Wikidata implementation details.
    if re.search(r"подкласс", query_ru, flags=re.IGNORECASE) or re.search(r"subclass", query_en, flags=re.IGNORECASE):
        problems.append("query_text_exposes_subclass_matching")
    # "Well-known" is intentionally not encoded in SPARQL; it is only a
    # natural-language hint to make the prompt sound like real model usage.
    sparql_text = str(record.get("sparql_query") or "")
    ask_text = str(record.get("ask_validator_sparql") or "")
    # Prompts say simply "с профессией X" / "occupation is X", so hidden
    # subclass closure is not allowed in the accepted gold query.
    if "wdt:P106/wdt:P279*" in sparql_text or "wdt:P106/wdt:P279*" in ask_text:
        problems.append("occupation_subclass_path_not_allowed")

    # Birth-year prompts must reject imprecise Wikidata dates such as "20th century".
    # Older records with only wdt:P569 are rejected on resume.
    if ("birth_year_from" in (record.get("constraints") or {}) or re.search(r"родивш|born from", query_ru + " " + query_en, flags=re.IGNORECASE)):
        if "wikibase:timePrecision" not in sparql_text or "wikibase:timePrecision" not in ask_text:
            problems.append("birth_range_without_time_precision_filter")
    constraints = record.get("constraints") or {}
    if _p_cyrillic(_p_json_sig(constraints)):
        problems.append("cyrillic_in_constraints")
    if _p_constraints_have_internal_metadata(constraints):
        problems.append("internal_metadata_in_constraints")

    # Language knowledge (P1412) is too sparse/noisy in Wikidata for this domain.
    # Old records with language constraints are rejected on resume.
    if constraints.get("language_spoken_written_or_signed") or "wdt:P1412" in sparql_text or "wdt:P1412" in ask_text or "language_" in str(record.get("template_id") or ""):
        problems.append("language_criterion_not_allowed")

    # Public award constraints should be concrete named awards, never award_group.
    if constraints.get("award_group") or "одну из наград группы" in query_ru or "award in the" in query_en:
        problems.append("award_group_public_constraint_not_allowed")

    # The bridge is still P19 -> P17 internally, but the user-facing wording
    # should be natural Russian/English.
    if "родившихся в месте, расположенном в стране" in query_ru or "born in a place located in" in query_en:
        problems.append("awkward_birth_country_prompt_wording")
    if "получивших образование в организации, расположенной в стране" in query_ru or "educated at an institution located in" in query_en:
        problems.append("awkward_education_country_prompt_wording")

    # Reject old award-family queries that did not include the direct root-award case.
    if "(wdt:P31|wdt:P279|wdt:P361)+" in sparql_text and "VALUES ?awardReceived" not in sparql_text:
        problems.append("award_family_query_missing_direct_root_case")

    # Reject old nested award-quote wording.
    if "«премия «" in query_ru:
        problems.append("nested_award_quotes_in_query_text")

    # Reject semantically redundant field constraints by QID, not by surface labels.
    # Clean public constraints intentionally contain only English values, while the
    # canonical entity IDs live in gold_collection_meta.bridge_meta.constraint_entities.
    # Labels are used only as a fallback for old files that predate bridge_meta.
    if constraints.get("occupation") and constraints.get("field_of_work"):
        occ_qid = _people_entity_qid_from_record(
            record, entity_key="occupation_qid", public_key="occupation", vocab=OCCUPATIONS
        )
        field_qid = _people_entity_qid_from_record(
            record, entity_key="field_of_work_qid", public_key="field_of_work", vocab=FIELDS
        )

        if not occ_qid or not field_qid:
            problems.append("field_of_work_missing_qid_metadata")
        else:
            if (occ_qid, field_qid) in ROOT_OCCUPATION_FIELD_QID_PAIRS:
                problems.append("redundant_field_of_work_matches_occupation")
            if (occ_qid, field_qid) not in SPECIALIZED_OCCUPATION_FIELD_QID_PAIRS:
                problems.append("field_of_work_not_specialized_for_occupation")

    # Complexity gates: L2 must not be a renamed L1 direct citizenship/birth task.
    if record.get("complexity") == "L2" and record.get("template_id") in {
        "people_occ_citizenship",
        "people_occ_sex_citizenship",
        "people_occ_birthrange_citizenship",
    }:
        problems.append("l2_uses_l1_direct_template")
    # L3+ should be bridge/multi-hop templates, not direct-only occupation/citizenship/award prompts.
    if record.get("complexity") in {"L3", "L4", "L5"} and record.get("template_id") in {
        "people_occ_citizenship",
        "people_occ_sex_citizenship",
        "people_occ_birthrange_citizenship",
        "people_occ_award",
        "people_field_country",
        "people_occ_specialized_field_country",
        "people_language_award_occ",
        "people_award_country_birthrange",
    }:
        problems.append("advanced_level_uses_direct_only_template")

    # L3 must be visibly harder than L2.  Birth-country is a technical Wikidata
    # bridge (P19 -> P17), but for users it reads as a simple direct criterion,
    # so old L3 birthplace-country records are demoted/rejected on resume.
    if record.get("complexity") == "L3" and record.get("template_family") == "birthplace_country_bridge":
        problems.append("l3_birth_country_too_easy")
    if record.get("complexity") == "L3" and constraints.get("birth_country") and not any(
        constraints.get(k) for k in ("education_country", "educated_at_in_country", "member_of_sports_team_in_country", "notable_work_genre")
    ):
        problems.append("l3_birth_country_too_easy")

    # v29: disable notable-work-genre bridge.  It is technically a bridge
    # (P800/P136), but the natural prompt sounds awkward ("known work genre"),
    # and with citizenship/birth filters it was the main L3 speed regression.
    if record.get("template_family") == "notable_work_genre_bridge" or constraints.get("notable_work_genre"):
        problems.append("notable_work_genre_prompt_disabled")

    # v29: L3 should not be merely occupation + one hidden bridge + birth range.
    # Education L3 prompts need an extra visible direct criterion (citizenship)
    # to stay clearly above L2. Sports-team L3 is exempt because it already has
    # the extra playing-position constraint.
    if record.get("complexity") == "L3" and record.get("template_family") in {"education_country_bridge"}:
        if not (constraints.get("citizenship_country") and constraints.get("birth_year_from") is not None):
            problems.append("l3_bridge_missing_extra_citizenship")

    if record.get("complexity") == "L4":
        fam = record.get("template_family")
        if fam == "birthplace_country_bridge" and not (
            constraints.get("birth_country") and constraints.get("citizenship_country") and
            constraints.get("birth_year_from") is not None and constraints.get("birth_year_to") is not None
        ):
            problems.append("l4_birth_country_missing_extra_constraints")
        if fam == "sports_team_country_bridge" and not (
            constraints.get("member_of_sports_team_in_country") and constraints.get("playing_position") and
            constraints.get("citizenship_country") and constraints.get("birth_year_from") is not None
        ):
            problems.append("l4_team_country_too_close_to_l3")

    if record.get("complexity") == "L5":
        fam = record.get("template_family")
        if fam == "sports_team_country_bridge":
            problems.append("l5_sports_team_template_too_close_to_l4")
        if fam == "notable_work_genre_bridge" and not (
            constraints.get("notable_work_genre") and constraints.get("citizenship_country") and
            constraints.get("birth_year_from") is not None
        ):
            problems.append("l5_work_genre_missing_extra_constraints")
        if fam == "birthplace_and_education_bridge" and not (
            constraints.get("birth_country") and (constraints.get("education_country") or constraints.get("educated_at_in_country")) and
            constraints.get("citizenship_country") and constraints.get("birth_year_from") is not None
        ):
            problems.append("l5_two_bridge_missing_extra_constraints")

    # Person citizenship (P27) is only accepted with an explicit modern birth-year
    # range in the user-visible query/constraints. This prevents historical
    # anachronisms like 17th-century figures being returned for modern-country
    # citizenship prompts. Advisor citizenship has its own advisor birth range.
    if constraints.get("citizenship_country") and (constraints.get("birth_year_from") is None or constraints.get("birth_year_to") is None):
        problems.append("person_citizenship_without_birth_range")
    if constraints.get("citizenship_country") and constraints.get("birth_year_from") is not None:
        try:
            if int(constraints.get("birth_year_from")) < 1950:
                problems.append("person_citizenship_birth_range_not_modern")
        except Exception:
            problems.append("bad_birth_year_from")
    if constraints.get("doctoral_advisor_citizenship_country") and (constraints.get("doctoral_advisor_birth_year_from") is None or constraints.get("doctoral_advisor_birth_year_to") is None):
        problems.append("advisor_citizenship_without_advisor_birth_range")
    if constraints.get("doctoral_advisor_citizenship_country") and constraints.get("doctoral_advisor_birth_year_from") is not None:
        try:
            if int(constraints.get("doctoral_advisor_birth_year_from")) < 1950:
                problems.append("advisor_citizenship_birth_range_not_modern")
        except Exception:
            problems.append("bad_advisor_birth_year_from")

    # Award-family public labels such as "Academy Award" or "Nobel Prize" are
    # allowed as award_received.  The SPARQL should match direct root P166 as well
    # as official sub-awards/categories via _p_award_where_lines; it should not use
    # public award_group constraints.

    qids = record.get("gold_answer_qids") or []
    labels_ru = record.get("gold_answer_labels_ru") or []
    labels_en = record.get("gold_answer_labels_en") or []
    n = len(qids)

    if n < int(record.get("requested_count") or 0):
        problems.append("gold_less_than_requested")
    min_gold = PEOPLE_MIN_GOLD_BY_LEVEL.get(record.get("complexity"), int(record.get("requested_count") or 0))
    if n < int(min_gold):
        problems.append("gold_below_min_level_threshold")
    if n != len(labels_ru) or n != len(labels_en):
        problems.append("gold_lengths_mismatch")
    if len(set(qids)) != len(qids):
        problems.append("duplicate_gold_qids")
    if PEOPLE_REJECT_DUPLICATE_ANSWER_LABELS:
        if len(set(str(x).casefold() for x in labels_en)) != len(labels_en):
            problems.append("duplicate_gold_labels_en")
        if len(set(str(x).casefold() for x in labels_ru)) != len(labels_ru):
            problems.append("duplicate_gold_labels_ru")

    max_gold = PEOPLE_MAX_GOLD_BY_LEVEL.get(record.get("complexity"), PEOPLE_GOLD_LIMIT)
    if n > max_gold:
        problems.append("gold_too_broad")

    if record.get("gold_truncated"):
        problems.append("gold_truncated_true")
    if not record.get("sparql_query") or "SELECT DISTINCT" not in record.get("sparql_query"):
        problems.append("missing_select_sparql")
    if not record.get("ask_validator_sparql") or "ASK WHERE" not in record.get("ask_validator_sparql"):
        problems.append("missing_ask_validator")
    lv = record.get("local_validator") or {}
    if not lv or lv.get("type") != "none_wdqs_only":
        problems.append("bad_local_validator")
    if lv and "match_key" not in lv:
        problems.append("local_validator_missing_match_key")

    meta = record.get("gold_collection_meta") or {}
    if not meta:
        problems.append("missing_gold_collection_meta")
    else:
        if meta.get("gold_may_be_incomplete_due_to_wdqs_limit"):
            problems.append("gold_may_be_incomplete")
        if "match_key" not in meta:
            problems.append("gold_meta_missing_match_key")
        # v10 keeps low-level QID/path bridge details under bridge_meta, not as
        # loose top-level gold_collection_meta keys.
        if "constraint_entities" in meta or "constraint_property_paths" in meta or "quality_filters" in meta:
            problems.append("gold_meta_not_cinema_style")
        if "template_id" in meta or "template_family" in meta:
            problems.append("gold_meta_duplicates_top_level_template")
        if "P106/P279*" in _p_json_sig(meta):
            problems.append("gold_meta_contains_old_occupation_subclass_path")

    return problems


def validate_people_records(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    problems = []
    seen_ids = set()
    seen_keys = set()

    for i, r in enumerate(records):
        rid = r.get("id")
        if rid in seen_ids:
            problems.append({"index": i, "id": rid, "problem": "duplicate_id"})
        seen_ids.add(rid)

        key = people_make_record_key(r)
        if key in seen_keys:
            problems.append({"index": i, "id": rid, "problem": "duplicate_query_constraints"})
        seen_keys.add(key)

        for p in people_record_quality_problems(r):
            problems.append({"index": i, "id": rid, "problem": p})

    return {
        "n_examples": len(records),
        "by_level": dict(Counter(r.get("complexity") for r in records)),
        "template_families": dict(Counter(people_record_family(r) for r in records)),
        "templates": dict(Counter(people_record_template(r) for r in records)),
        "gold_count_min": min([people_gold_count(r) for r in records], default=0),
        "gold_count_max": max([people_gold_count(r) for r in records], default=0),
        "problem_count": len(problems),
        "problems_preview": problems[:100],
    }




def people_record_has_award(record: Dict[str, Any]) -> bool:
    constraints = record.get("constraints") or {}
    family = str(record.get("template_family") or "")
    template = str(record.get("template_id") or "")
    return bool(
        constraints.get("award_group")
        or constraints.get("award_received")
        or "award" in family
        or "award" in template
    )


def people_l2_award_limit(target_n: int) -> int:
    return max(1, int(math.floor(int(target_n) * float(PEOPLE_L2_MAX_AWARD_SHARE))))


def people_l2_award_count(records: List[Dict[str, Any]]) -> int:
    return sum(1 for r in records if r.get("complexity") == "L2" and people_record_has_award(r))


def generate_people_example(
    *,
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = PEOPLE_MAX_ATTEMPTS_PER_EXAMPLE,
) -> BenchmarkExample:
    last_error = None

    for _attempt in range(int(max_attempts)):
        try:
            tpl = _p_pick_template(complexity, rng)
            ex = tpl(complexity, idx, rng)
            if ex is None:
                continue
            ex.id = f"people_{complexity.lower()}_{idx:04d}"
            return ex
        except KeyboardInterrupt:
            raise
        except Exception as e:
            last_error = e
            if globals().get("DEBUG_GENERATOR_ERRORS", False):
                    continue

    raise RuntimeError(f"Could not generate valid people example for {complexity} after {max_attempts} attempts; last_error={last_error}")


def people_current_audit(records: List[Dict[str, Any]], skipped: List[Dict[str, Any]], bad_lines: List[Dict[str, Any]]):
    return {
        "domain": PEOPLE_DOMAIN,
        "target_plan": PEOPLE_TARGET_PER_LEVEL,
        "quality_filters": {
            "max_gold_by_level": PEOPLE_MAX_GOLD_BY_LEVEL,
            "reject_duplicate_answer_labels": PEOPLE_REJECT_DUPLICATE_ANSWER_LABELS,
            "ru_label_required": False,
            "label_languages": ["ru", "en"],
            "min_birth_date_precision": int(PEOPLE_MIN_BIRTH_DATE_PRECISION),
            "person_citizenship_birth_year_min": 1950,
            "award_family_match_keys": sorted(PEOPLE_AWARD_FAMILY_MATCH_KEYS),
            "constraints_policy": "public English-only constraints; QIDs/property paths are stored in gold_collection_meta.bridge_meta",
            "query_text_policy": "do not expose Wikidata subclass/path wording in RU/EN prompt text",
            "max_same_occupation_per_level": PEOPLE_MAX_SAME_OCCUPATION_PER_LEVEL,
            "field_specialization_policy": "occupation/field compatibility is validated by Wikidata QID pairs, with label fallback only for legacy records",
            "language_policy": "P1412 language knowledge is forbidden because it is sparse/noisy in Wikidata",
            "award_policy": "public constraints use award_received; famous award families match direct root award or official subaward/category internally",
            "l2_template_policy": "L2 is direct multi-criteria but award templates are capped; non-award field/sports-position templates are preferred; language constraints are forbidden",
            "l2_max_award_share": PEOPLE_L2_MAX_AWARD_SHARE,
            "template_family_caps_by_level": PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL,
            "max_consecutive_same_family_by_level": PEOPLE_MAX_CONSECUTIVE_SAME_FAMILY_BY_LEVEL,
            "l3_policy": "L3 requires a non-trivial bridge; education/work-genre L3 also require citizenship + modern birth range, while sports-team L3 requires playing position + birth range",
            "l4_l5_policy": "L4 requires bridge plus extra constraints or two bridges; L5 requires related-human/work constraints or two bridges plus citizenship/birth range",
        },
        "records_total": len(records),
        "counts_by_complexity": dict(Counter(r.get("complexity") for r in records)),
        "counts_by_complexity_and_template_family": {
            level: dict(Counter(people_record_family(r) for r in records if r.get("complexity") == level))
            for level in PEOPLE_TARGET_PER_LEVEL
        },
        "template_counts": dict(Counter(people_record_template(r) for r in records)),
        "gold_count": {
            "min": min([people_gold_count(r) for r in records], default=0),
            "max": max([people_gold_count(r) for r in records], default=0),
        },
        "bad_json_lines": len(bad_lines),
        "bad_json_preview": bad_lines[:20],
        "skipped_count": len(skipped),
        "skipped_preview": skipped[-100:],
        "output_path": str(PEOPLE_OUTPUT_PATH),
        "audit_path": str(PEOPLE_AUDIT_PATH),
        "checkpoint_path": str(PEOPLE_CHECKPOINT_PATH),
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

print("✅ Generation helpers loaded")


## 9. Generation engine — v31 hotfix

Overrides the base template registry with faster, cleaner L3/L5 people templates.


In [ ]:
PEOPLE_OUTPUT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v31_fast_clean.jsonl"
PEOPLE_AUDIT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v31_fast_clean_audit.json"
PEOPLE_CHECKPOINT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v31_fast_clean_checkpoint.json"

# Focused supplement: your current curated file is already large; this adds the
# missing L3 variety and better hard L5 alternatives. Raise these numbers if you
# want a larger supplement after checking speed on your WDQS connection.
PEOPLE_TARGET_PER_LEVEL = {
    "L1": 0,
    "L2": 0,
    "L3": 35,
    "L4": 0,
    "L5": 35,
}

PEOPLE_RANDOM_SEED = 8131
PEOPLE_MAX_ATTEMPTS_PER_LEVEL = 1800
PEOPLE_MAX_ATTEMPTS_PER_EXAMPLE = 80
PEOPLE_WDQS_LIMIT_DEFAULT = 101  # > max accepted gold 50; still detects broad/incomplete queries
PEOPLE_MAX_GOLD_BY_LEVEL.update({"L3": 50, "L5": 50})
PEOPLE_MIN_GOLD_BY_LEVEL.update({"L3": 8, "L5": 8})
PEOPLE_MAX_SAME_OCCUPATION_PER_LEVEL.update({"L3": 7, "L5": 8})

# Be much less patient with slow failed candidates. Accepted patterns below are
# deliberately selective, so a query that needs repeated long retries is usually
# a bad candidate rather than a useful example.
try:
    wd.timeout = 25
    wd.max_retries = 1
except Exception:
    pass

# Try harder to find the existing curated file for dedup/id continuation. This
# prevents regenerating variants that are already present when the notebook is
# run from a project subdirectory.
_ref_candidates = [
    Path("people_final_curated_v2.jsonl"),
    Path("08_people") / "people_final_curated_v2.jsonl",
    Path("data") / "people_final_curated_v2.jsonl",
    PEOPLE_OUTPUT_DIR / "people_final_curated_v2.jsonl",
    PEOPLE_OUTPUT_DIR / "people.jsonl",
]
try:
    for _p in list(Path.cwd().glob("**/people_final_curated_v2.jsonl"))[:20]:
        _ref_candidates.append(_p)
except Exception:
    pass
PEOPLE_REFERENCE_JSONL_PATHS = list(dict.fromkeys(_ref_candidates))

# Smaller, high-yield pools. These avoid random combinations that are technically
# valid but almost never produce enough golds.
FAST_CREATIVE_COUNTRIES = ["United States", "United Kingdom", "France", "Germany", "Italy", "Spain", "Japan", "Canada", "Australia"]
FAST_SCIENCE_COUNTRIES = ["United States", "United Kingdom", "France", "Germany", "Italy", "Canada", "Australia", "Netherlands", "Sweden"]
FAST_LANG_BY_COUNTRY = {
    "United States": "English",
    "United Kingdom": "English",
    "Canada": "English",
    "Australia": "English",
    "France": "French",
    "Germany": "German",
    "Italy": "Italian",
    "Spain": "Spanish",
    "Japan": "Japanese",
}
FAST_FILM_LANGUAGES = ["English", "French", "German", "Italian", "Spanish", "Japanese"]
FAST_WORK_LANGUAGES = ["English", "French", "German", "Italian", "Spanish", "Russian", "Japanese", "Polish"]
FAST_FILM_GENRES = ["drama", "comedy", "action film", "thriller film", "science fiction"]
FAST_WORK_GENRES = ["science fiction", "crime fiction", "drama", "comedy"]


def _p_fast_country_and_language(rng: random.Random) -> Tuple[str, str]:
    country_key = _p_choice(rng, FAST_CREATIVE_COUNTRIES)
    lang_key = FAST_LANG_BY_COUNTRY.get(country_key) or _p_choice(rng, FAST_FILM_LANGUAGES)
    return country_key, lang_key


def _tpl_directed_film_lang_genre_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: film director -> directed film -> original language + genre.

    Replaces v30's directed_film_country_of_origin pattern. Original language is
    a clearer, user-facing film criterion than Wikidata's P495 country of origin.
    """
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "film director"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None

    citizenship_key, lang_key = _p_fast_country_and_language(rng)
    genre_key = _p_choice(rng, FAST_FILM_GENRES)
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    sex_key = _p_choice(rng, list(SEXES.keys())) if rng.random() < 0.35 else None

    occ = OCCUPATIONS[occ_key]
    citizenship_country = COUNTRIES[citizenship_key]
    lang = LANGUAGES[lang_key]
    genre = FILM_GENRES[genre_key]

    # Bind film first; this is typically much faster than scanning all directors.
    where = [
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        f"?film wdt:P364 wd:{lang['qid']} .",
        "?film wdt:P57 ?person .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [
        ru_head,
        _occ_phrase_ru(occ_key),
        f"снявших фильм жанра «{genre['ru']}» на языке оригинала: {lang['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ]
    en_parts = [
        en_head,
        _occ_phrase_en(occ_key),
        f"who directed a {genre['en']} film whose original language is {lang['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))

    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "directed_film_genre": genre["en"],
        "directed_film_genre_qid": genre["qid"],
        "directed_film_original_language": lang["en"],
        "directed_film_original_language_qid": lang["qid"],
        "directed_film_match_path": "inverse P57 + P136/P364",
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_directed_film_lang_genre_fast",
        template_family="directed_film_lang_genre_bridge", q_ru=_ru_join(ru_parts), q_en=_en_join(en_parts),
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person <- directed film -> film genre and original language"},
    )


def _tpl_author_work_lang_genre_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: writer/poet -> authored work -> work genre + language."""
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = _p_choice(rng, ["writer", "poet"])
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    country_key = _p_choice(rng, FAST_CREATIVE_COUNTRIES + ["Poland", "Russia"])
    lang_key = FAST_LANG_BY_COUNTRY.get(country_key) or _p_choice(rng, FAST_WORK_LANGUAGES)
    if lang_key not in LANGUAGES:
        lang_key = _p_choice(rng, FAST_WORK_LANGUAGES)
    genre_key = _p_choice(rng, FAST_WORK_GENRES)
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    country = COUNTRIES[country_key]
    lang = LANGUAGES[lang_key]
    genre = LITERARY_WORK_GENRES[genre_key]

    where = [
        f"?work wdt:P136 wd:{genre['qid']} .",
        f"?work wdt:P407 wd:{lang['qid']} .",
        "?work wdt:P50 ?person .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        f"у которых есть авторское произведение жанра «{genre['ru']}» на языке: {lang['ru']}",
        _country_phrase_ru(country_key),
        _birth_phrase_ru(start, end),
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        f"who authored a work in the genre {genre['en']} whose language is {lang['en']}",
        _country_phrase_en(country_key),
        _birth_phrase_en(start, end),
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "authored_work_genre": genre["en"],
        "authored_work_genre_qid": genre["qid"],
        "authored_work_language": lang["en"],
        "authored_work_language_qid": lang["qid"],
        "authored_work_match_path": "inverse P50 + P136/P407",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_author_work_lang_genre_fast",
        template_family="authored_work_lang_genre_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person <- authored work -> work genre and work language"},
    )


def _tpl_employer_country_bridge_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: person -> employer -> employer country, plus visible direct filters."""
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_pool = ["mathematician", "physicist", "chemist", "biologist", "economist", "historian", "journalist", "architect"]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, occ_pool)
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    employer_country_key = _p_choice(rng, MAJOR_EMPLOYER_COUNTRIES)
    citizenship_key = _p_choice(rng, FAST_SCIENCE_COUNTRIES)
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    employer_country = COUNTRIES[employer_country_key]
    citizenship_country = COUNTRIES[citizenship_key]
    where = [
        f"?employer wdt:P17 wd:{employer_country['qid']} .",
        "?person wdt:P108 ?employer .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        f"работавших в организации, связанной со страной: {employer_country['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        f"who worked for an employer associated with {employer_country['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "employer_country": employer_country["en"],
        "employer_country_qid": employer_country["qid"],
        "employer_country_path": "P108/P17",
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_employer_country_fast",
        template_family="employer_country_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> employer -> employer country"},
    )


def _tpl_party_country_bridge_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: politician -> political party -> party country."""
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "politician"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    party_country_key = _p_choice(rng, MAJOR_PARTY_COUNTRIES)
    citizenship_key = party_country_key if rng.random() < 0.75 else _p_choice(rng, MAJOR_PARTY_COUNTRIES)
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)
    sex_key = _p_choice(rng, list(SEXES.keys())) if rng.random() < 0.30 else None

    occ = OCCUPATIONS[occ_key]
    party_country = COUNTRIES[party_country_key]
    citizenship_country = COUNTRIES[citizenship_key]
    where = [
        f"?party wdt:P17 wd:{party_country['qid']} .",
        "?person wdt:P102 ?party .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [
        ru_head,
        _occ_phrase_ru(occ_key),
        f"состоявших в политической партии страны: {party_country['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ]
    en_parts = [
        en_head,
        _occ_phrase_en(occ_key),
        f"who were members of a political party from {party_country['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))

    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "political_party_country": party_country["en"],
        "political_party_country_qid": party_country["qid"],
        "political_party_country_path": "P102/P17",
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None,
        "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_party_country_fast",
        template_family="political_party_country_bridge", q_ru=_ru_join(ru_parts), q_en=_en_join(en_parts),
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> political party -> party country"},
    )


def _tpl_football_team_country_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L3: football player -> team -> country + position."""
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "association football player"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    team_country_key = _p_choice(rng, ["United Kingdom", "France", "Germany", "Italy", "Spain", "Brazil", "Argentina", "Netherlands"])
    citizenship_key = team_country_key if rng.random() < 0.55 else _p_choice(rng, ["Brazil", "Argentina", "France", "Germany", "Italy", "Spain", "Netherlands"])
    pos_key = _p_choice(rng, list(FOOTBALL_POSITIONS.keys()))
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    team_country = COUNTRIES[team_country_key]
    citizenship_country = COUNTRIES[citizenship_key]
    pos = FOOTBALL_POSITIONS[pos_key]
    where = [
        f"?team wdt:P17 wd:{team_country['qid']} .",
        "?person wdt:P54 ?team .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P413 wd:{pos['qid']} .",
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        f"игравших за футбольный клуб из страны: {team_country['ru']}",
        f"с игровой позицией: {pos['ru']}",
        _country_phrase_ru(citizenship_key),
        _birth_phrase_ru(start, end),
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        f"who played for a football club from {team_country['en']}",
        f"whose playing position is {pos['en']}",
        _country_phrase_en(citizenship_key),
        _birth_phrase_en(start, end),
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "member_of_sports_team_in_country": team_country["en"],
        "member_of_sports_team_in_country_qid": team_country["qid"],
        "member_of_sports_team_country_path": "P54/P17",
        "playing_position": pos["en"],
        "playing_position_qid": pos["qid"],
        "citizenship_country": citizenship_country["en"],
        "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_football_team_country_fast",
        template_family="sports_team_country_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> sports team -> team country; person -> playing position"},
    )


### 9b. v31 — film, writer, advisor and family-relation templates

In [ ]:
def _tpl_actor_film_director_award_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: actor -> film -> director -> director award, plus film genre/language."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "actor"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    actor_country_key, lang_key = _p_fast_country_and_language(rng)
    genre_key = _p_choice(rng, FAST_FILM_GENRES)
    director_award_key = _p_choice(rng, ["Academy Award", "BAFTA Award", "César Award", "Grammy Award"])
    actor_start, actor_end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    actor_country = COUNTRIES[actor_country_key]
    lang = LANGUAGES[lang_key]
    genre = FILM_GENRES[genre_key]
    award = AWARDS[director_award_key]
    where = [
        *_p_award_where_lines_for_var(director_award_key, var="director", value_var="?directorAwardReceived"),
        "?director wdt:P31 wd:Q5 .",
        "?film wdt:P57 ?director .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        f"?film wdt:P364 wd:{lang['qid']} .",
        "?film wdt:P161 ?person .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{actor_country['qid']} .",
        *_p_birth_range_lines_scoped(actor_start, actor_end, var="person", scope="person"),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(actor_country_key),
        _birth_phrase_ru(actor_start, actor_end),
        f"снимавшихся в фильме жанра «{genre['ru']}» на языке оригинала: {lang['ru']}",
        f"режиссёр которого получил награду: {award['ru']}",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(actor_country_key),
        _birth_phrase_en(actor_start, actor_end),
        f"who acted in a {genre['en']} film whose original language is {lang['en']}",
        f"whose director received the {award['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": actor_country["en"],
        "citizenship_country_qid": actor_country["qid"],
        "birth_year_from": actor_start,
        "birth_year_to": actor_end,
        "acted_in_film_genre": genre["en"],
        "acted_in_film_genre_qid": genre["qid"],
        "acted_in_film_original_language": lang["en"],
        "acted_in_film_original_language_qid": lang["qid"],
        "film_director_award_received": award["en"],
        "film_director_award_received_qid": award["qid"],
        "actor_film_director_award_path": "inverse P161/P57/P166 + film P136/P364",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_actor_film_director_award_fast",
        template_family="actor_film_director_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta={"bridge": "actor <- film -> director -> director award; film -> genre/original language"},
    )


def _tpl_director_film_cast_award_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: director -> film -> cast member -> cast award, plus film genre/language."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "film director"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    director_country_key, lang_key = _p_fast_country_and_language(rng)
    genre_key = _p_choice(rng, FAST_FILM_GENRES)
    cast_award_key = _p_choice(rng, ["Academy Award", "BAFTA Award", "César Award", "Grammy Award"])
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    director_country = COUNTRIES[director_country_key]
    lang = LANGUAGES[lang_key]
    genre = FILM_GENRES[genre_key]
    cast_award = AWARDS[cast_award_key]
    where = [
        *_p_award_where_lines_for_var(cast_award_key, var="castMember", value_var="?castAwardReceived"),
        "?castMember wdt:P31 wd:Q5 .",
        "?film wdt:P161 ?castMember .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        f"?film wdt:P364 wd:{lang['qid']} .",
        "?film wdt:P57 ?person .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{director_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(director_country_key),
        _birth_phrase_ru(start, end),
        f"снявших фильм жанра «{genre['ru']}» на языке оригинала: {lang['ru']}",
        f"в актёрском составе которого есть человек, получивший награду: {cast_award['ru']}",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(director_country_key),
        _birth_phrase_en(start, end),
        f"who directed a {genre['en']} film whose original language is {lang['en']}",
        f"whose cast includes a person who received the {cast_award['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": director_country["en"],
        "citizenship_country_qid": director_country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "directed_film_genre": genre["en"],
        "directed_film_genre_qid": genre["qid"],
        "directed_film_original_language": lang["en"],
        "directed_film_original_language_qid": lang["qid"],
        "cast_member_award_received": cast_award["en"],
        "cast_member_award_received_qid": cast_award["qid"],
        "cast_member_award_bridge_path": "inverse P57/P161/P166 + film P136/P364",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_director_film_cast_award_fast",
        template_family="director_film_cast_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "director <- film -> cast member -> cast award; film -> genre/original language"},
    )


def _tpl_writer_work_film_adaptation_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: writer -> authored work -> film adaptation -> film genre."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "writer"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    country_key = _p_choice(rng, ["United States", "United Kingdom", "France", "Germany", "Italy", "Spain", "Japan", "Russia", "Poland", "Canada"])
    lang_key = FAST_LANG_BY_COUNTRY.get(country_key) or _p_choice(rng, FAST_WORK_LANGUAGES)
    if lang_key not in LANGUAGES:
        lang_key = _p_choice(rng, FAST_WORK_LANGUAGES)
    work_genre_key = _p_choice(rng, FAST_WORK_GENRES)
    film_genre_key = _p_choice(rng, ["drama", "comedy", "thriller film", "science fiction"])
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    country = COUNTRIES[country_key]
    lang = LANGUAGES[lang_key]
    work_genre = LITERARY_WORK_GENRES[work_genre_key]
    film_genre = FILM_GENRES[film_genre_key]
    where = [
        f"?work wdt:P136 wd:{work_genre['qid']} .",
        f"?work wdt:P407 wd:{lang['qid']} .",
        "?work wdt:P50 ?person .",
        "?film wdt:P144 ?work .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{film_genre['qid']} .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(country_key),
        _birth_phrase_ru(start, end),
        f"у которых есть авторское произведение жанра «{work_genre['ru']}» на языке: {lang['ru']}",
        f"и по этому произведению снят фильм жанра «{film_genre['ru']}»",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(country_key),
        _birth_phrase_en(start, end),
        f"who authored a work in the genre {work_genre['en']} whose language is {lang['en']}",
        f"and a {film_genre['en']} film is based on that work",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "authored_work_genre": work_genre["en"],
        "authored_work_genre_qid": work_genre["qid"],
        "authored_work_language": lang["en"],
        "authored_work_language_qid": lang["qid"],
        "adaptation_film_genre": film_genre["en"],
        "adaptation_film_genre_qid": film_genre["qid"],
        "adaptation_bridge_path": "inverse P50 + P136/P407; film P144/P136",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_writer_work_film_adaptation_fast",
        template_family="writer_work_film_adaptation_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "writer <- authored work -> film adaptation -> film genre"},
    )


### 9c. v31 — advisor award, spouse/parent award, and quality override

In [ ]:
def _tpl_advisor_award_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5: scholar -> doctoral advisor -> advisor award, plus specialized field."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, ["mathematician", "physicist", "chemist", "biologist", "economist"])
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    field_key = _p_choice(rng, FIELD_SPECIALIZATIONS_BY_OCCUPATION[occ_key])
    award_key = _p_choice(rng, {
        "mathematician": ["Fields Medal", "Wolf Prize"],
        "physicist": ["Nobel Prize", "Wolf Prize"],
        "chemist": ["Nobel Prize", "Wolf Prize"],
        "biologist": ["Nobel Prize", "Wolf Prize"],
        "economist": ["Nobel Prize", "Wolf Prize"],
    }[occ_key])
    advisor_start, advisor_end = rng.choice(RELATED_PERSON_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    field = FIELDS[field_key]
    award = AWARDS[award_key]
    where = [
        *_p_award_where_lines_for_var(award_key, var="advisor", value_var="?advisorAwardReceived"),
        "?advisor wdt:P31 wd:Q5 .",
        *_p_birth_range_lines_scoped(advisor_start, advisor_end, var="advisor", scope="advisor"),
        "?person wdt:P184 ?advisor .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P101 wd:{field['qid']} .",
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _field_phrase_ru(field_key),
        f"у которых научный руководитель получил награду: {award['ru']}",
        _advisor_birth_phrase_ru(advisor_start, advisor_end),
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _field_phrase_en(field_key),
        f"whose doctoral advisor received the {award['en']}",
        _advisor_birth_phrase_en(advisor_start, advisor_end),
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "field_of_work": field["en"],
        "field_of_work_qid": field["qid"],
        "doctoral_advisor_award_received": award["en"],
        "doctoral_advisor_award_received_qid": award["qid"],
        "doctoral_advisor_award_path": "P184/P166",
        "doctoral_advisor_birth_year_from": advisor_start,
        "doctoral_advisor_birth_year_to": advisor_end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_advisor_award_fast",
        template_family="advisor_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta={"bridge": "person -> doctoral advisor -> advisor award"},
    )


def _tpl_spouse_award_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5 fallback: person -> spouse -> spouse occupation + spouse award."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    spouse_occ_key = _p_choice(rng, ["actor", "film director", "writer", "singer", "composer"])
    award_key = _p_choice(rng, AWARDS_BY_OCCUPATION[spouse_occ_key])
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, ["actor", "film director", "writer", "singer", "politician", "journalist"])
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    country_key = _p_choice(rng, FAST_CREATIVE_COUNTRIES)
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    spouse_occ = OCCUPATIONS[spouse_occ_key]
    country = COUNTRIES[country_key]
    award = AWARDS[award_key]
    where = [
        f"?spouse wdt:P106 wd:{spouse_occ['qid']} .",
        *_p_award_where_lines_for_var(award_key, var="spouse", value_var="?spouseAwardReceived"),
        "?spouse wdt:P31 wd:Q5 .",
        "?person wdt:P26 ?spouse .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(country_key),
        _birth_phrase_ru(start, end),
        f"у которых супруг или супруга имеет профессию «{spouse_occ['ru']}» и получил(а) награду: {award['ru']}",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(country_key),
        _birth_phrase_en(start, end),
        f"whose spouse's occupation is {spouse_occ['en']} and whose spouse received the {award['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "spouse_occupation": spouse_occ["en"],
        "spouse_occupation_qid": spouse_occ["qid"],
        "spouse_award_received": award["en"],
        "spouse_award_received_qid": award["qid"],
        "spouse_bridge_path": "P26/P106/P166",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_spouse_award_fast",
        template_family="spouse_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> spouse -> spouse occupation and award"},
    )


def _tpl_parent_award_fast(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """L5 fallback: person -> parent -> parent occupation + parent award."""
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    parent_occ_key = _p_choice(rng, ["actor", "film director", "writer", "singer", "composer", "physicist"])
    award_key = _p_choice(rng, AWARDS_BY_OCCUPATION[parent_occ_key])
    occ_candidates = _p_runtime_allowed_occupation_keys(complexity, ["actor", "film director", "writer", "singer", "politician", "journalist"])
    if not occ_candidates:
        return None
    occ_key = _p_choice(rng, occ_candidates)
    country_key = _p_choice(rng, FAST_CREATIVE_COUNTRIES)
    start, end = rng.choice(MODERN_CITIZENSHIP_BIRTH_RANGES)

    occ = OCCUPATIONS[occ_key]
    parent_occ = OCCUPATIONS[parent_occ_key]
    country = COUNTRIES[country_key]
    award = AWARDS[award_key]
    where = [
        f"?parent wdt:P106 wd:{parent_occ['qid']} .",
        *_p_award_where_lines_for_var(award_key, var="parent", value_var="?parentAwardReceived"),
        "?parent wdt:P31 wd:Q5 .",
        "?person wdt:P22|wdt:P25 ?parent .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([
        ru_head,
        _occ_phrase_ru(occ_key),
        _country_phrase_ru(country_key),
        _birth_phrase_ru(start, end),
        f"у которых отец или мать имеет профессию «{parent_occ['ru']}» и получил(а) награду: {award['ru']}",
    ])
    q_en = _en_join([
        en_head,
        _occ_phrase_en(occ_key),
        _country_phrase_en(country_key),
        _birth_phrase_en(start, end),
        f"whose parent is a {parent_occ['en']} and whose parent received the {award['en']}",
    ])
    constraints = {
        "kind": "human",
        "occupation": occ["en"],
        "occupation_qid": occ["qid"],
        "occupation_match": "P106",
        "citizenship_country": country["en"],
        "citizenship_country_qid": country["qid"],
        "birth_year_from": start,
        "birth_year_to": end,
        "parent_occupation": parent_occ["en"],
        "parent_occupation_qid": parent_occ["qid"],
        "parent_award_received": award["en"],
        "parent_award_received_qid": award["qid"],
        "parent_bridge_path": "P22/P25 -> P106/P166",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_parent_award_fast",
        template_family="parent_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> parent -> parent occupation and award"},
    )


# Registry rewritten for v31. Old country-of-origin template is intentionally not present.
PEOPLE_TEMPLATES_BY_LEVEL = {
    "L1": [],
    "L2": [],
    "L3": [
        (_tpl_author_work_lang_genre_fast, 0.20),
        (_tpl_directed_film_lang_genre_fast, 0.18),
        (_tpl_employer_country_bridge_fast, 0.18),
        (_tpl_party_country_bridge_fast, 0.14),
        (_tpl_football_team_country_fast, 0.18),
        (_tpl_educated_in_country, 0.12),
    ],
    "L4": [],
    "L5": [
        (_tpl_actor_film_director_award_fast, 0.22),
        (_tpl_director_film_cast_award_fast, 0.20),
        (_tpl_writer_work_film_adaptation_fast, 0.15),
        (_tpl_advisor_award_fast, 0.18),
        (_tpl_spouse_award_fast, 0.13),
        (_tpl_parent_award_fast, 0.12),
    ],
}

PEOPLE_TEMPLATE_FN_TO_FAMILY.update({
    "_tpl_author_work_lang_genre_fast": "authored_work_lang_genre_bridge",
    "_tpl_directed_film_lang_genre_fast": "directed_film_lang_genre_bridge",
    "_tpl_employer_country_bridge_fast": "employer_country_bridge",
    "_tpl_party_country_bridge_fast": "political_party_country_bridge",
    "_tpl_football_team_country_fast": "sports_team_country_bridge",
    "_tpl_actor_film_director_award_fast": "actor_film_director_award_bridge",
    "_tpl_director_film_cast_award_fast": "director_film_cast_award_bridge",
    "_tpl_writer_work_film_adaptation_fast": "writer_work_film_adaptation_bridge",
    "_tpl_advisor_award_fast": "advisor_award_bridge",
    "_tpl_spouse_award_fast": "spouse_award_bridge",
    "_tpl_parent_award_fast": "parent_award_bridge",
})

PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL = {
    "L3": {
        "authored_work_lang_genre_bridge": 0.25,
        "directed_film_lang_genre_bridge": 0.24,
        "employer_country_bridge": 0.24,
        "political_party_country_bridge": 0.20,
        "sports_team_country_bridge": 0.24,
        "education_country_bridge": 0.20,
    },
    "L5": {
        "actor_film_director_award_bridge": 0.24,
        "director_film_cast_award_bridge": 0.24,
        "writer_work_film_adaptation_bridge": 0.20,
        "advisor_award_bridge": 0.22,
        "spouse_award_bridge": 0.16,
        "parent_award_bridge": 0.16,
        "spouse_occupation_bridge": 0.00,
        "parent_occupation_bridge": 0.00,
    },
}
PEOPLE_L3_TEMPLATE_FAMILY_CAPS = PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL["L3"]
PEOPLE_MAX_CONSECUTIVE_SAME_FAMILY_BY_LEVEL = {"L3": 2, "L5": 2}

# Explicit quality guard so existing/bad rows from v30 are cleaned on resume and
# the old criterion cannot silently return through an old output file. Keep this
# idempotent so rerunning the cell in Jupyter does not wrap the function twice.
if "_people_record_quality_problems_base_v31" not in globals():
    _people_record_quality_problems_base_v31 = people_record_quality_problems

def people_record_quality_problems(record: Dict[str, Any]) -> List[str]:
    problems = list(_people_record_quality_problems_base_v31(record))
    query_text = f"{record.get('query_text_ru') or ''} {record.get('query_text_en') or ''}"
    constraints = record.get("constraints") or {}
    sparql_text = f"{record.get('sparql_query') or ''} {record.get('ask_validator_sparql') or ''}"
    meta_text = _p_json_sig(record.get("gold_collection_meta") or {})
    if (
        constraints.get("directed_film_country_of_origin")
        or "country of origin" in query_text.casefold()
        or "страной происхождения" in query_text.casefold()
        or "directed_film_country_of_origin" in meta_text
        or "wdt:P495" in sparql_text
    ):
        problems.append("film_country_of_origin_criterion_disabled")
    return problems

print("✅ v31 people hotfix loaded:")
print("   output:", PEOPLE_OUTPUT_PATH.resolve())
print("   targets:", PEOPLE_TARGET_PER_LEVEL)
print("   WDQS timeout/retries:", getattr(wd, "timeout", None), getattr(wd, "max_retries", None))
print("   L3 families:", [PEOPLE_TEMPLATE_FN_TO_FAMILY.get(fn.__name__) for fn, _ in PEOPLE_TEMPLATES_BY_LEVEL["L3"]])
print("   L5 families:", [PEOPLE_TEMPLATE_FN_TO_FAMILY.get(fn.__name__) for fn, _ in PEOPLE_TEMPLATES_BY_LEVEL["L5"]])


## 10. Generation engine — v32 hotfix

Overrides the v31 registry and quality filters with cleaner L3/L5 people templates, stricter wording checks, and a new candidate-selection strategy.


## 11. Generation engine — v33 hotfix

Overrides the v32 registry with stricter wording, curated award/film combinations, and improved label-quality guards.


In [ ]:
PEOPLE_OUTPUT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v33_fast_quality_keep_famous.jsonl"
PEOPLE_AUDIT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v33_fast_quality_keep_famous_audit.json"
PEOPLE_CHECKPOINT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v33_fast_quality_keep_famous_checkpoint.json"

PEOPLE_TARGET_PER_LEVEL = {
    "L1": 0,
    "L2": 0,
    "L3": 35,
    "L4": 0,
    "L5": 35,
}

PEOPLE_RANDOM_SEED = 8132
PEOPLE_MAX_ATTEMPTS_PER_LEVEL = 900
PEOPLE_MAX_ATTEMPTS_PER_EXAMPLE = 35
PEOPLE_WDQS_LIMIT_DEFAULT = 81
PEOPLE_MIN_GOLD_BY_LEVEL.update({"L3": 8, "L5": 6})
PEOPLE_MAX_GOLD_BY_LEVEL.update({"L3": 60, "L5": 70})
PEOPLE_MAX_SAME_OCCUPATION_PER_LEVEL.update({"L3": 10, "L5": 12})

# Give bad candidates less time. The v32 templates are intended to be selective;
# a slow query is usually a bad parameter combination, not something worth waiting on.
try:
    wd.timeout = 18
    wd.max_retries = 1
except Exception:
    pass

# Public query head: the "well-known" phrase is kept as natural benchmark wording only.
# It is intentionally NOT encoded as a notability/fame filter in SPARQL.
def _p_head(requested_count: int) -> Tuple[str, str]:
    return f"Назови {requested_count} известных персон", f"Name {requested_count} well-known people"


def _p_article_en(phrase: str) -> str:
    phrase = str(phrase or "").strip()
    if not phrase:
        return "a"
    article = "an" if phrase[0].lower() in set("aeiou") else "a"
    return f"{article} {phrase}"


def _p_film_genre_noun_en(genre_key: str) -> str:
    label = FILM_GENRES[genre_key]["en"].strip()
    return label if label.casefold().endswith("film") else f"{label} film"


def _p_film_genre_phrase_en(genre_key: str) -> str:
    return _p_article_en(_p_film_genre_noun_en(genre_key))


# Curated parameter pools. They are intentionally small and conservative: the
# goal is fewer rejected WDQS calls, not exhaustive random exploration.
L3_DIRECTED_FILM_COMBOS_V32 = [
    {"country": "United Kingdom", "language": "English", "genre": "drama", "birth": (1970, 1989), "sex": "female"},
    {"country": "Japan", "language": "Japanese", "genre": "thriller film", "birth": (1950, 1969), "sex": "male"},
    {"country": "France", "language": "French", "genre": "drama", "birth": (1950, 1969), "sex": None},
    {"country": "Italy", "language": "Italian", "genre": "drama", "birth": (1950, 1969), "sex": None},
    {"country": "Spain", "language": "Spanish", "genre": "drama", "birth": (1950, 1969), "sex": None},
    {"country": "Canada", "language": "English", "genre": "drama", "birth": (1970, 1989), "sex": None},
    {"country": "Australia", "language": "English", "genre": "comedy", "birth": (1970, 1989), "sex": None},
    {"country": "United States", "language": "English", "genre": "science fiction", "birth": (1970, 1989), "sex": "female"},
]

L3_AUTHOR_WORK_COMBOS_V32 = [
    {"occupation": "writer", "country": "United States", "language": "English", "genre": "crime fiction", "birth": (1950, 1969), "sex": None},
    {"occupation": "writer", "country": "United Kingdom", "language": "English", "genre": "science fiction", "birth": (1950, 1969), "sex": None},
    {"occupation": "writer", "country": "France", "language": "French", "genre": "drama", "birth": (1950, 1969), "sex": None},
    {"occupation": "writer", "country": "Germany", "language": "German", "genre": "crime fiction", "birth": (1950, 1969), "sex": None},
    {"occupation": "writer", "country": "Spain", "language": "Spanish", "genre": "drama", "birth": (1950, 1969), "sex": None},
    {"occupation": "writer", "country": "Japan", "language": "Japanese", "genre": "science fiction", "birth": (1950, 1969), "sex": None},
    {"occupation": "poet", "country": "United States", "language": "English", "genre": "drama", "birth": (1950, 1969), "sex": None},
    {"occupation": "writer", "country": "Canada", "language": "English", "genre": "crime fiction", "birth": (1970, 1989), "sex": None},
]

L3_EMPLOYER_COMBOS_V32 = [
    {"occupation": "economist", "employer_country": "United States", "citizenship": "France", "birth": (1970, 1989)},
    {"occupation": "economist", "employer_country": "United States", "citizenship": "United Kingdom", "birth": (1950, 1969)},
    {"occupation": "mathematician", "employer_country": "United States", "citizenship": "France", "birth": (1950, 1969)},
    {"occupation": "physicist", "employer_country": "United States", "citizenship": "Germany", "birth": (1950, 1969)},
    {"occupation": "biologist", "employer_country": "United States", "citizenship": "United Kingdom", "birth": (1950, 1969)},
    {"occupation": "historian", "employer_country": "United Kingdom", "citizenship": "United States", "birth": (1950, 1969)},
    {"occupation": "journalist", "employer_country": "United States", "citizenship": "Canada", "birth": (1970, 1989)},
    {"occupation": "architect", "employer_country": "United States", "citizenship": "Italy", "birth": (1950, 1969)},
]

L3_EDUCATION_COMBOS_V32 = [
    {"occupation": "economist", "education_country": "United States", "citizenship": "France", "birth": (1970, 1989)},
    {"occupation": "mathematician", "education_country": "United States", "citizenship": "Germany", "birth": (1950, 1969)},
    {"occupation": "physicist", "education_country": "United Kingdom", "citizenship": "India", "birth": (1950, 1969)},
    {"occupation": "writer", "education_country": "United States", "citizenship": "Canada", "birth": (1950, 1969)},
    {"occupation": "film director", "education_country": "United Kingdom", "citizenship": "United Kingdom", "birth": (1970, 1989)},
    {"occupation": "actor", "education_country": "United Kingdom", "citizenship": "United Kingdom", "birth": (1970, 1989)},
]

L5_ACTOR_DIRECTOR_AWARD_COMBOS_V32 = [
    {"country": "United Kingdom", "language": "English", "genre": "drama", "award": "Academy Award", "birth": (1970, 1989), "sex": None},
    {"country": "Canada", "language": "English", "genre": "drama", "award": "Academy Award", "birth": (1970, 1989), "sex": None},
    {"country": "Australia", "language": "English", "genre": "drama", "award": "BAFTA Award", "birth": (1970, 1989), "sex": None},
    {"country": "France", "language": "French", "genre": "drama", "award": "César Award", "birth": (1950, 1969), "sex": None},
    {"country": "United States", "language": "English", "genre": "science fiction", "award": "Academy Award", "birth": (1970, 1989), "sex": None},
    {"country": "United Kingdom", "language": "English", "genre": "comedy", "award": "BAFTA Award", "birth": (1950, 1969), "sex": None},
]

L5_DIRECTOR_CAST_AWARD_COMBOS_V32 = [
    {"country": "United Kingdom", "language": "English", "genre": "drama", "award": "Academy Award", "birth": (1970, 1989), "sex": None},
    {"country": "United States", "language": "English", "genre": "science fiction", "award": "Academy Award", "birth": (1970, 1989), "sex": None},
    {"country": "France", "language": "French", "genre": "drama", "award": "César Award", "birth": (1950, 1969), "sex": None},
    {"country": "Italy", "language": "Italian", "genre": "drama", "award": "Academy Award", "birth": (1950, 1969), "sex": None},
    {"country": "Japan", "language": "Japanese", "genre": "thriller film", "award": "Academy Award", "birth": (1950, 1969), "sex": None},
    {"country": "Canada", "language": "English", "genre": "drama", "award": "BAFTA Award", "birth": (1970, 1989), "sex": None},
]

L5_WRITER_ADAPTATION_COMBOS_V32 = [
    {"country": "United States", "language": "English", "work_genre": "crime fiction", "film_genre": "drama", "birth": (1950, 1969)},
    {"country": "United Kingdom", "language": "English", "work_genre": "science fiction", "film_genre": "science fiction", "birth": (1950, 1969)},
    {"country": "France", "language": "French", "work_genre": "drama", "film_genre": "drama", "birth": (1950, 1969)},
    {"country": "Japan", "language": "Japanese", "work_genre": "science fiction", "film_genre": "science fiction", "birth": (1950, 1969)},
    {"country": "Canada", "language": "English", "work_genre": "crime fiction", "film_genre": "thriller film", "birth": (1950, 1969)},
]

L5_ADVISOR_AWARD_COMBOS_V32 = [
    {"occupation": "mathematician", "field": "number theory", "award": "Fields Medal", "advisor_birth": (1925, 1949)},
    {"occupation": "mathematician", "field": "topology", "award": "Fields Medal", "advisor_birth": (1925, 1949)},
    {"occupation": "physicist", "field": "theoretical physics", "award": "Nobel Prize", "advisor_birth": (1925, 1949)},
    {"occupation": "physicist", "field": "particle physics", "award": "Nobel Prize", "advisor_birth": (1930, 1959)},
    {"occupation": "chemist", "field": "biochemistry", "award": "Nobel Prize", "advisor_birth": (1930, 1959)},
    {"occupation": "economist", "field": "econometrics", "award": "Nobel Prize", "advisor_birth": (1930, 1959)},
]


def _tpl_directed_film_lang_genre_v32(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "film director"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    combo = dict(rng.choice(L3_DIRECTED_FILM_COMBOS_V32))
    country_key, lang_key, genre_key = combo["country"], combo["language"], combo["genre"]
    start, end = combo["birth"]
    sex_key = combo.get("sex")

    occ = OCCUPATIONS[occ_key]
    country = COUNTRIES[country_key]
    lang = LANGUAGES[lang_key]
    genre = FILM_GENRES[genre_key]
    where = [
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        f"?film wdt:P364 wd:{lang['qid']} .",
        "?film wdt:P57 ?person .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), f"снявших фильм жанра «{genre['ru']}» на языке оригинала: {lang['ru']}", _country_phrase_ru(country_key), _birth_phrase_ru(start, end)]
    en_parts = [en_head, _occ_phrase_en(occ_key), f"who directed {_p_film_genre_phrase_en(genre_key)} whose original language is {lang['en']}", _country_phrase_en(country_key), _birth_phrase_en(start, end)]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))

    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "directed_film_genre": genre["en"], "directed_film_genre_qid": genre["qid"],
        "directed_film_original_language": lang["en"], "directed_film_original_language_qid": lang["qid"],
        "directed_film_match_path": "inverse P57 + P136/P364",
        "citizenship_country": country["en"], "citizenship_country_qid": country["qid"],
        "birth_year_from": start, "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None, "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_directed_film_lang_genre_v32",
        template_family="directed_film_lang_genre_bridge", q_ru=_ru_join(ru_parts), q_en=_en_join(en_parts),
        constraints=constraints, where_lines=where, requested_count=requested, extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person <- directed film -> film genre and original language"},
    )


def _tpl_author_work_lang_genre_v32(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    combo = dict(rng.choice(L3_AUTHOR_WORK_COMBOS_V32))
    occ_key = combo["occupation"]
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    country_key, lang_key, genre_key = combo["country"], combo["language"], combo["genre"]
    start, end = combo["birth"]
    sex_key = combo.get("sex")

    occ = OCCUPATIONS[occ_key]
    country = COUNTRIES[country_key]
    lang = LANGUAGES[lang_key]
    genre = LITERARY_WORK_GENRES[genre_key]
    where = [
        f"?work wdt:P136 wd:{genre['qid']} .",
        f"?work wdt:P407 wd:{lang['qid']} .",
        "?work wdt:P50 ?person .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), f"у которых есть написанное ими произведение жанра «{genre['ru']}» на языке: {lang['ru']}", _country_phrase_ru(country_key), _birth_phrase_ru(start, end)]
    en_parts = [en_head, _occ_phrase_en(occ_key), f"who authored a work in the genre {genre['en']} whose language is {lang['en']}", _country_phrase_en(country_key), _birth_phrase_en(start, end)]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key))
        en_parts.append(_sex_phrase_en(sex_key))

    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "authored_work_genre": genre["en"], "authored_work_genre_qid": genre["qid"],
        "authored_work_language": lang["en"], "authored_work_language_qid": lang["qid"],
        "authored_work_match_path": "inverse P50 + P136/P407",
        "citizenship_country": country["en"], "citizenship_country_qid": country["qid"],
        "birth_year_from": start, "birth_year_to": end,
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None, "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_author_work_lang_genre_v32",
        template_family="authored_work_lang_genre_bridge", q_ru=_ru_join(ru_parts), q_en=_en_join(en_parts),
        constraints=constraints, where_lines=where, requested_count=requested, extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person <- authored work -> work genre and work language"},
    )


def _tpl_employer_country_bridge_v32(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    combo = dict(rng.choice(L3_EMPLOYER_COMBOS_V32))
    occ_key = combo["occupation"]
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    employer_country_key, citizenship_key = combo["employer_country"], combo["citizenship"]
    start, end = combo["birth"]

    occ = OCCUPATIONS[occ_key]
    employer_country = COUNTRIES[employer_country_key]
    citizenship_country = COUNTRIES[citizenship_key]
    where = [
        f"?employer wdt:P17 wd:{employer_country['qid']} .",
        "?employer wdt:P31/wdt:P279* wd:Q43229 .",  # organization
        "?person wdt:P108 ?employer .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), f"работавших в организации из страны: {employer_country['ru']}", _country_phrase_ru(citizenship_key), _birth_phrase_ru(start, end)])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), f"who worked for an organization from {employer_country['en']}", _country_phrase_en(citizenship_key), _birth_phrase_en(start, end)])
    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "employer_country": employer_country["en"], "employer_country_qid": employer_country["qid"],
        "employer_country_path": "P108/P17", "employer_instance_class": "organization", "employer_instance_class_qid": "Q43229",
        "citizenship_country": citizenship_country["en"], "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start, "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_employer_country_v32",
        template_family="employer_country_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested, extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> employer -> employer country; employer is an organization"},
    )


def _tpl_education_country_v32(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L3":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    combo = dict(rng.choice(L3_EDUCATION_COMBOS_V32))
    occ_key = combo["occupation"]
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    edu_country_key, citizenship_key = combo["education_country"], combo["citizenship"]
    start, end = combo["birth"]

    occ = OCCUPATIONS[occ_key]
    edu_country = COUNTRIES[edu_country_key]
    citizenship_country = COUNTRIES[citizenship_key]
    where = [
        f"?institution wdt:P17 wd:{edu_country['qid']} .",
        "?person wdt:P69 ?institution .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{citizenship_country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), f"получивших образование в стране: {edu_country['ru']}", _country_phrase_ru(citizenship_key), _birth_phrase_ru(start, end)])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), f"educated in {edu_country['en']}", _country_phrase_en(citizenship_key), _birth_phrase_en(start, end)])
    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "education_country": edu_country["en"], "education_country_qid": edu_country["qid"], "education_country_path": "P69/P17",
        "citizenship_country": citizenship_country["en"], "citizenship_country_qid": citizenship_country["qid"],
        "birth_year_from": start, "birth_year_to": end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_education_country_v32",
        template_family="education_country_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested, extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> educated at -> institution country"},
    )


### 11b. v33 — film direction, writer, advisor templates

In [ ]:
def _tpl_actor_film_director_award_v32(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "actor"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    combo = dict(rng.choice(L5_ACTOR_DIRECTOR_AWARD_COMBOS_V32))
    country_key, lang_key, genre_key, award_key = combo["country"], combo["language"], combo["genre"], combo["award"]
    start, end = combo["birth"]
    sex_key = combo.get("sex")

    occ = OCCUPATIONS[occ_key]
    country = COUNTRIES[country_key]
    lang = LANGUAGES[lang_key]
    genre = FILM_GENRES[genre_key]
    award = AWARDS[award_key]
    where = [
        *_p_award_where_lines_for_var(award_key, var="director", value_var="?directorAwardReceived"),
        "?director wdt:P31 wd:Q5 .",
        "?film wdt:P57 ?director .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        f"?film wdt:P364 wd:{lang['qid']} .",
        "?film wdt:P161 ?person .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines_scoped(start, end, var="person", scope="person"),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end), f"снимавшихся в фильме жанра «{genre['ru']}» на языке оригинала: {lang['ru']}", f"режиссёр которого получил награду: {award['ru']}"]
    en_parts = [en_head, _occ_phrase_en(occ_key), _country_phrase_en(country_key), _birth_phrase_en(start, end), f"who acted in {_p_film_genre_phrase_en(genre_key)} whose original language is {lang['en']}", f"whose director received the {award['en']}"]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key)); en_parts.append(_sex_phrase_en(sex_key))
    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "citizenship_country": country["en"], "citizenship_country_qid": country["qid"], "birth_year_from": start, "birth_year_to": end,
        "acted_in_film_genre": genre["en"], "acted_in_film_genre_qid": genre["qid"],
        "acted_in_film_original_language": lang["en"], "acted_in_film_original_language_qid": lang["qid"],
        "film_director_award_received": award["en"], "film_director_award_received_qid": award["qid"],
        "actor_film_director_award_path": "inverse P161/P57/P166 + film P136/P364",
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None, "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_actor_film_director_award_v32",
        template_family="actor_film_director_award_bridge", q_ru=_ru_join(ru_parts), q_en=_en_join(en_parts),
        constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta={"bridge": "actor <- film -> director -> director award; film -> genre/original language"},
    )


def _tpl_director_film_cast_award_v32(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "film director"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    combo = dict(rng.choice(L5_DIRECTOR_CAST_AWARD_COMBOS_V32))
    country_key, lang_key, genre_key, award_key = combo["country"], combo["language"], combo["genre"], combo["award"]
    start, end = combo["birth"]
    sex_key = combo.get("sex")

    occ = OCCUPATIONS[occ_key]
    country = COUNTRIES[country_key]
    lang = LANGUAGES[lang_key]
    genre = FILM_GENRES[genre_key]
    award = AWARDS[award_key]
    where = [
        *_p_award_where_lines_for_var(award_key, var="castMember", value_var="?castAwardReceived"),
        "?castMember wdt:P31 wd:Q5 .",
        "?film wdt:P161 ?castMember .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{genre['qid']} .",
        f"?film wdt:P364 wd:{lang['qid']} .",
        "?film wdt:P57 ?person .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    if sex_key:
        where.append(f"?person wdt:P21 wd:{SEXES[sex_key]['qid']} .")

    ru_head, en_head = _p_head(requested)
    ru_parts = [ru_head, _occ_phrase_ru(occ_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end), f"снявших фильм жанра «{genre['ru']}» на языке оригинала: {lang['ru']}", f"в актёрском составе которого есть человек, получивший награду: {award['ru']}"]
    en_parts = [en_head, _occ_phrase_en(occ_key), _country_phrase_en(country_key), _birth_phrase_en(start, end), f"who directed {_p_film_genre_phrase_en(genre_key)} whose original language is {lang['en']}", f"whose cast includes a person who received the {award['en']}"]
    if sex_key:
        ru_parts.append(_sex_phrase_ru(sex_key)); en_parts.append(_sex_phrase_en(sex_key))
    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "citizenship_country": country["en"], "citizenship_country_qid": country["qid"], "birth_year_from": start, "birth_year_to": end,
        "directed_film_genre": genre["en"], "directed_film_genre_qid": genre["qid"],
        "directed_film_original_language": lang["en"], "directed_film_original_language_qid": lang["qid"],
        "cast_member_award_received": award["en"], "cast_member_award_received_qid": award["qid"],
        "cast_member_award_bridge_path": "inverse P57/P161/P166 + film P136/P364",
        "sex_or_gender": SEXES[sex_key]["en"] if sex_key else None, "sex_or_gender_qid": SEXES[sex_key]["qid"] if sex_key else None,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_director_film_cast_award_v32",
        template_family="director_film_cast_award_bridge", q_ru=_ru_join(ru_parts), q_en=_en_join(en_parts),
        constraints=constraints, where_lines=where, requested_count=requested, extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "director <- film -> cast member -> cast award; film -> genre/original language"},
    )


def _tpl_writer_work_film_adaptation_v32(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = "writer"
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    combo = dict(rng.choice(L5_WRITER_ADAPTATION_COMBOS_V32))
    country_key, lang_key = combo["country"], combo["language"]
    work_genre_key, film_genre_key = combo["work_genre"], combo["film_genre"]
    start, end = combo["birth"]

    occ = OCCUPATIONS[occ_key]
    country = COUNTRIES[country_key]
    lang = LANGUAGES[lang_key]
    work_genre = LITERARY_WORK_GENRES[work_genre_key]
    film_genre = FILM_GENRES[film_genre_key]
    where = [
        f"?work wdt:P136 wd:{work_genre['qid']} .",
        f"?work wdt:P407 wd:{lang['qid']} .",
        "?work wdt:P50 ?person .",
        "?film wdt:P144 ?work .",
        "?film wdt:P31/wdt:P279* wd:Q11424 .",
        f"?film wdt:P136 wd:{film_genre['qid']} .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end), f"у которых есть написанное ими произведение жанра «{work_genre['ru']}» на языке: {lang['ru']}", f"и по этому произведению снят фильм жанра «{film_genre['ru']}»"])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), _country_phrase_en(country_key), _birth_phrase_en(start, end), f"who authored a work in the genre {work_genre['en']} whose language is {lang['en']}", f"and {_p_film_genre_phrase_en(film_genre_key)} is based on that work"])
    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "citizenship_country": country["en"], "citizenship_country_qid": country["qid"], "birth_year_from": start, "birth_year_to": end,
        "authored_work_genre": work_genre["en"], "authored_work_genre_qid": work_genre["qid"],
        "authored_work_language": lang["en"], "authored_work_language_qid": lang["qid"],
        "adaptation_film_genre": film_genre["en"], "adaptation_film_genre_qid": film_genre["qid"],
        "adaptation_bridge_path": "inverse P50 + P136/P407; film P144/P136",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_writer_work_film_adaptation_v32",
        template_family="writer_work_film_adaptation_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested, extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "writer <- authored work -> film adaptation -> film genre"},
    )


def _tpl_advisor_award_v32(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    combo = dict(rng.choice(L5_ADVISOR_AWARD_COMBOS_V32))
    occ_key, field_key, award_key = combo["occupation"], combo["field"], combo["award"]
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    advisor_start, advisor_end = combo["advisor_birth"]

    occ = OCCUPATIONS[occ_key]
    field = FIELDS[field_key]
    award = AWARDS[award_key]
    where = [
        *_p_award_where_lines_for_var(award_key, var="advisor", value_var="?advisorAwardReceived"),
        "?advisor wdt:P31 wd:Q5 .",
        *_p_birth_range_lines_scoped(advisor_start, advisor_end, var="advisor", scope="advisor"),
        "?person wdt:P184 ?advisor .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P101 wd:{field['qid']} .",
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), _field_phrase_ru(field_key), f"у которых научный руководитель получил награду: {award['ru']}", _advisor_birth_phrase_ru(advisor_start, advisor_end)])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), _field_phrase_en(field_key), f"whose doctoral advisor received the {award['en']}", _advisor_birth_phrase_en(advisor_start, advisor_end)])
    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "field_of_work": field["en"], "field_of_work_qid": field["qid"],
        "doctoral_advisor_award_received": award["en"], "doctoral_advisor_award_received_qid": award["qid"],
        "doctoral_advisor_award_path": "P184/P166", "doctoral_advisor_birth_year_from": advisor_start, "doctoral_advisor_birth_year_to": advisor_end,
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_advisor_award_v32",
        template_family="advisor_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta={"bridge": "person -> doctoral advisor -> advisor award"},
    )


# Registry: keep L3 cleaner. Football and political-party templates are disabled
# because the observed v31 rows exposed noisy P413 and poor RU label coverage.
PEOPLE_TEMPLATES_BY_LEVEL = {
    "L1": [],
    "L2": [],
    "L3": [
        (_tpl_directed_film_lang_genre_v32, 0.30),
        (_tpl_author_work_lang_genre_v32, 0.25),
        (_tpl_employer_country_bridge_v32, 0.25),
        (_tpl_education_country_v32, 0.20),
    ],
    "L4": [],
    "L5": [
        (_tpl_actor_film_director_award_v32, 0.28),
        (_tpl_director_film_cast_award_v32, 0.25),
        (_tpl_advisor_award_v32, 0.20),
        (_tpl_writer_work_film_adaptation_v32, 0.15),
        (_tpl_spouse_award_fast, 0.06),
        (_tpl_parent_award_fast, 0.06),
    ],
}

PEOPLE_TEMPLATE_FN_TO_FAMILY.update({
    "_tpl_directed_film_lang_genre_v32": "directed_film_lang_genre_bridge",
    "_tpl_author_work_lang_genre_v32": "authored_work_lang_genre_bridge",
    "_tpl_employer_country_bridge_v32": "employer_country_bridge",
    "_tpl_education_country_v32": "education_country_bridge",
    "_tpl_actor_film_director_award_v32": "actor_film_director_award_bridge",
    "_tpl_director_film_cast_award_v32": "director_film_cast_award_bridge",
    "_tpl_writer_work_film_adaptation_v32": "writer_work_film_adaptation_bridge",
    "_tpl_advisor_award_v32": "advisor_award_bridge",
})

PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL = {
    "L3": {
        "directed_film_lang_genre_bridge": 0.36,
        "authored_work_lang_genre_bridge": 0.32,
        "employer_country_bridge": 0.32,
        "education_country_bridge": 0.25,
        "sports_team_country_bridge": 0.00,
        "political_party_country_bridge": 0.00,
    },
    "L5": {
        "actor_film_director_award_bridge": 0.32,
        "director_film_cast_award_bridge": 0.30,
        "advisor_award_bridge": 0.26,
        "writer_work_film_adaptation_bridge": 0.22,
        "spouse_award_bridge": 0.10,
        "parent_award_bridge": 0.10,
        "spouse_occupation_bridge": 0.00,
        "parent_occupation_bridge": 0.00,
    },
}
PEOPLE_L3_TEMPLATE_FAMILY_CAPS = PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL["L3"]
PEOPLE_MAX_CONSECUTIVE_SAME_FAMILY_BY_LEVEL = {"L3": 2, "L5": 2}

# v32 label-quality policy. This is deliberately less strict than "all RU labels"
# because many valid Wikidata people lack Russian labels, but it rejects rows like
# the observed Argentina-politician example where every RU label was just EN fallback.
PEOPLE_MIN_RU_LABEL_SHARE = 0.25
PEOPLE_REJECT_ZERO_RU_LABEL_RECORDS = True

if "_p_passes_label_quality_base_v32" not in globals():
    _p_passes_label_quality_base_v32 = _p_passes_label_quality


def _p_passes_label_quality(gold_items: List[Dict[str, Any]], meta: Dict[str, Any]) -> bool:
    if not _p_passes_label_quality_base_v32(gold_items, meta):
        return False
    n = len(gold_items or [])
    label_sources = (meta or {}).get("label_sources") or {}
    ru_count = int(label_sources.get("ru_label", 0) or 0)
    if PEOPLE_REJECT_ZERO_RU_LABEL_RECORDS and n and ru_count == 0:
        return False
    if n:
        requested = 5
        min_by_share = int(math.ceil(n * float(PEOPLE_MIN_RU_LABEL_SHARE)))
        min_ru = min(max(2, min_by_share), requested)
        if ru_count < min_ru:
            return False
    return True

# Explicit quality guard for resume cleanup and newly generated rows.
if "_people_record_quality_problems_base_v33" not in globals():
    _people_record_quality_problems_base_v33 = people_record_quality_problems


def people_record_quality_problems(record: Dict[str, Any]) -> List[str]:
    problems = list(_people_record_quality_problems_base_v33(record))
    query_ru = str(record.get("query_text_ru") or "")
    query_en = str(record.get("query_text_en") or "")
    query_text = f"{query_ru} {query_en}"
    constraints = record.get("constraints") or {}
    sparql_text = f"{record.get('sparql_query') or ''} {record.get('ask_validator_sparql') or ''}"
    meta = record.get("gold_collection_meta") or {}
    label_sources = meta.get("label_sources") or {}
    ru_count = int(label_sources.get("ru_label", 0) or 0)
    total_gold = len(record.get("gold_answer_qids") or [])

    if "film film" in query_en.casefold():
        problems.append("duplicate_film_wording")
    if (
        constraints.get("directed_film_country_of_origin")
        or "country of origin" in query_text.casefold()
        or "страной происхождения" in query_text.casefold()
        or "wdt:P495" in sparql_text
    ):
        problems.append("film_country_of_origin_criterion_disabled")
    if record.get("template_family") == "sports_team_country_bridge" or constraints.get("playing_position"):
        problems.append("sports_position_bridge_disabled_due_to_noisy_p413")
    if record.get("template_family") == "political_party_country_bridge":
        problems.append("political_party_bridge_disabled_due_to_low_ru_label_yield")
    if "организации, связанной со страной" in query_ru or "associated with" in query_en:
        problems.append("awkward_employer_country_wording")
    if record.get("template_family") == "employer_country_bridge" and "wd:Q43229" not in sparql_text:
        problems.append("employer_country_without_organization_class")
    if total_gold and ru_count == 0:
        problems.append("no_ru_gold_labels")
    if total_gold:
        min_ru = min(max(2, int(math.ceil(total_gold * float(PEOPLE_MIN_RU_LABEL_SHARE)))), int(record.get("requested_count") or 5))
        if ru_count < min_ru:
            problems.append("too_many_en_fallback_ru_labels")
    return problems

print("✅ v33 people hotfix loaded:")
print("   output:", PEOPLE_OUTPUT_PATH.resolve())
print("   targets:", PEOPLE_TARGET_PER_LEVEL)
print("   WDQS timeout/retries:", getattr(wd, "timeout", None), getattr(wd, "max_retries", None))
print("   L3 families:", [PEOPLE_TEMPLATE_FN_TO_FAMILY.get(fn.__name__) for fn, _ in PEOPLE_TEMPLATES_BY_LEVEL["L3"]])
print("   L5 families:", [PEOPLE_TEMPLATE_FN_TO_FAMILY.get(fn.__name__) for fn, _ in PEOPLE_TEMPLATES_BY_LEVEL["L5"]])


## 12. Generation engine — v34 (queue-based)

Replaces the random-sampling generator with a deterministic candidate queue: each template/constraint pair is tried at most once, and failed WDQS results are memoized.


In [ ]:
PEOPLE_OUTPUT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v34_queue.jsonl"
PEOPLE_AUDIT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v34_queue_audit.json"
PEOPLE_CHECKPOINT_PATH = PEOPLE_OUTPUT_DIR / "people_l3_l5_patch_v34_queue_checkpoint.json"

PEOPLE_TARGET_PER_LEVEL = {"L1": 0, "L2": 0, "L3": 35, "L4": 0, "L5": 35}
PEOPLE_RANDOM_SEED = 8144
PEOPLE_MAX_ATTEMPTS_PER_LEVEL = 450
PEOPLE_MAX_ATTEMPTS_PER_EXAMPLE = 4
PEOPLE_WDQS_LIMIT_DEFAULT = 76
PEOPLE_MIN_GOLD_BY_LEVEL.update({"L3": 6, "L5": 5})
PEOPLE_MAX_GOLD_BY_LEVEL.update({"L3": 70, "L5": 70})
PEOPLE_MAX_SAME_OCCUPATION_PER_LEVEL.update({"L3": 12, "L5": 14})

# Keep rows with some real Russian coverage, but do not throw away otherwise good
# Wikidata golds just because a few less-known people lack ru labels.
PEOPLE_MIN_RU_LABEL_SHARE = 0.15
PEOPLE_REJECT_ZERO_RU_LABEL_RECORDS = True

try:
    wd.timeout = 12
    wd.max_retries = 1
except Exception:
    pass


def _people_find_reference_jsonl_paths_v34() -> List[Path]:
    """Find final/previous people JSONLs for dedup and id continuation."""
    names = [
        "people_final_curated_v2.jsonl",
        "people_final_curated.jsonl",
        "people.jsonl",
        "people_l3_l5_patch_v33_fast_quality_keep_famous.jsonl",
        "people_l3_l5_patch_v31_fast_clean.jsonl",
    ]
    roots: List[Path] = []
    try:
        cwd = Path.cwd()
        roots.extend([cwd, *list(cwd.parents)[:5]])
    except Exception:
        roots.append(Path("."))
    roots.extend([PEOPLE_OUTPUT_DIR, PEOPLE_OUTPUT_DIR.parent, Path("data"), Path("08_people")])

    out: List[Path] = []
    for root in roots:
        try:
            root = Path(root)
            for name in names:
                for cand in [root / name, root / "domain_outputs" / name, root / "out_wikidata_benchmark" / "domain_outputs" / name]:
                    if cand.exists() and cand not in out and cand != PEOPLE_OUTPUT_PATH:
                        out.append(cand)
        except Exception:
            continue

    # A shallow recursive fallback inside the project tree only. This avoids a
    # full home-directory crawl but catches common notebook subfolder layouts.
    try:
        for root in [Path.cwd(), PEOPLE_OUTPUT_DIR.parent]:
            if root.exists():
                for cand in list(root.glob("**/people_final_curated_v2.jsonl"))[:25]:
                    if cand.exists() and cand not in out and cand != PEOPLE_OUTPUT_PATH:
                        out.append(cand)
    except Exception:
        pass
    return out


PEOPLE_REFERENCE_JSONL_PATHS = _people_find_reference_jsonl_paths_v34()

# ------------------------------------------------------------------
# WDQS result memoization. The queue already avoids repeats, but this protects
# resume/restarts and duplicated candidate specs from hitting WDQS again.
# ------------------------------------------------------------------
if "_p_finalize_wdqs_example_base_v34" not in globals():
    _p_finalize_wdqs_example_base_v34 = _p_finalize_wdqs_example

_PEOPLE_FINALIZE_CACHE_V34: Dict[str, Optional[Dict[str, Any]]] = globals().get("_PEOPLE_FINALIZE_CACHE_V34", {})
_PEOPLE_CANDIDATE_LOG_V34: List[Dict[str, Any]] = globals().get("_PEOPLE_CANDIDATE_LOG_V34", [])


def _people_finalize_cache_key_v34(template_id: str, constraints: Dict[str, Any], where_lines: List[str]) -> str:
    return json.dumps({
        "template_id": template_id,
        "constraints": _p_public_constraints(_p_clean_constraints(constraints)),
        "where_lines": where_lines,
    }, ensure_ascii=False, sort_keys=True)


def _people_clone_example_with_id_v34(record: Dict[str, Any], complexity: str, idx: int) -> BenchmarkExample:
    d = json.loads(json.dumps(record, ensure_ascii=False))
    d["id"] = f"people_{complexity.lower()}_{idx:04d}"
    d["complexity"] = complexity
    d["created_at"] = utc_now_z()
    return people_record_to_example(d)


def _p_finalize_wdqs_example(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    q_ru: str,
    q_en: str,
    constraints: Dict[str, Any],
    where_lines: List[str],
    requested_count: int,
    extra_select_vars: Optional[List[str]] = None,
    gold_limit: int = PEOPLE_GOLD_LIMIT,
    bridge_meta: Optional[Dict[str, Any]] = None,
) -> Optional[BenchmarkExample]:
    key = _people_finalize_cache_key_v34(template_id, constraints, where_lines)
    if key in _PEOPLE_FINALIZE_CACHE_V34:
        cached = _PEOPLE_FINALIZE_CACHE_V34[key]
        if cached is None:
            return None
        return _people_clone_example_with_id_v34(cached, complexity, idx)

    ex = _p_finalize_wdqs_example_base_v34(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family=template_family,
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where_lines,
        requested_count=requested_count,
        extra_select_vars=extra_select_vars,
        gold_limit=gold_limit,
        bridge_meta=bridge_meta,
    )
    _PEOPLE_FINALIZE_CACHE_V34[key] = None if ex is None else people_example_to_dict(ex)
    return ex


# ------------------------------------------------------------------
# Candidate pools. They intentionally over-generate distinct high-yield
# combinations; the quality filters still decide what enters the JSONL.
# ------------------------------------------------------------------
_COUNTRY_LANG_V34 = {
    "United States": "English",
    "United Kingdom": "English",
    "Canada": "English",
    "Australia": "English",
    "France": "French",
    "Germany": "German",
    "Italy": "Italian",
    "Spain": "Spanish",
    "Japan": "Japanese",
    "Russia": "Russian",
    "Poland": "Polish",
    "Netherlands": "Dutch",
    "Brazil": "Portuguese",
}


def _v34_birth_ranges(*pairs: Tuple[int, int]) -> List[Tuple[int, int]]:
    return list(pairs)


L3_DIRECTED_FILM_COMBOS_V34: List[Dict[str, Any]] = []
for _country, _genres, _births, _sexes in [
    ("United States", ["drama", "comedy", "science fiction", "action film", "thriller film"], _v34_birth_ranges((1950, 1969), (1970, 1989)), ["female", "male"]),
    ("United Kingdom", ["drama", "comedy", "science fiction", "thriller film"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None, "female"]),
    ("France", ["drama", "comedy", "thriller film"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("Germany", ["drama", "comedy"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("Italy", ["drama", "comedy"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("Spain", ["drama", "comedy", "thriller film"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("Japan", ["drama", "science fiction", "thriller film"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None, "male"]),
    ("Canada", ["drama", "comedy", "science fiction"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None, "female"]),
    ("Australia", ["drama", "comedy", "action film"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
]:
    for _genre in _genres:
        for _birth in _births:
            for _sex in _sexes:
                L3_DIRECTED_FILM_COMBOS_V34.append({"country": _country, "language": _COUNTRY_LANG_V34[_country], "genre": _genre, "birth": _birth, "sex": _sex})


L3_AUTHOR_WORK_COMBOS_V34: List[Dict[str, Any]] = []
for _occupation, _countries, _genres, _births, _sexes in [
    ("writer", ["United States", "United Kingdom", "Canada", "Australia"], ["science fiction", "crime fiction", "drama", "comedy"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None, "female"]),
    ("writer", ["France", "Germany", "Italy", "Spain", "Japan", "Russia", "Poland"], ["science fiction", "crime fiction", "drama"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("poet", ["United States", "United Kingdom", "France", "Russia", "Poland"], ["drama", "comedy"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
]:
    for _country in _countries:
        if _country not in _COUNTRY_LANG_V34:
            continue
        for _genre in _genres:
            for _birth in _births:
                for _sex in _sexes:
                    L3_AUTHOR_WORK_COMBOS_V34.append({"occupation": _occupation, "country": _country, "language": _COUNTRY_LANG_V34[_country], "genre": _genre, "birth": _birth, "sex": _sex})


L3_EMPLOYER_COMBOS_V34 = [
    {"occupation": "economist", "employer_country": "United States", "citizenship": "France", "birth": (1950, 1969)},
    {"occupation": "economist", "employer_country": "United States", "citizenship": "United Kingdom", "birth": (1950, 1969)},
    {"occupation": "economist", "employer_country": "United States", "citizenship": "Canada", "birth": (1970, 1989)},
    {"occupation": "mathematician", "employer_country": "United States", "citizenship": "France", "birth": (1950, 1969)},
    {"occupation": "mathematician", "employer_country": "United States", "citizenship": "Germany", "birth": (1950, 1969)},
    {"occupation": "physicist", "employer_country": "United States", "citizenship": "Germany", "birth": (1950, 1969)},
    {"occupation": "physicist", "employer_country": "United Kingdom", "citizenship": "India", "birth": (1950, 1969)},
    {"occupation": "chemist", "employer_country": "United States", "citizenship": "United Kingdom", "birth": (1950, 1969)},
    {"occupation": "biologist", "employer_country": "United States", "citizenship": "United Kingdom", "birth": (1950, 1969)},
    {"occupation": "historian", "employer_country": "United Kingdom", "citizenship": "United States", "birth": (1950, 1969)},
    {"occupation": "journalist", "employer_country": "United States", "citizenship": "Canada", "birth": (1970, 1989)},
    {"occupation": "architect", "employer_country": "United States", "citizenship": "Italy", "birth": (1950, 1969)},
]


L3_EDUCATION_COMBOS_V34 = [
    {"occupation": "economist", "education_country": "United States", "citizenship": "France", "birth": (1950, 1969)},
    {"occupation": "economist", "education_country": "United States", "citizenship": "United Kingdom", "birth": (1950, 1969)},
    {"occupation": "economist", "education_country": "United States", "citizenship": "Canada", "birth": (1970, 1989)},
    {"occupation": "mathematician", "education_country": "United States", "citizenship": "France", "birth": (1950, 1969)},
    {"occupation": "mathematician", "education_country": "United States", "citizenship": "Germany", "birth": (1950, 1969)},
    {"occupation": "physicist", "education_country": "United Kingdom", "citizenship": "India", "birth": (1950, 1969)},
    {"occupation": "writer", "education_country": "United States", "citizenship": "Canada", "birth": (1950, 1969)},
    {"occupation": "writer", "education_country": "United Kingdom", "citizenship": "France", "birth": (1950, 1969)},
    {"occupation": "film director", "education_country": "United Kingdom", "citizenship": "United Kingdom", "birth": (1970, 1989)},
    {"occupation": "actor", "education_country": "United Kingdom", "citizenship": "United Kingdom", "birth": (1970, 1989)},
    {"occupation": "politician", "education_country": "United States", "citizenship": "India", "birth": (1950, 1969)},
    {"occupation": "journalist", "education_country": "United States", "citizenship": "Canada", "birth": (1970, 1989)},
]


L5_ACTOR_DIRECTOR_AWARD_COMBOS_V34: List[Dict[str, Any]] = []
for _country, _genres, _awards, _births, _sexes in [
    ("United States", ["drama", "science fiction", "action film", "thriller film", "comedy"], ["Academy Award", "BAFTA Award"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None, "female"]),
    ("United Kingdom", ["drama", "comedy", "science fiction", "thriller film"], ["Academy Award", "BAFTA Award"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None, "female"]),
    ("Canada", ["drama", "comedy", "science fiction"], ["Academy Award", "BAFTA Award"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("Australia", ["drama", "comedy", "action film"], ["Academy Award", "BAFTA Award"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("France", ["drama", "comedy", "thriller film"], ["César Award", "Academy Award"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("Italy", ["drama", "comedy"], ["Academy Award"], _v34_birth_ranges((1950, 1969),), [None]),
    ("Japan", ["drama", "science fiction", "thriller film"], ["Academy Award"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
]:
    for _genre in _genres:
        for _award in _awards:
            for _birth in _births:
                for _sex in _sexes:
                    L5_ACTOR_DIRECTOR_AWARD_COMBOS_V34.append({"country": _country, "language": _COUNTRY_LANG_V34[_country], "genre": _genre, "award": _award, "birth": _birth, "sex": _sex})


L5_DIRECTOR_CAST_AWARD_COMBOS_V34: List[Dict[str, Any]] = []
for _country, _genres, _awards, _births, _sexes in [
    ("United States", ["drama", "science fiction", "action film", "thriller film", "comedy"], ["Academy Award", "BAFTA Award"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None, "female"]),
    ("United Kingdom", ["drama", "comedy", "science fiction", "thriller film"], ["Academy Award", "BAFTA Award"], _v34_birth_ranges((1950, 1969), (1970, 1989)), [None]),
    ("Canada", ["drama", "comedy"], ["Academy Award", "BAFTA Award"], _v34_birth_ranges((1970, 1989),), [None]),
    ("France", ["drama", "comedy", "thriller film"], ["César Award", "Academy Award"], _v34_birth_ranges((1950, 1969),), [None]),
    ("Italy", ["drama", "comedy"], ["Academy Award"], _v34_birth_ranges((1950, 1969),), [None]),
    ("Japan", ["drama", "science fiction", "thriller film"], ["Academy Award"], _v34_birth_ranges((1950, 1969),), [None]),
]:
    for _genre in _genres:
        for _award in _awards:
            for _birth in _births:
                for _sex in _sexes:
                    L5_DIRECTOR_CAST_AWARD_COMBOS_V34.append({"country": _country, "language": _COUNTRY_LANG_V34[_country], "genre": _genre, "award": _award, "birth": _birth, "sex": _sex})


L5_WRITER_ADAPTATION_COMBOS_V34: List[Dict[str, Any]] = []
for _country, _work_genres, _film_genres, _births in [
    ("United States", ["science fiction", "crime fiction", "drama"], ["drama", "science fiction", "thriller film"], _v34_birth_ranges((1950, 1969), (1970, 1989))),
    ("United Kingdom", ["science fiction", "crime fiction", "drama"], ["drama", "science fiction", "thriller film"], _v34_birth_ranges((1950, 1969), (1970, 1989))),
    ("Canada", ["science fiction", "crime fiction"], ["drama", "thriller film", "science fiction"], _v34_birth_ranges((1950, 1969), (1970, 1989))),
    ("France", ["drama", "crime fiction"], ["drama", "thriller film"], _v34_birth_ranges((1950, 1969),)),
    ("Japan", ["science fiction", "crime fiction"], ["science fiction", "drama"], _v34_birth_ranges((1950, 1969),)),
    ("Germany", ["crime fiction", "drama"], ["drama", "thriller film"], _v34_birth_ranges((1950, 1969),)),
]:
    for _wg in _work_genres:
        for _fg in _film_genres:
            for _birth in _births:
                L5_WRITER_ADAPTATION_COMBOS_V34.append({"country": _country, "language": _COUNTRY_LANG_V34[_country], "work_genre": _wg, "film_genre": _fg, "birth": _birth})


L5_ADVISOR_AWARD_COMBOS_V34 = [
    {"occupation": "mathematician", "field": "number theory", "award": "Fields Medal", "advisor_birth": (1925, 1949)},
    {"occupation": "mathematician", "field": "topology", "award": "Fields Medal", "advisor_birth": (1925, 1949)},
    {"occupation": "mathematician", "field": "algebra", "award": "Fields Medal", "advisor_birth": (1930, 1959)},
    {"occupation": "mathematician", "field": "geometry", "award": "Wolf Prize", "advisor_birth": (1925, 1949)},
    {"occupation": "physicist", "field": "theoretical physics", "award": "Nobel Prize", "advisor_birth": (1925, 1949)},
    {"occupation": "physicist", "field": "particle physics", "award": "Nobel Prize", "advisor_birth": (1930, 1959)},
    {"occupation": "physicist", "field": "quantum mechanics", "award": "Wolf Prize", "advisor_birth": (1925, 1949)},
    {"occupation": "chemist", "field": "biochemistry", "award": "Nobel Prize", "advisor_birth": (1930, 1959)},
    {"occupation": "biologist", "field": "genetics", "award": "Nobel Prize", "advisor_birth": (1930, 1959)},
    {"occupation": "economist", "field": "econometrics", "award": "Nobel Prize", "advisor_birth": (1930, 1959)},
]


L5_SPOUSE_AWARD_COMBOS_V34 = [
    {"occupation": "actor", "country": "United States", "birth": (1950, 1969), "spouse_occupation": "actor", "award": "Academy Award"},
    {"occupation": "actor", "country": "United Kingdom", "birth": (1950, 1969), "spouse_occupation": "actor", "award": "BAFTA Award"},
    {"occupation": "film director", "country": "United States", "birth": (1950, 1969), "spouse_occupation": "actor", "award": "Academy Award"},
    {"occupation": "singer", "country": "United States", "birth": (1950, 1969), "spouse_occupation": "singer", "award": "Grammy Award"},
    {"occupation": "writer", "country": "United States", "birth": (1950, 1969), "spouse_occupation": "writer", "award": "Pulitzer Prize"},
    {"occupation": "politician", "country": "United States", "birth": (1950, 1969), "spouse_occupation": "actor", "award": "Academy Award"},
    {"occupation": "journalist", "country": "United Kingdom", "birth": (1950, 1969), "spouse_occupation": "writer", "award": "Booker Prize"},
    {"occupation": "actor", "country": "France", "birth": (1950, 1969), "spouse_occupation": "actor", "award": "César Award"},
    {"occupation": "writer", "country": "United Kingdom", "birth": (1950, 1969), "spouse_occupation": "writer", "award": "Booker Prize"},
]


L5_PARENT_AWARD_COMBOS_V34 = [
    {"occupation": "actor", "country": "United States", "birth": (1970, 1989), "parent_occupation": "actor", "award": "Academy Award"},
    {"occupation": "actor", "country": "United Kingdom", "birth": (1970, 1989), "parent_occupation": "actor", "award": "BAFTA Award"},
    {"occupation": "film director", "country": "United States", "birth": (1970, 1989), "parent_occupation": "actor", "award": "Academy Award"},
    {"occupation": "singer", "country": "United States", "birth": (1970, 1989), "parent_occupation": "singer", "award": "Grammy Award"},
    {"occupation": "writer", "country": "United States", "birth": (1970, 1989), "parent_occupation": "writer", "award": "Pulitzer Prize"},
    {"occupation": "journalist", "country": "United States", "birth": (1970, 1989), "parent_occupation": "writer", "award": "Pulitzer Prize"},
    {"occupation": "actor", "country": "Canada", "birth": (1970, 1989), "parent_occupation": "actor", "award": "Academy Award"},
    {"occupation": "actor", "country": "France", "birth": (1970, 1989), "parent_occupation": "actor", "award": "César Award"},
]


# Swap one-combo lists into v32 templates so their existing carefully written
# wording/constraints/SPARQL stay the source of truth.
def _people_v34_call_combo_template(spec: Dict[str, Any], complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    fn = spec["fn"]
    combo_global = spec["combo_global"]
    combo = spec["combo"]
    old_value = globals().get(combo_global)
    globals()[combo_global] = [combo]
    try:
        return fn(complexity, idx, rng)
    finally:
        globals()[combo_global] = old_value


def _tpl_spouse_award_v34_from_combo(complexity: str, idx: int, rng: random.Random, combo: Dict[str, Any]) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = combo["occupation"]
    spouse_occ_key = combo["spouse_occupation"]
    country_key = combo["country"]
    award_key = combo["award"]
    start, end = combo["birth"]
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    occ = OCCUPATIONS[occ_key]
    spouse_occ = OCCUPATIONS[spouse_occ_key]
    country = COUNTRIES[country_key]
    award = AWARDS[award_key]
    where = [
        f"?spouse wdt:P106 wd:{spouse_occ['qid']} .",
        *_p_award_where_lines_for_var(award_key, var="spouse", value_var="?spouseAwardReceived"),
        "?spouse wdt:P31 wd:Q5 .",
        "?person wdt:P26 ?spouse .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end), f"у которых супруг или супруга имеет профессию «{spouse_occ['ru']}» и получил(а) награду: {award['ru']}"])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), _country_phrase_en(country_key), _birth_phrase_en(start, end), f"whose spouse is a {spouse_occ['en']} and received the {award['en']}"])
    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "citizenship_country": country["en"], "citizenship_country_qid": country["qid"], "birth_year_from": start, "birth_year_to": end,
        "spouse_occupation": spouse_occ["en"], "spouse_occupation_qid": spouse_occ["qid"],
        "spouse_award_received": award["en"], "spouse_award_received_qid": award["qid"],
        "spouse_bridge_path": "P26/P106/P166",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_spouse_award_v34",
        template_family="spouse_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested, extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> spouse -> spouse occupation and award"},
    )


def _tpl_parent_award_v34_from_combo(complexity: str, idx: int, rng: random.Random, combo: Dict[str, Any]) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    requested = PEOPLE_REQUESTED_COUNT[complexity]
    occ_key = combo["occupation"]
    parent_occ_key = combo["parent_occupation"]
    country_key = combo["country"]
    award_key = combo["award"]
    start, end = combo["birth"]
    if not _p_runtime_allowed_occupation_keys(complexity, [occ_key]):
        return None
    occ = OCCUPATIONS[occ_key]
    parent_occ = OCCUPATIONS[parent_occ_key]
    country = COUNTRIES[country_key]
    award = AWARDS[award_key]
    where = [
        f"?parent wdt:P106 wd:{parent_occ['qid']} .",
        *_p_award_where_lines_for_var(award_key, var="parent", value_var="?parentAwardReceived"),
        "?parent wdt:P31 wd:Q5 .",
        "?person wdt:P22|wdt:P25 ?parent .",
        _p_occupation_line(occ["qid"]),
        f"?person wdt:P27 wd:{country['qid']} .",
        *_p_birth_range_lines(start, end),
    ]
    ru_head, en_head = _p_head(requested)
    q_ru = _ru_join([ru_head, _occ_phrase_ru(occ_key), _country_phrase_ru(country_key), _birth_phrase_ru(start, end), f"у которых отец или мать имеет профессию «{parent_occ['ru']}» и получил(а) награду: {award['ru']}"])
    q_en = _en_join([en_head, _occ_phrase_en(occ_key), _country_phrase_en(country_key), _birth_phrase_en(start, end), f"whose parent is a {parent_occ['en']} and received the {award['en']}"])
    constraints = {
        "kind": "human", "occupation": occ["en"], "occupation_qid": occ["qid"], "occupation_match": "P106",
        "citizenship_country": country["en"], "citizenship_country_qid": country["qid"], "birth_year_from": start, "birth_year_to": end,
        "parent_occupation": parent_occ["en"], "parent_occupation_qid": parent_occ["qid"],
        "parent_award_received": award["en"], "parent_award_received_qid": award["qid"],
        "parent_bridge_path": "P22/P25 -> P106/P166",
    }
    return _p_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="people_parent_award_v34",
        template_family="parent_award_bridge", q_ru=q_ru, q_en=q_en,
        constraints=constraints, where_lines=where, requested_count=requested, extra_select_vars=["?birthDate"],
        bridge_meta={"bridge": "person -> parent -> parent occupation and award"},
    )


def _people_v34_call_builder_template(spec: Dict[str, Any], complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    return spec["builder"](complexity, idx, rng, spec["combo"])


def _people_v34_spec(level: str, family: str, fn_name: str, combo: Dict[str, Any], *, combo_global: Optional[str] = None, fn=None, builder=None) -> Dict[str, Any]:
    return {
        "level": level,
        "family": family,
        "fn_name": fn_name,
        "combo": combo,
        "combo_global": combo_global,
        "fn": fn,
        "builder": builder,
    }


### 12b. v34 — candidate queue builder and generation loop

In [ ]:
def _people_v34_build_candidate_specs() -> Dict[str, List[Dict[str, Any]]]:
    specs: Dict[str, List[Dict[str, Any]]] = {"L3": [], "L5": []}
    for combo in L3_DIRECTED_FILM_COMBOS_V34:
        specs["L3"].append(_people_v34_spec("L3", "directed_film_lang_genre_bridge", "_tpl_directed_film_lang_genre_v32", combo, combo_global="L3_DIRECTED_FILM_COMBOS_V32", fn=_tpl_directed_film_lang_genre_v32))
    for combo in L3_AUTHOR_WORK_COMBOS_V34:
        specs["L3"].append(_people_v34_spec("L3", "authored_work_lang_genre_bridge", "_tpl_author_work_lang_genre_v32", combo, combo_global="L3_AUTHOR_WORK_COMBOS_V32", fn=_tpl_author_work_lang_genre_v32))
    for combo in L3_EMPLOYER_COMBOS_V34:
        specs["L3"].append(_people_v34_spec("L3", "employer_country_bridge", "_tpl_employer_country_bridge_v32", combo, combo_global="L3_EMPLOYER_COMBOS_V32", fn=_tpl_employer_country_bridge_v32))
    for combo in L3_EDUCATION_COMBOS_V34:
        specs["L3"].append(_people_v34_spec("L3", "education_country_bridge", "_tpl_education_country_v32", combo, combo_global="L3_EDUCATION_COMBOS_V32", fn=_tpl_education_country_v32))

    for combo in L5_ACTOR_DIRECTOR_AWARD_COMBOS_V34:
        specs["L5"].append(_people_v34_spec("L5", "actor_film_director_award_bridge", "_tpl_actor_film_director_award_v32", combo, combo_global="L5_ACTOR_DIRECTOR_AWARD_COMBOS_V32", fn=_tpl_actor_film_director_award_v32))
    for combo in L5_DIRECTOR_CAST_AWARD_COMBOS_V34:
        specs["L5"].append(_people_v34_spec("L5", "director_film_cast_award_bridge", "_tpl_director_film_cast_award_v32", combo, combo_global="L5_DIRECTOR_CAST_AWARD_COMBOS_V32", fn=_tpl_director_film_cast_award_v32))
    for combo in L5_WRITER_ADAPTATION_COMBOS_V34:
        specs["L5"].append(_people_v34_spec("L5", "writer_work_film_adaptation_bridge", "_tpl_writer_work_film_adaptation_v32", combo, combo_global="L5_WRITER_ADAPTATION_COMBOS_V32", fn=_tpl_writer_work_film_adaptation_v32))
    for combo in L5_ADVISOR_AWARD_COMBOS_V34:
        specs["L5"].append(_people_v34_spec("L5", "advisor_award_bridge", "_tpl_advisor_award_v32", combo, combo_global="L5_ADVISOR_AWARD_COMBOS_V32", fn=_tpl_advisor_award_v32))
    for combo in L5_SPOUSE_AWARD_COMBOS_V34:
        specs["L5"].append(_people_v34_spec("L5", "spouse_award_bridge", "_tpl_spouse_award_v34_from_combo", combo, builder=_tpl_spouse_award_v34_from_combo))
    for combo in L5_PARENT_AWARD_COMBOS_V34:
        specs["L5"].append(_people_v34_spec("L5", "parent_award_bridge", "_tpl_parent_award_v34_from_combo", combo, builder=_tpl_parent_award_v34_from_combo))
    return specs


def _people_v34_interleave_by_family(specs: List[Dict[str, Any]], rng: random.Random) -> List[Dict[str, Any]]:
    by_family: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for spec in specs:
        by_family[spec["family"]].append(spec)
    for items in by_family.values():
        rng.shuffle(items)
    families = list(by_family.keys())
    rng.shuffle(families)
    out: List[Dict[str, Any]] = []
    while any(by_family.values()):
        for fam in list(families):
            if by_family[fam]:
                out.append(by_family[fam].pop())
    return out


_PEOPLE_CANDIDATE_SPECS_V34 = _people_v34_build_candidate_specs()
_PEOPLE_CANDIDATE_QUEUES_V34: Dict[str, List[Dict[str, Any]]] = {
    level: _people_v34_interleave_by_family(specs, random.Random(PEOPLE_RANDOM_SEED + i))
    for i, (level, specs) in enumerate(_PEOPLE_CANDIDATE_SPECS_V34.items())
}

# v34 family caps are enforced before WDQS calls, so once a family is full we do
# not spend time querying more candidates from it.
PEOPLE_TEMPLATE_FAMILY_CAPS_BY_LEVEL = {
    "L3": {
        "directed_film_lang_genre_bridge": 0.34,
        "authored_work_lang_genre_bridge": 0.34,
        "employer_country_bridge": 0.24,
        "education_country_bridge": 0.30,
    },
    "L5": {
        "actor_film_director_award_bridge": 0.30,
        "director_film_cast_award_bridge": 0.28,
        "writer_work_film_adaptation_bridge": 0.24,
        "advisor_award_bridge": 0.22,
        "spouse_award_bridge": 0.12,
        "parent_award_bridge": 0.12,
    },
}
PEOPLE_MAX_CONSECUTIVE_SAME_FAMILY_BY_LEVEL = {"L3": 2, "L5": 2}


def _people_v34_log_spec(spec: Dict[str, Any], status: str, error: Optional[str] = None) -> None:
    _PEOPLE_CANDIDATE_LOG_V34.append({
        "status": status,
        "level": spec.get("level"),
        "family": spec.get("family"),
        "template": spec.get("fn_name"),
        "combo": spec.get("combo"),
        "error": error,
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    })
    if len(_PEOPLE_CANDIDATE_LOG_V34) > 500:
        del _PEOPLE_CANDIDATE_LOG_V34[:100]


if "people_current_audit_base_v34" not in globals():
    people_current_audit_base_v34 = people_current_audit


def people_current_audit(records: List[Dict[str, Any]], skipped: List[Dict[str, Any]], bad_lines: List[Dict[str, Any]]):
    audit = people_current_audit_base_v34(records, skipped, bad_lines)
    audit["candidate_queue_v34"] = {
        "remaining_by_level": {level: len(q) for level, q in _PEOPLE_CANDIDATE_QUEUES_V34.items()},
        "initial_by_level": {level: len(specs) for level, specs in _PEOPLE_CANDIDATE_SPECS_V34.items()},
        "recent_candidate_log": _PEOPLE_CANDIDATE_LOG_V34[-80:],
        "finalize_cache_size": len(_PEOPLE_FINALIZE_CACHE_V34),
        "policy": "queue-based generation; each candidate combination is tried at most once and failed WDQS results are memoized",
    }
    return audit


def generate_people_example(
    *,
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = PEOPLE_MAX_ATTEMPTS_PER_EXAMPLE,
) -> BenchmarkExample:
    queue = _PEOPLE_CANDIDATE_QUEUES_V34.get(complexity, [])
    tried = 0
    last_error = None

    while queue and tried < int(max_attempts):
        spec = queue.pop(0)
        tried += 1

        # Skip full families before WDQS.
        if not _people_family_allowed_for_pick(complexity, spec.get("family")):
            _people_v34_log_spec(spec, "skipped_family_quota")
            continue

        try:
            local_rng = random.Random(hash(json.dumps(spec.get("combo"), sort_keys=True, ensure_ascii=False)) & 0xFFFFFFFF)
            if spec.get("combo_global"):
                ex = _people_v34_call_combo_template(spec, complexity, idx, local_rng)
            else:
                ex = _people_v34_call_builder_template(spec, complexity, idx, local_rng)
            if ex is None:
                _people_v34_log_spec(spec, "rejected_by_gold_or_quality")
                continue
            ex.id = f"people_{complexity.lower()}_{idx:04d}"
            _people_v34_log_spec(spec, "accepted_candidate")
            return ex
        except KeyboardInterrupt:
            raise
        except Exception as e:
            last_error = f"{type(e).__name__}: {str(e)[:400]}"
            _people_v34_log_spec(spec, "exception", last_error)
            continue

    raise RuntimeError(
        f"No valid queued candidate for people:{complexity} in this call; "
        f"tried={tried}, remaining={len(queue)}, last_error={last_error}"
    )


print("✅ v34 queue-based people patch loaded")
print("   output:", PEOPLE_OUTPUT_PATH.resolve())
print("   targets:", PEOPLE_TARGET_PER_LEVEL)
print("   WDQS timeout/retries:", getattr(wd, "timeout", None), getattr(wd, "max_retries", None))
print("   reference files:", [str(p) for p in PEOPLE_REFERENCE_JSONL_PATHS])
print("   queued candidates:", {level: len(q) for level, q in _PEOPLE_CANDIDATE_QUEUES_V34.items()})
print("   L3 families:", dict(Counter(s["family"] for s in _PEOPLE_CANDIDATE_SPECS_V34["L3"])))
print("   L5 families:", dict(Counter(s["family"] for s in _PEOPLE_CANDIDATE_SPECS_V34["L5"])))


## 13. Optional smoke test

In [12]:
RUN_PEOPLE_SMOKE_TEST = False

if RUN_PEOPLE_SMOKE_TEST:
    smoke_rng = random.Random(PEOPLE_RANDOM_SEED)
    ex = generate_people_example(complexity="L3", idx=1, rng=smoke_rng, max_attempts=4)
    print(json.dumps(asdict(ex), ensure_ascii=False, indent=2)[:6000])


## 14. Generate people L3/L5 supplement

In [13]:
# =========================
# Output paths
# =========================

OUT_DIR = PEOPLE_OUTPUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = PEOPLE_OUTPUT_PATH
audit_path = PEOPLE_AUDIT_PATH
checkpoint_path = PEOPLE_CHECKPOINT_PATH

print("output:", out_path.resolve())
print("audit:", audit_path.resolve())
print("checkpoint:", checkpoint_path.resolve())

# =========================
# Generation config
# =========================

TARGET_PLAN_PEOPLE = PEOPLE_TARGET_PER_LEVEL.copy()

SEED = PEOPLE_RANDOM_SEED
MAX_ATTEMPTS_PER_LEVEL = PEOPLE_MAX_ATTEMPTS_PER_LEVEL
MAX_ATTEMPTS_PER_EXAMPLE = PEOPLE_MAX_ATTEMPTS_PER_EXAMPLE

rng = random.Random(SEED)

# =========================
# Resume existing output
# =========================

records_raw, bad_lines = people_read_existing_jsonl(out_path)
records_raw = [people_normalize_record_format(r) for r in records_raw]

reference_records, reference_bad_lines = people_read_reference_jsonl(PEOPLE_REFERENCE_JSONL_PATHS)
print("reference records for dedup/id continuation:", len(reference_records))
if reference_bad_lines:
    print("reference bad json lines:", reference_bad_lines[:5])

# Keep resume safe: records generated by older/broader versions are removed from
# the active output before continuing. This is intentional because accepted records
# must satisfy the current quality filters.
records = []
skipped = []
for r in records_raw:
    problems = people_record_quality_problems(r)
    if problems:
        skipped.append({
            "complexity": r.get("complexity"),
            "reason": "existing_record_failed_quality_filter",
            "problems": problems,
            "id": r.get("id"),
            "query_text_ru": r.get("query_text_ru"),
            "constraints": r.get("constraints"),
            "gold_count": people_gold_count(r),
        })
        continue
    records.append(r)

if len(records) != len(records_raw):
    people_write_jsonl(out_path, records)
    print(f"cleaned existing output: kept {len(records)}/{len(records_raw)} records")

seen_keys = {people_make_record_key(r) for r in records}
seen_keys.update(people_make_record_key(r) for r in reference_records)
existing_counts = Counter(r.get("complexity") for r in records)

print("existing records:", len(records))
print("existing counts:", dict(existing_counts))

# =========================
# Generation
# =========================

overall_total = sum(TARGET_PLAN_PEOPLE.values())
already_done_total = sum(
    min(existing_counts.get(level, 0), target)
    for level, target in TARGET_PLAN_PEOPLE.items()
)

overall_bar = tqdm(
    total=overall_total,
    initial=already_done_total,
    desc="people total",
    position=0,
)



def _people_next_index_by_level(records):
    out = {level: 1 for level in TARGET_PLAN_PEOPLE}
    pat = re.compile(r"^people_(l\d+)_(\d{4})$")
    for rec in records:
        rid = str(rec.get("id") or "")
        m = pat.match(rid)
        if not m:
            continue
        level = m.group(1).upper()
        if level in out:
            out[level] = max(out[level], int(m.group(2)) + 1)
    return out

next_idx_by_level = _people_next_index_by_level(records + reference_records)
print("next indices by level:", next_idx_by_level)

PEOPLE_RUNTIME_RECORDS_FOR_QUOTAS = records
PEOPLE_RUNTIME_DISABLED_TEMPLATE_NAMES = {}

try:
    for complexity, target_n in TARGET_PLAN_PEOPLE.items():
        already_done = existing_counts.get(complexity, 0)

        if already_done >= target_n:
            print(f"SKIP: people:{complexity} already has {already_done}/{target_n}")
            continue

        level_records_count = already_done
        attempts = 0
        last_error = None

        level_bar = tqdm(
            total=target_n,
            initial=already_done,
            desc=f"people:{complexity}",
            position=1,
            leave=True,
        )

        while level_records_count < target_n and attempts < MAX_ATTEMPTS_PER_LEVEL:
            attempts += 1

            # Pre-flight generation context.  Avoid expensive WDQS calls for
            # template families that are already impossible due to quotas.
            PEOPLE_RUNTIME_RECORDS_FOR_QUOTAS = records
            disabled_templates = set()
            if complexity == "L2":
                l2_awards_now = people_l2_award_count(records)
                l2_award_cap = people_l2_award_limit(TARGET_PLAN_PEOPLE.get("L2", target_n))
                if l2_awards_now >= l2_award_cap:
                    disabled_templates.update({"_tpl_occ_award", "_tpl_award_country_birthrange"})
                if people_same_occupation_count(records, complexity, OCCUPATIONS["association football player"]["en"]) >= PEOPLE_MAX_SAME_OCCUPATION_PER_LEVEL.get(complexity, 999999):
                    disabled_templates.add("_tpl_football_position_birthrange")
            PEOPLE_RUNTIME_DISABLED_TEMPLATE_NAMES[complexity] = disabled_templates

            level_bar.set_postfix({
                "ok": level_records_count,
                "target": target_n,
                "attempts": attempts,
                "last_error": last_error,
            })

            try:
                ex = generate_people_example(
                    complexity=complexity,
                    idx=next_idx_by_level[complexity],
                    rng=rng,
                    max_attempts=MAX_ATTEMPTS_PER_EXAMPLE,
                )

                r = people_example_to_dict(ex)
                key = people_make_record_key(r)

                if key in seen_keys:
                    skipped.append({
                        "complexity": complexity,
                        "reason": "duplicate",
                        "query_text_ru": r.get("query_text_ru"),
                        "constraints": r.get("constraints"),
                    })
                    continue

                if complexity == "L2" and people_record_has_award(r):
                    l2_awards_now = people_l2_award_count(records)
                    l2_award_cap = people_l2_award_limit(TARGET_PLAN_PEOPLE.get("L2", target_n))
                    if l2_awards_now >= l2_award_cap:
                        skipped.append({
                            "complexity": complexity,
                            "reason": "l2_award_template_quota",
                            "l2_awards_now": l2_awards_now,
                            "l2_award_cap": l2_award_cap,
                            "query_text_ru": r.get("query_text_ru"),
                            "constraints": r.get("constraints"),
                            "template_id": r.get("template_id"),
                            "template_family": r.get("template_family"),
                        })
                        continue

                occ = people_record_occupation(r)
                occ_limit = PEOPLE_MAX_SAME_OCCUPATION_PER_LEVEL.get(complexity)
                if occ and occ_limit is not None and people_same_occupation_count(records, complexity, occ) >= int(occ_limit):
                    skipped.append({
                        "complexity": complexity,
                        "reason": "occupation_level_quota",
                        "occupation": occ,
                        "occupation_level_count": people_same_occupation_count(records, complexity, occ),
                        "occupation_level_limit": int(occ_limit),
                        "query_text_ru": r.get("query_text_ru"),
                        "constraints": r.get("constraints"),
                    })
                    continue

                quality_problems = people_record_quality_problems(r)
                if quality_problems:
                    skipped.append({
                        "complexity": complexity,
                        "reason": "new_record_failed_quality_filter",
                        "problems": quality_problems,
                        "query_text_ru": r.get("query_text_ru"),
                        "constraints": r.get("constraints"),
                        "gold_count": people_gold_count(r),
                    })
                    continue

                seen_keys.add(key)
                records.append(r)
                people_append_jsonl(out_path, r)

                next_idx_by_level[complexity] += 1
                level_records_count += 1

                level_bar.update(1)
                overall_bar.update(1)

                level_bar.set_postfix({
                    "ok": level_records_count,
                    "target": target_n,
                    "attempts": attempts,
                    "template": people_record_template(r),
                    "gold": people_gold_count(r),
                })

                audit = people_current_audit(records, skipped, bad_lines)
                people_write_json(audit_path, audit)
                people_write_json(checkpoint_path, {
                    "last_record": r,
                    "audit": audit,
                })

            except KeyboardInterrupt:
                raise

            except Exception as e:
                last_error = type(e).__name__
                skipped.append({
                    "complexity": complexity,
                    "reason": type(e).__name__,
                    "error": str(e)[:800],
                })
                audit = people_current_audit(records, skipped, bad_lines)
                people_write_json(audit_path, audit)
                continue

        level_bar.close()

        if level_records_count < target_n:
            print(
                f"WARN: only {level_records_count}/{target_n} "
                f"for people:{complexity} after {attempts} attempts"
            )
        else:
            print(
                f"OK: people:{complexity} {level_records_count}/{target_n} "
                f"generated in {attempts} attempts"
            )

finally:
    overall_bar.close()
    audit = people_current_audit(records, skipped, bad_lines)
    people_write_json(audit_path, audit)

    stats = validate_people_records(records)

    print()
    print("saved incrementally:", out_path.resolve())
    print("audit:", audit_path.resolve())
    print("checkpoint:", checkpoint_path.resolve())
    print("records:", len(records))
    print("counts:", dict(Counter(r.get("complexity") for r in records)))
    print("templates:", dict(Counter(people_record_template(r) for r in records)))
    print("skipped:", len(skipped))
    print("validation:", json.dumps(stats, ensure_ascii=False, indent=2))

    if stats["problem_count"]:
        raise RuntimeError("People dataset validation failed; inspect problems_preview above.")


output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_queue.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_queue_audit.json
checkpoint: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_queue_checkpoint.json
reference records for dedup/id continuation: 253
existing records: 0
existing counts: {}


people total:   0%|          | 0/70 [00:00<?, ?it/s]

next indices by level: {'L1': 21, 'L2': 26, 'L3': 9, 'L4': 46, 'L5': 56}
SKIP: people:L1 already has 0/0
SKIP: people:L2 already has 0/0


people:L3:  71%|███████▏  | 25/35 [20:13<08:05, 48.56s/it, ok=25, target=35, attempts=450, last_error=RuntimeError]


WARN: only 25/35 for people:L3 after 450 attempts
SKIP: people:L4 already has 0/0


people total:  80%|████████  | 56/70 [39:00<09:45, 41.79s/it] 


WARN: only 31/35 for people:L5 after 450 attempts

saved incrementally: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_queue.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_queue_audit.json
checkpoint: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/people_l3_l5_patch_v34_queue_checkpoint.json
records: 56
counts: {'L3': 25, 'L5': 31}
templates: {'people_education_country_v32': 4, 'people_author_work_lang_genre_v32': 12, 'people_directed_film_lang_genre_v32': 9, 'people_parent_award_v34': 3, 'people_director_film_cast_award_v32': 10, 'people_advisor_award_v32': 4, 'people_spouse_award_v34': 4, 'people_actor_film_director_award_v32': 8, 'people_writer_work_film_adaptation_v32': 2}
skipped: 844
validation: {
  "n_examples": 56,
  "by_level": {
    "L3": 25,
 

## 15. Output statistics

In [14]:
records, bad_lines = people_read_existing_jsonl(PEOPLE_OUTPUT_PATH)
stats = validate_people_records(records)
print(json.dumps(stats, ensure_ascii=False, indent=2))

# Compact human-readable preview.
for r in records[:5]:
    print("\n", r.get("id"), r.get("complexity"), r.get("template_id"), "gold=", people_gold_count(r))
    print("RU:", r.get("query_text_ru"))
    print("EN:", r.get("query_text_en"))

if bad_lines:
    print("Bad JSON lines:", bad_lines[:20])


{
  "n_examples": 56,
  "by_level": {
    "L3": 25,
    "L5": 31
  },
  "template_families": {
    "education_country_bridge": 4,
    "authored_work_lang_genre_bridge": 12,
    "directed_film_lang_genre_bridge": 9,
    "parent_award_bridge": 3,
    "director_film_cast_award_bridge": 10,
    "advisor_award_bridge": 4,
    "spouse_award_bridge": 4,
    "actor_film_director_award_bridge": 8,
    "writer_work_film_adaptation_bridge": 2
  },
  "templates": {
    "people_education_country_v32": 4,
    "people_author_work_lang_genre_v32": 12,
    "people_directed_film_lang_genre_v32": 9,
    "people_parent_award_v34": 3,
    "people_director_film_cast_award_v32": 10,
    "people_advisor_award_v32": 4,
    "people_spouse_award_v34": 4,
    "people_actor_film_director_award_v32": 8,
    "people_writer_work_film_adaptation_v32": 2
  },
  "gold_count_min": 5,
  "gold_count_max": 63,
  "problem_count": 0,
  "problems_preview": []
}

 people_l3_0009 L3 people_education_country_v32 gold= 12
RU: Назо